# RonaAir — Decision Fusion & Risk AI

**Early Warning System (EWS) + Decision Support System untuk kualitas air kolam ikan air tawar.**

Notebook ini **tidak** melatih ulang model Computer Vision RonaAir dan **tidak** melatih ulang
model pH-strip (PolynomialRidge2 / COMBINED_ALL). Notebook ini membangun lapisan
**Decision Fusion / Risk AI**: rule engine, benchmark supervised, recommendation engine,
dan artefak deployment Android.

---

### Prinsip kerja notebook ini

```
AUDIT -> HARMONIZATION -> COMPATIBILITY -> LEAKAGE CHECK -> ROLE ASSIGNMENT -> MODELING
```

Tidak ada angka, label, threshold, atau metrik yang dikarang. Semua yang tercetak adalah
hasil eksekusi aktual terhadap file dataset yang benar-benar tersedia.

**Peringatan ilmiah yang berlaku untuk seluruh notebook:** RonaAir adalah *early warning*
dan *decision support*, bukan detektor spesies alga, bukan pengukur konsentrasi toksin,
bukan pengganti laboratorium, dan bukan sistem medis. Setiap nilai DO yang tidak berasal
dari sensor DO fisik selalu ditulis sebagai **Estimated DO**, bukan *Measured DO*.


## Cell 1 — Setup, konfigurasi, dan reproducibility


In [1]:
import os
import sys
import json
import time
import zipfile
import platform
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

NOTEBOOK_VERSION = "1.0.0"
MODEL_VERSION = "ronair-risk-ai-1.0.0"

# --- Struktur folder output (relatif terhadap working directory notebook) ---
ROOT = Path(os.environ.get("RONAAIR_ROOT", ".")).resolve()
DIR_MODELS = ROOT / "models"
DIR_CONFIGS = ROOT / "configs"
DIR_DEPLOY = ROOT / "deployment"
DIR_RESULTS = ROOT / "results"
DIR_FIGS = DIR_RESULTS / "figures"
for d in [DIR_MODELS, DIR_CONFIGS, DIR_DEPLOY, DIR_RESULTS, DIR_FIGS]:
    d.mkdir(parents=True, exist_ok=True)

# --- Taksonomi risiko RonaAir (ordinal, internal) ---
RONAAIR_LABELS = ["NORMAL", "WASPADA", "SIAGA", "DARURAT"]
SEVERITY = {"NORMAL": 0, "WASPADA": 1, "SIAGA": 2, "DARURAT": 3}
HIGH_RISK = ["SIAGA", "DARURAT"]

# --- Fitur yang BENAR-BENAR tersedia di APK RonaAir saat inference ---
# Ini adalah kontrak deployment. Predictor di luar daftar ini tidak boleh menjadi
# fitur model produksi (lihat Section 18 master prompt).
DEPLOYMENT_FEATURES = {
    "visual_score":   "CV model (ronair_cv_classifier.tflite)",
    "visual_condition": "CV model (kelas visual)",
    "image_quality":  "OpenCV image quality check",
    "ph_visual_est":  "pH photo model (PolynomialRidge2 / COMBINED_ALL)",
    "ph_sensor":      "ESP32 pH probe",
    "water_temp":     "ESP32 temperature probe",
    "ec_value":       "ESP32 EC probe",
    "tds_ppm":        "ESP32 TDS (derived from EC)",
    "do_est":         "DO soft-sensor (ESTIMATED, not measured)",
    "hour":           "device clock",
}

def stamp():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def hr(title=""):
    print("=" * 78)
    if title:
        print(title)
        print("=" * 78)

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
    print(f"  [saved] {path.relative_to(ROOT) if ROOT in path.parents or path.parent == ROOT else path}")

def save_csv(df, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)
    print(f"  [saved] {path.name}  ({len(df)} rows)")

hr("RONAAIR — DECISION FUSION & RISK AI")
print(f"Run time          : {stamp()}")
print(f"Notebook version  : {NOTEBOOK_VERSION}")
print(f"Python            : {sys.version.split()[0]} on {platform.system()}")
print(f"numpy / pandas    : {np.__version__} / {pd.__version__}")
try:
    import sklearn
    print(f"scikit-learn      : {sklearn.__version__}")
except Exception as e:  # pragma: no cover
    raise RuntimeError("scikit-learn wajib tersedia") from e

HAS_XGB = False
try:
    import xgboost  # noqa: F401
    HAS_XGB = True
    print(f"xgboost           : {xgboost.__version__} (available)")
except Exception:
    print("xgboost           : NOT AVAILABLE -> akan di-skip secara graceful")

HAS_SCIPY = False
try:
    from scipy import stats as sp_stats
    HAS_SCIPY = True
    print("scipy             : available (KS-test & Wasserstein aktif)")
except Exception:
    print("scipy             : NOT AVAILABLE -> KS/Wasserstein di-skip")

print(f"Output root       : {ROOT}")


RONAAIR — DECISION FUSION & RISK AI
Run time          : 2026-09-18 01:24:22
Notebook version  : 1.0.0
Python            : 3.12.3 on Linux
numpy / pandas    : 2.4.4 / 3.0.2
scikit-learn      : 1.8.0
xgboost           : NOT AVAILABLE -> akan di-skip secara graceful
scipy             : available (KS-test & Wasserstein aktif)
Output root       : /home/claude/build


## Cell 2 — Lokasi dataset

Notebook mencari file dataset di beberapa lokasi kandidat. Di Google Colab, unggah
keempat file ke `/content/data` (atau jalankan sel upload di bawah).
**Tidak ada dataset yang di-download otomatis** — semua harus berasal dari file yang Anda sediakan.


In [2]:
CANDIDATE_DIRS = [
    Path("data"),
    Path("/content/data"),
    Path("/content"),
    Path("/content/drive/MyDrive/RonaAir/data"),
    Path("/mnt/user-data/uploads"),
    Path("."),
]

IN_COLAB = "google.colab" in sys.modules

# Sel upload opsional untuk Colab (aman untuk dijalankan di lingkungan lain).
if IN_COLAB and not any((d / "fishpond_dataset_multiclass_2153.csv").exists() for d in CANDIDATE_DIRS):
    try:
        from google.colab import files as colab_files
        print("Silakan unggah 4 file dataset (csv / xlsx / zip)...")
        Path("data").mkdir(exist_ok=True)
        uploaded = colab_files.upload()
        for fn in uploaded:
            Path(fn).replace(Path("data") / fn)
    except Exception as e:
        print(f"Upload dilewati: {e}")

DATASET_FILES = {
    "D1_AQUAPONDS": ["Aquaponds_Dataset-selected-columns.csv", "Aquaponds Dataset-selected-columns.csv"],
    "D2_FISHPOND_MULTICLASS": ["fishpond_dataset_multiclass_2153.csv"],
    "D3_IOTMLCQ_2024": ["Data_Model_IoTMLCQ_2024.xlsx"],
    "D4_AQUAPONIC_IOT": [
        "A_Simple_Dataset_of_Aquaponic_Fish_Pond_Water_Quality_Measurement_using_Internet_of_Things_devices.zip",
        "A Simple Dataset of Aquaponic Fish Pond Water Quality Measurement using Internet of Things devices.zip",
    ],
}

def find_file(names):
    for d in CANDIDATE_DIRS:
        for n in names:
            p = d / n
            if p.exists():
                return p.resolve()
    # pencarian rekursif terbatas sebagai fallback
    for d in CANDIDATE_DIRS:
        if not d.exists():
            continue
        for n in names:
            hits = list(d.rglob(n))
            if hits:
                return hits[0].resolve()
    return None

PATHS = {k: find_file(v) for k, v in DATASET_FILES.items()}

hr("LOKASI FILE DATASET")
for k, v in PATHS.items():
    print(f"  {k:26s} : {v if v else '*** TIDAK DITEMUKAN ***'}")

MISSING = [k for k, v in PATHS.items() if v is None]
if MISSING:
    print(f"\n  CATATAN: dataset berikut tidak ditemukan dan akan dilewati: {MISSING}")


LOKASI FILE DATASET
  D1_AQUAPONDS               : /home/claude/build/data/Aquaponds_Dataset-selected-columns.csv
  D2_FISHPOND_MULTICLASS     : /home/claude/build/data/fishpond_dataset_multiclass_2153.csv
  D3_IOTMLCQ_2024            : /home/claude/build/data/Data_Model_IoTMLCQ_2024.xlsx
  D4_AQUAPONIC_IOT           : /home/claude/build/data/A_Simple_Dataset_of_Aquaponic_Fish_Pond_Water_Quality_Measurement_using_Internet_of_Things_devices.zip


## Cell 3 — Loading dataset (tanpa asumsi, tanpa concat)

Aturan Section 6 master prompt: **jangan langsung concat**. Setiap dataset dimuat ke
objek terpisah dengan metadata provenance-nya sendiri.


In [3]:
RAW = {}        # dataset_key -> dict of DataFrame(s)
LOADERR = {}

def _norm_cols(df):
    df = df.copy()
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
    return df

# ---- D1 ----
if PATHS["D1_AQUAPONDS"]:
    try:
        d1 = _norm_cols(pd.read_csv(PATHS["D1_AQUAPONDS"]))
        # buang baris yang seluruhnya kosong (artefak export spreadsheet)
        d1_nonempty = d1.dropna(how="all")
        RAW["D1_AQUAPONDS"] = {"main": d1, "main_nonempty": d1_nonempty}
        print(f"D1 loaded: shape mentah {d1.shape}, baris non-kosong {len(d1_nonempty)}")
    except Exception as e:
        LOADERR["D1_AQUAPONDS"] = str(e)
        print(f"D1 GAGAL dimuat: {e}")

# ---- D2 ----
if PATHS["D2_FISHPOND_MULTICLASS"]:
    try:
        d2 = _norm_cols(pd.read_csv(PATHS["D2_FISHPOND_MULTICLASS"]))
        RAW["D2_FISHPOND_MULTICLASS"] = {"main": d2}
        print(f"D2 loaded: shape {d2.shape}")
    except Exception as e:
        LOADERR["D2_FISHPOND_MULTICLASS"] = str(e)
        print(f"D2 GAGAL dimuat: {e}")

# ---- D3 ----
if PATHS["D3_IOTMLCQ_2024"]:
    try:
        xls = pd.ExcelFile(PATHS["D3_IOTMLCQ_2024"])
        sheets = {s: _norm_cols(xls.parse(s)) for s in xls.sheet_names}
        RAW["D3_IOTMLCQ_2024"] = sheets
        print(f"D3 loaded: sheets={list(sheets)} shapes={[v.shape for v in sheets.values()]}")
    except Exception as e:
        LOADERR["D3_IOTMLCQ_2024"] = str(e)
        print(f"D3 GAGAL dimuat: {e}")

# ---- D4 (zip berisi raw + filtered) ----
if PATHS["D4_AQUAPONIC_IOT"]:
    try:
        extract_dir = ROOT / "data" / "_d4_extracted"
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(PATHS["D4_AQUAPONIC_IOT"]) as zf:
            members = [m for m in zf.namelist() if m.lower().endswith(".csv")]
            zf.extractall(extract_dir)
        d4 = {}
        for m in members:
            p = extract_dir / m
            key = "raw" if "raw" in Path(m).stem.lower() else "filtered"
            df = _norm_cols(pd.read_csv(p))
            d4[key] = df
            print(f"D4 [{key}] loaded: {Path(m).name} shape {df.shape}")
        RAW["D4_AQUAPONIC_IOT"] = d4
    except Exception as e:
        LOADERR["D4_AQUAPONIC_IOT"] = str(e)
        print(f"D4 GAGAL dimuat: {e}")

print(f"\nDataset berhasil dimuat: {list(RAW)}")


D1 loaded: shape mentah (124, 10), baris non-kosong 0
D2 loaded: shape (2153, 10)
D3 loaded: sheets=['Sheet1'] shapes=[(4383, 28)]
D4 [filtered] loaded: pond_iot_2023.csv shape (118286, 5)
D4 [raw] loaded: pond_iot_2023_raw.csv shape (505730, 5)

Dataset berhasil dimuat: ['D1_AQUAPONDS', 'D2_FISHPOND_MULTICLASS', 'D3_IOTMLCQ_2024', 'D4_AQUAPONIC_IOT']


## Cell 4 — Dataset audit wajib (Section 7)

Untuk setiap tabel: ukuran file, dimensi, tipe, missing, duplikat, range numerik,
ketersediaan timestamp / site / pond / sample, kandidat target, dan variabilitas target.


In [4]:
TIME_HINTS = ["time", "date", "datetime", "timestamp", "created"]
SITE_HINTS = ["site", "station", "pond", "kolam", "tank", "location", "device", "sensor_id"]
SAMPLE_HINTS = ["sample", "id", "index", "no"]

def guess_role_columns(df):
    cols = list(df.columns)
    low = {c: str(c).lower() for c in cols}
    ts = [c for c in cols if any(h in low[c] for h in TIME_HINTS)]
    site = [c for c in cols if any(h in low[c] for h in SITE_HINTS)]
    samp = [c for c in cols if any(h == low[c] or low[c].endswith("_" + h) for h in SAMPLE_HINTS)]
    return ts, site, samp

def target_candidates(df, max_card=12):
    """Kolom kategorikal berkardinalitas rendah ATAU kolom bernama seperti label."""
    out = []
    name_hints = ["class", "label", "status", "risk", "alert", "target", "health", "grade", "category"]
    for c in df.columns:
        s = df[c]
        n = s.nunique(dropna=True)
        looks_named = any(h in str(c).lower() for h in name_hints)
        is_cat = (not pd.api.types.is_numeric_dtype(s)) and 1 <= n <= max_card
        is_lowcard_num = pd.api.types.is_numeric_dtype(s) and 1 <= n <= 6
        if looks_named or is_cat or is_lowcard_num:
            out.append({"column": c, "n_unique": int(n),
                        "values": sorted(map(str, pd.Series(s.dropna().unique()).head(12).tolist())),
                        "variability": "CONSTANT (0 informasi)" if n <= 1 else "OK"})
    return out

AUDIT_ROWS = []
AUDIT_FULL = {}

def audit_table(dataset_key, table_key, df, file_path):
    ts_cols, site_cols, samp_cols = guess_role_columns(df)
    num = df.select_dtypes(include=[np.number])
    ranges = {c: {"min": float(num[c].min()), "max": float(num[c].max()),
                  "mean": float(num[c].mean()), "std": float(num[c].std()),
                  "n_unique": int(num[c].nunique())}
              for c in num.columns} if len(num.columns) and len(df) else {}
    tcands = target_candidates(df)
    fsize = Path(file_path).stat().st_size if file_path and Path(file_path).exists() else np.nan

    info = {
        "dataset_name": dataset_key,
        "table": table_key,
        "file_name": Path(file_path).name if file_path else "",
        "file_format": Path(file_path).suffix.lower().lstrip(".") if file_path else "",
        "file_size_bytes": int(fsize) if fsize == fsize else None,
        "n_rows": int(len(df)),
        "n_columns": int(df.shape[1]),
        "column_names": list(map(str, df.columns)),
        "dtypes": {str(c): str(t) for c, t in df.dtypes.items()},
        "missing_values": {str(c): int(v) for c, v in df.isna().sum().items()},
        "total_missing": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "numeric_ranges": ranges,
        "timestamp_columns": ts_cols,
        "site_pond_columns": site_cols,
        "sample_id_columns": samp_cols,
        "target_candidates": tcands,
        # dataset ini tidak berisi citra
        "n_images": 0,
        "image_metadata": "NOT APPLICABLE — dataset tabular, tidak ada citra",
    }
    AUDIT_FULL[f"{dataset_key}::{table_key}"] = info
    AUDIT_ROWS.append({
        "dataset": dataset_key,
        "table": table_key,
        "file_name": info["file_name"],
        "file_format": info["file_format"],
        "file_size_bytes": info["file_size_bytes"],
        "n_rows": info["n_rows"],
        "n_columns": info["n_columns"],
        "total_missing": info["total_missing"],
        "duplicate_rows": info["duplicate_rows"],
        "has_timestamp": bool(ts_cols),
        "timestamp_cols": "|".join(map(str, ts_cols)),
        "has_site_pond": bool(site_cols),
        "site_pond_cols": "|".join(map(str, site_cols)),
        "has_sample_id": bool(samp_cols),
        "n_target_candidates": len(tcands),
        "target_candidates": "|".join(t["column"] for t in tcands),
        "constant_target_candidates": "|".join(t["column"] for t in tcands if t["variability"].startswith("CONSTANT")),
        "n_images": 0,
    })
    return info

hr("SECTION 7 — DATASET AUDIT")
for dkey, tables in RAW.items():
    for tkey, df in tables.items():
        if dkey == "D1_AQUAPONDS" and tkey == "main_nonempty":
            continue  # tabel turunan, bukan file asli
        info = audit_table(dkey, tkey, df, PATHS[dkey])
        print(f"\n--- {dkey} / {tkey} ---")
        print(f"  file            : {info['file_name']} ({info['file_format']}, {info['file_size_bytes']:,} bytes)")
        print(f"  shape           : {info['n_rows']:,} rows x {info['n_columns']} cols")
        print(f"  missing (total) : {info['total_missing']:,}")
        print(f"  duplicate rows  : {info['duplicate_rows']:,}")
        print(f"  timestamp cols  : {info['timestamp_columns'] or 'TIDAK ADA'}")
        print(f"  site/pond cols  : {info['site_pond_columns'] or 'TIDAK ADA'}")
        print(f"  target kandidat : {[t['column'] for t in info['target_candidates']] or 'TIDAK ADA'}")
        for t in info["target_candidates"]:
            if t["variability"].startswith("CONSTANT"):
                print(f"      !! {t['column']} -> {t['variability']}")

audit_df = pd.DataFrame(AUDIT_ROWS)
print()
save_csv(audit_df, DIR_RESULTS / "dataset_audit.csv")
save_json(AUDIT_FULL, DIR_RESULTS / "dataset_audit.json")


SECTION 7 — DATASET AUDIT

--- D1_AQUAPONDS / main ---
  file            : Aquaponds_Dataset-selected-columns.csv (csv, 1,322 bytes)
  shape           : 124 rows x 10 cols
  missing (total) : 1,240
  duplicate rows  : 123
  timestamp cols  : ['Date', 'Time']
  site/pond cols  : ['station']
  target kandidat : TIDAK ADA

--- D2_FISHPOND_MULTICLASS / main ---
  file            : fishpond_dataset_multiclass_2153.csv (csv, 158,936 bytes)
  shape           : 2,153 rows x 10 cols
  missing (total) : 0
  duplicate rows  : 0
  timestamp cols  : ['timestamp']
  site/pond cols  : TIDAK ADA
  target kandidat : ['class']

--- D3_IOTMLCQ_2024 / Sheet1 ---
  file            : Data_Model_IoTMLCQ_2024.xlsx (xlsx, 589,563 bytes)
  shape           : 4,383 rows x 28 cols
  missing (total) : 0
  duplicate rows  : 0
  timestamp cols  : ['Datetime']
  site/pond cols  : TIDAK ADA
  target kandidat : ['Month', 'Oxygenation Interventions', 'Corrective Interventions', 'Average Temperature (°C)', 'High Temperatu

## Cell 5 — Jangan percaya nama kolom (Section 8)

Setiap kolom diverifikasi terhadap **rentang fisik yang masuk akal**, bukan namanya.
Kolom yang namanya menyiratkan parameter tertentu tetapi nilainya di luar rentang fisik
ditandai `UNVERIFIED` sampai provenance jelas.


In [5]:
# Rentang plausibilitas fisik (bukan threshold risiko). Dipakai hanya untuk verifikasi identitas kolom.
PHYSICAL_RANGE = {
    "ph":          (0.0, 14.0, "unit pH"),
    "temperature": (0.0, 45.0, "deg C"),
    "do":          (0.0, 25.0, "mg/L"),
    "ec":          (0.0, 100000.0, "uS/cm"),
    "tds":         (0.0, 50000.0, "mg/L atau ppm"),
    "turbidity":   (0.0, 4000.0, "NTU"),
    "orp":         (-2000.0, 2000.0, "mV"),
}

CONCEPT_HINTS = {
    "ph": ["ph"],
    "temperature": ["temp", "suhu"],
    "do": ["do", "dissolved oxygen", "oxygen", "oxigeno"],
    "ec": ["ec", "conduct"],
    "tds": ["tds"],
    "turbidity": ["turbid", "turbidez", "kekeruhan", "ntu"],
    "orp": ["orp", "redox"],
}

def infer_concept(colname):
    low = str(colname).lower()
    # cocokkan hint terpanjang lebih dulu agar 'ph' tidak menangkap 'graph'
    best = None
    for concept, hints in CONCEPT_HINTS.items():
        for h in hints:
            if h == "ph":
                ok = low == "ph" or low.startswith("ph") or "_ph" in low or "ph_" in low or "(ph" in low
            elif h == "do":
                ok = low == "do" or low.startswith("do_") or low.endswith("_do") or "dissolved" in low
            else:
                ok = h in low
            if ok:
                if best is None or len(h) > len(best[1]):
                    best = (concept, h)
    return best[0] if best else None

prov_rows = []
for full_key, info in AUDIT_FULL.items():
    dkey, tkey = full_key.split("::")
    df = RAW[dkey][tkey]
    for c in df.columns:
        concept = infer_concept(c)
        if concept is None or not pd.api.types.is_numeric_dtype(df[c]) or len(df) == 0:
            continue
        lo, hi, unit = PHYSICAL_RANGE[concept]
        s = df[c].dropna()
        if len(s) == 0:
            status, note = "UNVERIFIED", "kolom kosong"
        else:
            n_out = int(((s < lo) | (s > hi)).sum())
            frac_out = n_out / len(s)
            if frac_out == 0:
                status, note = "PLAUSIBLE", f"100% nilai dalam rentang fisik {lo}-{hi} {unit}"
            elif frac_out < 0.01:
                status, note = "PLAUSIBLE_WITH_OUTLIERS", f"{n_out} nilai ({frac_out:.3%}) di luar rentang fisik -> indikasi sensor fault"
            else:
                status, note = "UNVERIFIED", f"{frac_out:.1%} nilai di luar rentang fisik {lo}-{hi} {unit}; identitas kolom tidak terkonfirmasi"
        prov_rows.append({
            "dataset": dkey, "table": tkey, "column": str(c),
            "name_implies": concept, "assumed_unit": unit,
            "observed_min": float(s.min()) if len(s) else np.nan,
            "observed_max": float(s.max()) if len(s) else np.nan,
            "verification_status": status, "note": note,
        })

prov_df = pd.DataFrame(prov_rows)
hr("SECTION 8 — VERIFIKASI IDENTITAS KOLOM (bukan berdasarkan nama)")
if len(prov_df):
    print(prov_df[["dataset", "table", "column", "name_implies", "observed_min",
                   "observed_max", "verification_status"]].to_string(index=False))
    print()
    flagged = prov_df[prov_df.verification_status != "PLAUSIBLE"]
    if len(flagged):
        print("Kolom yang TIDAK lolos verifikasi polos:")
        for _, r in flagged.iterrows():
            print(f"  - {r['dataset']}/{r['table']}::{r['column']} -> {r['verification_status']}: {r['note']}")
else:
    print("Tidak ada kolom numerik yang namanya menyiratkan parameter kualitas air.")
save_csv(prov_df, DIR_RESULTS / "column_provenance_check.csv")


SECTION 8 — VERIFIKASI IDENTITAS KOLOM (bukan berdasarkan nama)
               dataset    table                    column name_implies  observed_min  observed_max     verification_status
          D1_AQUAPONDS     main                        PH           ph           NaN           NaN              UNVERIFIED
          D1_AQUAPONDS     main                      TEMP  temperature           NaN           NaN              UNVERIFIED
          D1_AQUAPONDS     main                        DO           do           NaN           NaN              UNVERIFIED
          D1_AQUAPONDS     main                 TURBIDITY    turbidity           NaN           NaN              UNVERIFIED
D2_FISHPOND_MULTICLASS     main                    orp_mV          orp   -249.600000    898.880000               PLAUSIBLE
D2_FISHPOND_MULTICLASS     main                   ec_uScm           ec      1.040000   4965.580000               PLAUSIBLE
D2_FISHPOND_MULTICLASS     main                   tds_mgL          tds     

## Cell 6 — DATASET 1: Aquaponds (Section 9)

Peran dataset **tidak** ditentukan dari nama file. Kita periksa isinya.


In [6]:
D1_FINDINGS = {"dataset": "D1_AQUAPONDS"}
hr("SECTION 9 — AUDIT DATASET 1 (AQUAPONDS)")

if "D1_AQUAPONDS" in RAW:
    d1 = RAW["D1_AQUAPONDS"]["main"]
    d1ne = RAW["D1_AQUAPONDS"]["main_nonempty"]
    print(f"Kolom terdaftar   : {list(d1.columns)}")
    print(f"Baris di file     : {len(d1)}")
    print(f"Baris berisi data : {len(d1ne)}")
    print(f"Sel non-null      : {int(d1.notna().sum().sum())}")
    D1_FINDINGS.update({
        "n_rows_in_file": int(len(d1)),
        "n_rows_with_data": int(len(d1ne)),
        "n_nonnull_cells": int(d1.notna().sum().sum()),
        "columns": list(map(str, d1.columns)),
        "has_pH": any("ph" == str(c).lower() for c in d1.columns),
        "has_DO": any(str(c).lower() == "do" for c in d1.columns),
        "has_temperature": any("temp" in str(c).lower() for c in d1.columns),
        "has_turbidity": any("turbid" in str(c).lower() for c in d1.columns),
        "has_tds_ec": any(("tds" in str(c).lower() or "ec" == str(c).lower()) for c in d1.columns),
        "has_timestamp": any(str(c).lower() in ("date", "time") for c in d1.columns),
        "has_site_id": any("station" in str(c).lower() for c in d1.columns),
        "has_risk_label": False,
        "has_fish_outcome": False,
    })
    if len(d1ne) == 0:
        D1_FINDINGS["verdict"] = "NOT SUITABLE"
        D1_FINDINGS["reason"] = (
            "File hanya berisi baris header. Nol observasi aktual. "
            "Skema kolom menjanjikan station/Date/Time/NITRATE/PH/AMMONIA/TEMP/DO/TURBIDITY/MANGANESE "
            "tetapi tidak ada satu pun nilai. Tidak dapat dipakai untuk training, validation, "
            "maupun distribution reference."
        )
        print("\n  *** VERDICT: NOT SUITABLE ***")
        print("  " + D1_FINDINGS["reason"])
        print("\n  Catatan: skema kolomnya justru berharga sebagai TEMPLATE pengambilan data lapangan")
        print("  RonaAir (station + timestamp + parameter). Dicatat di ronair_missing_data_plan.csv.")
    else:
        D1_FINDINGS["verdict"] = "REVIEW"
        D1_FINDINGS["reason"] = f"Terdapat {len(d1ne)} baris berisi data; perlu audit lanjutan."
        print(d1ne.describe(include="all").to_string())
else:
    D1_FINDINGS["verdict"] = "FILE NOT FOUND"
    D1_FINDINGS["reason"] = "File tidak ditemukan di lingkungan eksekusi."
    print("File D1 tidak ditemukan — dilewati.")


SECTION 9 — AUDIT DATASET 1 (AQUAPONDS)
Kolom terdaftar   : ['station', 'Date', 'Time', 'NITRATE(PPM)', 'PH', 'AMMONIA(mg/l)', 'TEMP', 'DO', 'TURBIDITY', 'MANGANESE(mg/l)']
Baris di file     : 124
Baris berisi data : 0
Sel non-null      : 0

  *** VERDICT: NOT SUITABLE ***
  File hanya berisi baris header. Nol observasi aktual. Skema kolom menjanjikan station/Date/Time/NITRATE/PH/AMMONIA/TEMP/DO/TURBIDITY/MANGANESE tetapi tidak ada satu pun nilai. Tidak dapat dipakai untuk training, validation, maupun distribution reference.

  Catatan: skema kolomnya justru berharga sebagai TEMPLATE pengambilan data lapangan
  RonaAir (station + timestamp + parameter). Dicatat di ronair_missing_data_plan.csv.


## Cell 7 — DATASET 2: fishpond_dataset_multiclass_2153 (Section 10)

Audit penuh + **forensik label**: apakah `class` merupakan label biologis independen,
atau sekadar turunan aturan threshold?


In [7]:
from sklearn.tree import DecisionTreeClassifier, export_text

D2_FINDINGS = {"dataset": "D2_FISHPOND_MULTICLASS"}
hr("SECTION 10 — AUDIT DATASET 2 (FISHPOND MULTICLASS)")

D2 = None
if "D2_FISHPOND_MULTICLASS" in RAW:
    D2 = RAW["D2_FISHPOND_MULTICLASS"]["main"].copy()
    D2["ts"] = pd.to_datetime(D2["timestamp"], errors="coerce")
    feat2 = ["orp_mV", "ec_uScm", "tds_mgL", "turbidity_NTU", "temp_C", "pH", "do_mgL"]

    print(f"n_samples        : {len(D2):,}")
    print(f"n_features       : {len(feat2)}  -> {feat2}")
    print(f"kolom lengkap    : {list(D2.columns)}")
    print(f"missing          : {int(D2.isna().sum().sum())}")
    print(f"duplicate rows   : {int(D2.duplicated().sum())}")
    print(f"duplicate sample : {int(D2['sample_id'].duplicated().sum())}")
    print(f"timestamp range  : {D2['ts'].min()}  ->  {D2['ts'].max()}")
    gaps = D2.sort_values("ts")["ts"].diff().dropna().value_counts()
    print(f"interval sampling: {gaps.index[0]} untuk {gaps.iloc[0]:,} dari {len(D2)-1:,} transisi")
    print(f"pond/site ID     : TIDAK ADA (hanya sample_id berurutan)")
    print("\nDistribusi kelas:")
    print(D2["class"].value_counts().to_string())
    print("\nRange per parameter:")
    print(D2[feat2].agg(["min", "max", "mean", "std"]).T.round(3).to_string())

    # ---- FORENSIK LABEL ----
    print("\n--- FORENSIK LABEL: apakah `class` rule-derived? ---")
    single_feature_recovery = {}
    for f in feat2:
        t = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_SEED).fit(D2[[f]], D2["class"])
        single_feature_recovery[f] = float(t.score(D2[[f]], D2["class"]))
    rec = pd.Series(single_feature_recovery).sort_values(ascending=False)
    print("Akurasi recovery label dari SATU fitur saja (decision tree depth<=4, fit pada seluruh data):")
    print(rec.round(4).to_string())

    top_feat = rec.index[0]
    top_acc = float(rec.iloc[0])
    perfect = [f for f, v in single_feature_recovery.items() if v >= 0.999]
    near_perfect = [f for f, v in single_feature_recovery.items() if 0.95 <= v < 0.999]
    D2_FINDINGS["label_recovery_single_feature"] = {k: round(v, 4) for k, v in single_feature_recovery.items()}
    D2_FINDINGS["label_driver"] = top_feat
    D2_FINDINGS["label_recovery_accuracy"] = round(top_acc, 4)
    D2_FINDINGS["features_that_fully_recover_label"] = perfect
    D2_FINDINGS["features_that_nearly_recover_label"] = near_perfect

    print(f"\n*** Fitur yang memulihkan label 100% SENDIRIAN: {perfect} ***")
    print(f"*** Fitur yang memulihkan >=95% sendirian   : {near_perfect} ***")
    print("Artinya SETIAP kanal parameter membawa informasi label secara penuh atau hampir penuh.")

    if top_acc >= 0.999:
        tree = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_SEED).fit(D2[[top_feat]], D2["class"])
        rule_text = export_text(tree, feature_names=[top_feat])
        print(f"\nAturan yang terpulihkan dari `{top_feat}` saja:")
        print(rule_text)
        bounds = D2.groupby("class")[top_feat].agg(["min", "max", "count"]).sort_values("min")
        print(f"Rentang nilai aktual `{top_feat}` per kelas:")
        print(bounds.round(3).to_string())

        # Struktur generatif: partisi terpisah, atau pita konsentris (nested)?
        b = bounds.sort_values("min")
        disjoint = all(b["max"].iloc[i] <= b["min"].iloc[i + 1] for i in range(len(b) - 1))
        widths = (bounds["max"] - bounds["min"]).sort_values()
        nested = all(
            bounds.loc[widths.index[i], "min"] >= bounds.loc[widths.index[i + 1], "min"] and
            bounds.loc[widths.index[i], "max"] <= bounds.loc[widths.index[i + 1], "max"]
            for i in range(len(widths) - 1))
        if disjoint:
            structure = "DISJOINT PARTITION (threshold berjenjang, rentang antar kelas tidak tumpang tindih)"
        elif nested:
            structure = ("NESTED CONCENTRIC BANDS (setiap kelas adalah pita jarak dari nilai optimal; "
                         "kelas yang lebih parah = pita yang lebih lebar dan melingkupi kelas ringan)")
        else:
            structure = "TIDAK TERIDENTIFIKASI SECARA SEDERHANA"
        print(f"\nStruktur generatif pada `{top_feat}`: {structure}")
        print("Urutan lebar rentang per kelas: " +
              ", ".join(f"{i}={w:.1f}" for i, w in widths.items()))

        D2_FINDINGS["recovered_rule"] = rule_text
        D2_FINDINGS["generative_structure"] = structure
        D2_FINDINGS["class_bounds_on_driver"] = {
            str(i): {"min": float(r["min"]), "max": float(r["max"]), "n": int(r["count"])}
            for i, r in bounds.iterrows()
        }
        D2_FINDINGS["label_type"] = ("RULE-DERIVED (label tertanam di SEMUA parameter sebagai "
                                     "pita rentang per kelas)")
    else:
        D2_FINDINGS["label_type"] = "RULE-DERIVED (multi-parameter) atau tidak terkonfirmasi"

    # Uji kedua: apakah SELURUH ruang fitur terpartisi per kelas?
    from sklearn.ensemble import RandomForestClassifier as _RF
    others = [f for f in feat2 if f != top_feat]
    _rf = _RF(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1).fit(D2[others], D2["class"])
    acc_wo = float(_rf.score(D2[others], D2["class"]))
    print(f"\nAkurasi recovery label TANPA `{top_feat}` (RandomForest, seluruh fitur lain): {acc_wo:.4f}")
    D2_FINDINGS["label_recovery_without_driver"] = round(acc_wo, 4)
    if acc_wo > 0.95:
        print("Label juga hampir sepenuhnya terpulihkan dari fitur lain. Ini menunjukkan setiap")
        print("kelas dibangkitkan dari KOTAK RENTANG tersendiri pada semua parameter sekaligus,")
        print("sehingga seluruh ruang fitur terpartisi menurut kelas. Ciri khas data sintetis.")

    # Apakah DO ikut menentukan label? (uji kontradiksi biologis)
    do_min_by_class = D2.groupby("class")["do_mgL"].min()
    hypoxic_rows = D2[D2["do_mgL"] < 3.0]["class"].value_counts().to_dict()
    print(f"\nUji kontradiksi biologis — baris dengan DO < 3.0 mg/L (hipoksia berat) diberi label:")
    print(f"  {hypoxic_rows}")
    print(f"DO minimum per kelas: {do_min_by_class.round(2).to_dict()}")
    D2_FINDINGS["hypoxic_rows_labelled_as"] = {str(k): int(v) for k, v in hypoxic_rows.items()}
    D2_FINDINGS["do_min_per_class"] = {str(k): float(v) for k, v in do_min_by_class.items()}

    D2_FINDINGS.update({
        "n_rows": int(len(D2)),
        "n_classes": int(D2["class"].nunique()),
        "classes": sorted(D2["class"].unique().tolist()),
        "class_distribution": {str(k): int(v) for k, v in D2["class"].value_counts().items()},
        "has_timestamp": True,
        "has_pond_id": False,
        "sampling_interval": str(gaps.index[0]),
        "timestamp_min": str(D2["ts"].min()),
        "timestamp_max": str(D2["ts"].max()),
        "missing": int(D2.isna().sum().sum()),
        "duplicates": int(D2.duplicated().sum()),
    })
    print(f"\nLABEL TYPE = {D2_FINDINGS['label_type']}")
    print("Label ini BUKAN independent biological ground truth.")
else:
    D2_FINDINGS["verdict"] = "FILE NOT FOUND"
    print("File D2 tidak ditemukan — dilewati.")


SECTION 10 — AUDIT DATASET 2 (FISHPOND MULTICLASS)
n_samples        : 2,153
n_features       : 7  -> ['orp_mV', 'ec_uScm', 'tds_mgL', 'turbidity_NTU', 'temp_C', 'pH', 'do_mgL']
kolom lengkap    : ['sample_id', 'timestamp', 'orp_mV', 'ec_uScm', 'tds_mgL', 'turbidity_NTU', 'temp_C', 'pH', 'do_mgL', 'class', 'ts']
missing          : 0
duplicate rows   : 0
duplicate sample : 0
timestamp range  : 2026-01-01 00:00:00  ->  2026-01-08 11:20:00
interval sampling: 0 days 00:05:00 untuk 2,152 dari 2,152 transisi
pond/site ID     : TIDAK ADA (hanya sample_id berurutan)

Distribusi kelas:
class
Caution    670
Normal     638
Warning    478
Severe     367

Range per parameter:
                  min      max     mean       std
orp_mV        -249.60   898.88  291.254   261.019
ec_uScm          1.04  4965.58  655.756  1025.007
tds_mgL          0.00  3267.23  359.809   563.782
turbidity_NTU    1.02   598.68  101.353   138.438
temp_C          15.00    37.99   27.437     5.193
pH               4.51    10.5

## Cell 8 — DATASET 3: Data_Model_IoTMLCQ_2024.xlsx (Section 11)

Audit file aktual (bukan deskripsi internet), termasuk uji **derived label / target leakage**
dan uji apakah deret waktu benar-benar berisi observasi independen.


In [8]:
D3_FINDINGS = {"dataset": "D3_IOTMLCQ_2024"}
hr("SECTION 11 — AUDIT DATASET 3 (IoTMLCQ 2024)")

D3 = None
if "D3_IOTMLCQ_2024" in RAW:
    sheet_name = list(RAW["D3_IOTMLCQ_2024"])[0]
    D3 = RAW["D3_IOTMLCQ_2024"][sheet_name].copy()
    print(f"Sheet            : {sheet_name}")
    print(f"shape            : {D3.shape}")
    print(f"Datetime range   : {D3['Datetime'].min()} -> {D3['Datetime'].max()}")
    print(f"duplicate rows   : {int(D3.duplicated().sum())}")
    print(f"site/pond column : TIDAK ADA")

    print("\nKolom yang disebut master prompt vs yang benar-benar ada di file:")
    expected = ["Datetime", "Month", "Average Fish Weight", "Survival Rate", "Disease Occurrence",
                "Temperature", "Dissolved Oxygen", "pH", "Turbidity", "Oxygenation Automatic",
                "Oxygenation Interventions", "Corrective Interventions", "Thermal Risk Index",
                "Low Oxygen Alert", "Health Status"]
    actual_l = [str(c) for c in D3.columns]
    for e in expected:
        hit = [c for c in actual_l if c.lower().startswith(e.lower())]
        print(f"  {e:30s} -> {hit if hit else 'TIDAK ADA'}")
    extra = [c for c in actual_l if not any(c.lower().startswith(e.lower()) for e in expected)]
    print(f"  Kolom tambahan di file (tidak disebut prompt): {extra}")

    # ---- variabilitas target ----
    print("\nVariabilitas kandidat label:")
    label_cols = ["Thermal Risk Index", "Low Oxygen Alert", "Health Status"]
    for c in label_cols:
        if c in D3.columns:
            vc = D3[c].value_counts()
            print(f"  {c:20s}: {vc.to_dict()}")
            if D3[c].nunique() <= 1:
                print(f"      !! KONSTAN — nol informasi, tidak dapat dijadikan target.")

    # ---- uji kesetaraan label (leakage) ----
    print("\n--- UJI DERIVED LABEL / TARGET LEAKAGE ---")
    leak_notes = []
    if "Health Status" in D3.columns and "Thermal Risk Index" in D3.columns:
        ct = pd.crosstab(D3["Thermal Risk Index"], D3["Health Status"])
        print("Crosstab Thermal Risk Index x Health Status:")
        print(ct.to_string())
        # apakah pemetaan 1:1 deterministik?
        deterministic = (ct > 0).sum(axis=1).max() == 1 and (ct > 0).sum(axis=0).max() == 1
        print(f"Pemetaan deterministik 1:1? {deterministic}")
        if deterministic:
            leak_notes.append("Health Status adalah fungsi deterministik 1:1 dari Thermal Risk Index "
                              "(bukan outcome independen).")
        D3_FINDINGS["health_status_equals_thermal_risk"] = bool(deterministic)

    if "Thermal Risk Index" in D3.columns and "Temperature (°C)" in D3.columns:
        g = D3.groupby("Thermal Risk Index")["Temperature (°C)"].agg(["min", "max"])
        print("\nRentang suhu per nilai Thermal Risk Index:")
        print(g.round(4).to_string())
        # cari threshold pemisah
        hi_label = g["min"].idxmax()
        lo_label = g["max"].idxmin()
        thr = None
        if g.loc[hi_label, "min"] > g.loc[lo_label, "max"]:
            thr = float((g.loc[hi_label, "min"] + g.loc[lo_label, "max"]) / 2)
            print(f"Kelas terpisah sempurna oleh suhu. Threshold implisit ~ {thr:.3f} deg C "
                  f"('{hi_label}' jika suhu di atasnya).")
            leak_notes.append(f"Thermal Risk Index adalah threshold deterministik pada suhu (~{thr:.2f} C), "
                              f"bukan label ahli maupun outcome terobservasi.")
        D3_FINDINGS["thermal_risk_threshold_on_temperature"] = thr

    # ---- uji independensi observasi (interpolasi?) ----
    wq = [c for c in ["Temperature (°C)", "Dissolved Oxygen (mg/L)", "pH", "Turbidity (NTU)"] if c in D3.columns]
    if wq:
        uniq_combo = D3[wq].drop_duplicates().shape[0]
        print(f"\nObservasi kualitas air unik: {uniq_combo} kombinasi dari {len(D3):,} baris "
              f"({uniq_combo/len(D3):.2%})")
        # apakah deretnya interpolasi linier? -> selisih berturut-turut nyaris konstan per segmen
        s = D3.sort_values("Datetime")[wq[0]]
        d1_ = s.diff().dropna()
        nz = d1_[d1_.abs() > 1e-12]
        frac_repeat_step = float(nz.round(6).value_counts(normalize=True).iloc[0]) if len(nz) else 0.0
        print(f"Fraksi step selisih '{wq[0]}' yang identik (indikator interpolasi linier): {frac_repeat_step:.2%}")
        seg = D3.sort_values("Datetime")[wq].round(6)
        n_anchor_blocks = int((seg.diff().round(6) != seg.diff().round(6).shift()).any(axis=1).sum())
        print(f"Perkiraan jumlah segmen linier (anchor): ~{n_anchor_blocks}")
        D3_FINDINGS["unique_wq_observations"] = int(uniq_combo)
        D3_FINDINGS["interpolation_step_repeat_fraction"] = round(frac_repeat_step, 4)
        if frac_repeat_step > 0.3:
            leak_notes.append("Kolom kualitas air adalah hasil interpolasi linier antar sedikit titik jangkar; "
                              "jumlah observasi independen efektif JAUH lebih kecil dari jumlah baris.")

    # ---- outcome: independen atau kolinear? ----
    if "Survival Rate (%)" in D3.columns and "Temperature (°C)" in D3.columns:
        r = float(np.corrcoef(D3["Survival Rate (%)"], D3["Temperature (°C)"])[0, 1])
        print(f"\nKorelasi Survival Rate vs Temperature: r = {r:+.4f}")
        D3_FINDINGS["corr_survival_vs_temperature"] = round(r, 4)
        if abs(r) > 0.95:
            leak_notes.append("Survival Rate hampir kolinear sempurna dengan Temperature -> "
                              "bukan outcome yang terobservasi independen.")

    D3_FINDINGS.update({
        "n_rows": int(len(D3)),
        "n_columns": int(D3.shape[1]),
        "datetime_min": str(D3["Datetime"].min()),
        "datetime_max": str(D3["Datetime"].max()),
        "has_pond_id": False,
        "constant_label_columns": [c for c in label_cols if c in D3.columns and D3[c].nunique() <= 1],
        "leakage_notes": leak_notes,
        "label_type": "DERIVED LABEL / POTENTIAL TARGET LEAKAGE",
    })
    print("\n*** LABEL TYPE = DERIVED LABEL / POTENTIAL TARGET LEAKAGE ***")
    for n in leak_notes:
        print(f"  - {n}")
    print("\nKonsekuensi: Thermal Risk Index, Low Oxygen Alert, dan Health Status TIDAK BOLEH")
    print("dipakai bersamaan sebagai predictor+target, dan Health Status tidak boleh disebut")
    print("sebagai outcome kesehatan ikan yang terobservasi.")
else:
    D3_FINDINGS["verdict"] = "FILE NOT FOUND"
    print("File D3 tidak ditemukan — dilewati.")


SECTION 11 — AUDIT DATASET 3 (IoTMLCQ 2024)
Sheet            : Sheet1
shape            : (4383, 28)
Datetime range   : 2024-01-01 00:00:00 -> 2024-07-01 14:00:00
duplicate rows   : 0
site/pond column : TIDAK ADA

Kolom yang disebut master prompt vs yang benar-benar ada di file:
  Datetime                       -> ['Datetime']
  Month                          -> ['Month', 'Month_Num', 'month_x', 'month_y']
  Average Fish Weight            -> ['Average Fish Weight (g)']
  Survival Rate                  -> ['Survival Rate (%)']
  Disease Occurrence             -> ['Disease Occurrence (Cases)']
  Temperature                    -> ['Temperature (°C)']
  Dissolved Oxygen               -> ['Dissolved Oxygen (mg/L)']
  pH                             -> ['pH', 'ph']
  Turbidity                      -> ['Turbidity (NTU)']
  Oxygenation Automatic          -> ['Oxygenation Automatic']
  Oxygenation Interventions      -> ['Oxygenation Interventions']
  Corrective Interventions       -> ['Corrective

## Cell 9 — DATASET 4: Aquaponic Fish Pond IoT (Section 12)

Audit raw vs filtered, termasuk **uji reproduksi aturan filter** untuk mendeteksi
range-filter bias / circularity.


In [9]:
D4_FINDINGS = {"dataset": "D4_AQUAPONIC_IOT"}
hr("SECTION 12 — AUDIT DATASET 4 (AQUAPONIC IoT)")

D4_RAW = D4_FILT = None
if "D4_AQUAPONIC_IOT" in RAW:
    tabs = RAW["D4_AQUAPONIC_IOT"]
    D4_RAW = tabs.get("raw")
    D4_FILT = tabs.get("filtered")
    for nm, df in [("raw", D4_RAW), ("filtered", D4_FILT)]:
        if df is None:
            continue
        df["ts"] = pd.to_datetime(df["created_date"], format="%m/%d/%Y %H:%M", errors="coerce")
        print(f"\n--- {nm} ---")
        print(f"  n_rows        : {len(df):,}")
        print(f"  kolom         : {[c for c in df.columns if c != 'ts']}")
        print(f"  ts range      : {df['ts'].min()} -> {df['ts'].max()}  (NaT: {int(df['ts'].isna().sum())})")
        print(f"  missing       : {int(df.drop(columns=['ts']).isna().sum().sum())}")
        print(f"  duplicate rows: {int(df.duplicated(subset=[c for c in df.columns if c != 'ts']).sum())}")
        print(df[["water_pH", "TDS", "water_temp"]].agg(["min", "max", "mean", "std"]).T.round(3).to_string())
        per_min = df.groupby("ts").size()
        print(f"  observasi/menit: median {per_min.median():.0f}, max {per_min.max()}, "
              f"menit unik {df['ts'].nunique():,}")

    print(f"\n  Jumlah pond/site: 1 (tidak ada kolom site/pond/device; satu aliran sensor)")
    print(f"  Parameter tersedia: pH, TDS, suhu.  TIDAK ADA: DO, EC terpisah, turbidity, label risiko.")

    # ---- deteksi sensor fault di raw ----
    n_ph_impossible = int((D4_RAW["water_pH"] > 14).sum() + (D4_RAW["water_pH"] < 0).sum())
    print(f"\n  Sensor fault di raw: {n_ph_impossible} baris dengan pH di luar 0-14 "
          f"({n_ph_impossible/len(D4_RAW):.3%}) -> bukti nyata kegagalan pembacaan sensor.")

    # ---- uji reproduksi aturan filter ----
    print("\n--- UJI REPRODUKSI ATURAN FILTER (circularity / range-filter bias) ---")
    subset_ok = bool(D4_FILT["id"].isin(D4_RAW["id"]).all())
    print(f"  Semua id filtered adalah subset raw? {subset_ok}")
    ph_lo, ph_hi = float(D4_FILT["water_pH"].min()), float(D4_FILT["water_pH"].max())
    t_lo, t_hi = float(D4_FILT["water_temp"].min()), float(D4_FILT["water_temp"].max())
    tds_lo, tds_hi = float(D4_FILT["TDS"].min()), float(D4_FILT["TDS"].max())
    mask = (D4_RAW["water_pH"].between(ph_lo, ph_hi) &
            D4_RAW["water_temp"].between(t_lo, t_hi) &
            D4_RAW["TDS"].between(tds_lo, tds_hi))
    n_rule = int(mask.sum())
    exact = (n_rule == len(D4_FILT))
    print(f"  Batas filtered: pH [{ph_lo}, {ph_hi}], temp [{t_lo}, {t_hi}], TDS [{tds_lo}, {tds_hi}]")
    print(f"  Baris raw yang memenuhi batas tsb: {n_rule:,}   |   baris filtered: {len(D4_FILT):,}")
    print(f"  Aturan filter tereproduksi persis? {exact}")
    if exact:
        print("  *** POTENSI CIRCULARITY / RANGE-FILTER BIAS TERKONFIRMASI ***")
        print("  File 'filtered' dibentuk dengan MEMBUANG semua pembacaan di luar rentang 'optimal'.")
        print("  Melatih atau mengevaluasi model risiko pada file ini akan menghapus justru kondisi")
        print("  yang ingin dideteksi RonaAir. Untuk RonaAir, file RAW lebih informatif.")
    retain = len(D4_FILT) / len(D4_RAW)
    print(f"  Retensi data: {retain:.2%} (dibuang {1-retain:.2%})")

    D4_FINDINGS.update({
        "n_rows_raw": int(len(D4_RAW)),
        "n_rows_filtered": int(len(D4_FILT)),
        "n_sites_ponds": 1,
        "has_timestamp": True,
        "ts_min": str(D4_RAW["ts"].min()),
        "ts_max": str(D4_RAW["ts"].max()),
        "parameters": ["water_pH", "TDS", "water_temp"],
        "parameters_absent": ["dissolved_oxygen", "turbidity", "EC", "risk_label", "fish_outcome"],
        "median_obs_per_minute": float(D4_RAW.groupby("ts").size().median()),
        "ph_out_of_physical_range_rows": n_ph_impossible,
        "filter_rule_reproduced_exactly": bool(exact),
        "filter_bounds": {"pH": [ph_lo, ph_hi], "water_temp": [t_lo, t_hi], "TDS": [tds_lo, tds_hi]},
        "retention_rate": round(retain, 4),
        "label_type": "NO LABEL (unlabelled sensor stream)",
    })
else:
    D4_FINDINGS["verdict"] = "FILE NOT FOUND"
    print("File D4 tidak ditemukan — dilewati.")

save_json({"D1": D1_FINDINGS, "D2": D2_FINDINGS, "D3": D3_FINDINGS, "D4": D4_FINDINGS},
          DIR_RESULTS / "dataset_deep_audit.json")


SECTION 12 — AUDIT DATASET 4 (AQUAPONIC IoT)

--- raw ---
  n_rows        : 505,730
  kolom         : ['id', 'created_date', 'water_pH', 'TDS', 'water_temp']
  ts range      : 2023-01-26 10:38:00 -> 2023-03-22 17:53:00  (NaT: 0)
  missing       : 0
  duplicate rows: 0
               min      max     mean      std
water_pH      5.51    15.78    7.900    1.386
TDS         200.00  1813.00  388.979  137.266
water_temp   21.63    27.19   24.187    0.973
  observasi/menit: median 9, max 32, menit unik 48,081

--- filtered ---
  n_rows        : 118,286
  kolom         : ['id', 'created_date', 'water_pH', 'TDS', 'water_temp']
  ts range      : 2023-01-26 11:24:00 -> 2023-03-22 17:53:00  (NaT: 0)
  missing       : 0
  duplicate rows: 0
              min     max     mean     std
water_pH      6.5    8.50    7.508   0.556
TDS         241.0  498.00  335.416  43.739
water_temp   24.0   26.19   24.656   0.452
  observasi/menit: median 4, max 24, menit unik 19,509

  Jumlah pond/site: 1 (tidak ada ko

## Cell 10 — Sistem budidaya & relevansi domain (Section 36)

Aquaponics tidak otomatis dibuang, tetapi juga tidak otomatis disamakan dengan kolam
budidaya air tawar. Klasifikasi dilakukan dari bukti dalam file + judul dataset.


In [10]:
SYSTEM_TABLE = pd.DataFrame([
    {"dataset": "D1_AQUAPONDS", "system_type": "UNKNOWN (aquaponics/pond, tidak dapat diverifikasi)",
     "water_type": "UNKNOWN", "species": "TIDAK DILAPORKAN", "sensor_setup": "TIDAK DILAPORKAN",
     "evidence": "File kosong; hanya header. Tidak ada bukti isi.",
     "relevance_to_ronaair": "NONE"},
    {"dataset": "D2_FISHPOND_MULTICLASS", "system_type": "SYNTHETIC / SIMULATED fish pond",
     "water_type": "Freshwater (implisit)", "species": "TIDAK DILAPORKAN",
     "sensor_setup": "TIDAK DILAPORKAN (7 parameter, grid 5 menit sempurna, timestamp 2026)",
     "evidence": "Timestamp berada di masa depan, interval 5 menit tanpa satu pun gap, "
                 "batas kelas tidak tumpang tindih pada satu parameter -> data dibangkitkan.",
     "relevance_to_ronaair": "MEDIUM — struktur parameter mirip, tetapi bukan pengukuran lapangan"},
    {"dataset": "D3_IOTMLCQ_2024", "system_type": "Modelled/simulated aquaculture pond (IoT+ML study)",
     "water_type": "Freshwater", "species": "TIDAK DILAPORKAN (ada bobot ikan & survival rate)",
     "sensor_setup": "Gabungan agregat bulanan + deret skala per jam",
     "evidence": "Kolom kualitas air terinterpolasi linier; label = threshold suhu deterministik.",
     "relevance_to_ronaair": "LOW — struktur target tidak dapat dipakai"},
    {"dataset": "D4_AQUAPONIC_IOT", "system_type": "AQUAPONICS (fish pond dalam sistem aquaponik)",
     "water_type": "Freshwater", "species": "TIDAK DILAPORKAN",
     "sensor_setup": "IoT nyata: pH, TDS, suhu; sampling sub-menit; terdapat sensor fault",
     "evidence": "Nilai pH >14 pada raw, laju sampling tidak seragam, drift -> ciri sensor lapangan asli.",
     "relevance_to_ronaair": "HIGH untuk perilaku sensor — aquaponics mengubah kimia air "
                             "(nitrifikasi, tanaman menyerap nutrien) sehingga distribusi TDS/pH "
                             "tidak dapat dianggap identik dengan kolam budidaya biasa"},
])
hr("SECTION 36 — SISTEM BUDIDAYA & RELEVANSI DOMAIN")
print(SYSTEM_TABLE.to_string(index=False, max_colwidth=60))
save_csv(SYSTEM_TABLE, DIR_RESULTS / "dataset_system_classification.csv")


SECTION 36 — SISTEM BUDIDAYA & RELEVANSI DOMAIN
               dataset                                         system_type            water_type                                           species                                                 sensor_setup                                                     evidence                                         relevance_to_ronaair
          D1_AQUAPONDS UNKNOWN (aquaponics/pond, tidak dapat diverifikasi)               UNKNOWN                                  TIDAK DILAPORKAN                                             TIDAK DILAPORKAN              File kosong; hanya header. Tidak ada bukti isi.                                                         NONE
D2_FISHPOND_MULTICLASS                     SYNTHETIC / SIMULATED fish pond Freshwater (implisit)                                  TIDAK DILAPORKAN TIDAK DILAPORKAN (7 parameter, grid 5 menit sempurna, tim... Timestamp berada di masa depan, interval 5 menit tanpa sa... MEDIUM — struktur param

## Cell 11 — Role assignment dataset (Section 13)

Peran ditentukan **dari hasil audit**, bukan dari nama file. Tidak semua dataset dipaksa
masuk training.


In [11]:
def _d2_ok():
    return D2 is not None and len(D2) > 100

ROLE_ROWS = [
    {
        "dataset": "D1_AQUAPONDS",
        "role": "NOT SUITABLE",
        "training": False, "validation": False, "external_test": False,
        "auxiliary": False, "reference": True,
        "reason": f"Nol observasi ({D1_FINDINGS.get('n_rows_with_data', 'n/a')} baris berisi data). "
                  f"Hanya header.",
        "limitations": "Tidak ada data. Skema kolomnya dipakai sebagai template rencana "
                       "pengambilan data lapangan RonaAir, bukan sebagai data.",
    },
    {
        "dataset": "D2_FISHPOND_MULTICLASS",
        "role": "RULE-EMULATION BENCHMARK (training terbatas + internal test)",
        "training": bool(_d2_ok()), "validation": bool(_d2_ok()), "external_test": False,
        "auxiliary": True, "reference": True,
        "reason": f"Satu-satunya dataset berlabel 4 kelas dengan timestamp. Namun label "
                  f"tereproduksi {D2_FINDINGS.get('label_recovery_accuracy', float('nan')):.2%} "
                  f"hanya dari `{D2_FINDINGS.get('label_driver')}` -> label adalah aturan threshold, "
                  f"bukan ground truth biologis.",
        "limitations": "Data bersifat sintetis (timestamp 2026, grid sempurna, batas kelas tidak "
                       "tumpang tindih). Model apa pun yang dilatih di sini hanya meniru aturan "
                       "threshold turbidity, dan TIDAK memprediksi risiko biologis nyata. "
                       "Tidak ada pond ID -> tidak mungkin group split.",
    },
    {
        "dataset": "D3_IOTMLCQ_2024",
        "role": "REFERENCE ONLY (tidak untuk training)",
        "training": False, "validation": False, "external_test": False,
        "auxiliary": False, "reference": True,
        "reason": "Health Status = fungsi 1:1 dari Thermal Risk Index; Low Oxygen Alert konstan "
                  f"('{list(D3['Low Oxygen Alert'].unique()) if D3 is not None and 'Low Oxygen Alert' in D3.columns else 'n/a'}'); "
                  f"hanya ~{D3_FINDINGS.get('unique_wq_observations', 'n/a')} observasi kualitas air unik "
                  f"pada {D3_FINDINGS.get('n_rows', 'n/a')} baris karena interpolasi linier.",
        "limitations": "Jumlah observasi independen efektif sangat kecil. Rentang suhu hanya "
                       "~26.5-28.4 C dan DO tidak pernah rendah -> tidak memuat kondisi berisiko "
                       "yang ingin dideteksi RonaAir. Outcome (survival, bobot) kolinear dengan suhu.",
    },
    {
        "dataset": "D4_AQUAPONIC_IOT",
        "role": "AUXILIARY — sensor realism, distribusi operasional, OOD & fault modelling",
        "training": False, "validation": False, "external_test": False,
        "auxiliary": True, "reference": True,
        "reason": "Satu-satunya data sensor lapangan asli (pH/TDS/suhu, ~2 bulan). Tidak berlabel "
                  "sehingga tidak dapat dipakai sebagai training risk classifier, tetapi menjadi "
                  "basis empiris untuk plausible-range, laju perubahan, dan deteksi sensor fault.",
        "limitations": "Sistem aquaponik (kimia air berbeda dari kolam budidaya murni), satu pond, "
                       "tanpa DO/turbidity/label, dan varian 'filtered' bersifat sirkular "
                       "(dibentuk dengan membuang nilai di luar rentang optimal).",
    },
]
ROLE_DF = pd.DataFrame(ROLE_ROWS)
ROLE_DF_OUT = ROLE_DF.rename(columns={
    "training": "usable_for_training", "validation": "usable_for_validation",
    "external_test": "usable_for_external_test"})

hr("SECTION 13 — DATASET ROLE ASSIGNMENT")
print(ROLE_DF[["dataset", "role", "training", "validation", "external_test",
               "auxiliary", "reference"]].to_string(index=False))
print()
for r in ROLE_ROWS:
    print(f"* {r['dataset']}\n    ALASAN     : {r['reason']}\n    LIMITASI   : {r['limitations']}\n")
save_csv(ROLE_DF_OUT[["dataset", "role", "reason", "usable_for_training", "usable_for_validation",
                      "usable_for_external_test", "auxiliary", "reference", "limitations"]],
         DIR_RESULTS / "dataset_role_decision.csv")

TRAINING_DATASET = "D2_FISHPOND_MULTICLASS" if _d2_ok() else None
print(f"Dataset training terpilih: {TRAINING_DATASET}")


SECTION 13 — DATASET ROLE ASSIGNMENT
               dataset                                                                      role  training  validation  external_test  auxiliary  reference
          D1_AQUAPONDS                                                              NOT SUITABLE     False       False          False      False       True
D2_FISHPOND_MULTICLASS              RULE-EMULATION BENCHMARK (training terbatas + internal test)      True        True          False       True       True
       D3_IOTMLCQ_2024                                     REFERENCE ONLY (tidak untuk training)     False       False          False      False       True
      D4_AQUAPONIC_IOT AUXILIARY — sensor realism, distribusi operasional, OOD & fault modelling     False       False          False       True       True

* D1_AQUAPONDS
    ALASAN     : Nol observasi (0 baris berisi data). Hanya header.
    LIMITASI   : Tidak ada data. Skema kolomnya dipakai sebagai template rencana pengambilan data l

## Cell 12 — Domain shift analysis (Section 35)

Perbandingan distribusi parameter yang **benar-benar ada di lebih dari satu dataset**.
Dataset yang berbeda tidak dianggap identik.


In [12]:
HARMONISED = {}   # dataset -> DataFrame dengan nama parameter kanonik

if D2 is not None:
    HARMONISED["D2_FISHPOND_MULTICLASS"] = pd.DataFrame({
        "water_temp": D2["temp_C"], "ph_sensor": D2["pH"], "do_mgL": D2["do_mgL"],
        "ec_uScm": D2["ec_uScm"], "tds_ppm": D2["tds_mgL"], "turbidity_NTU": D2["turbidity_NTU"],
    })
if D3 is not None:
    HARMONISED["D3_IOTMLCQ_2024"] = pd.DataFrame({
        "water_temp": D3["Temperature (°C)"], "ph_sensor": D3["pH"],
        "do_mgL": D3["Dissolved Oxygen (mg/L)"], "turbidity_NTU": D3["Turbidity (NTU)"],
    })
if D4_RAW is not None:
    HARMONISED["D4_AQUAPONIC_IOT_raw"] = pd.DataFrame({
        "water_temp": D4_RAW["water_temp"], "ph_sensor": D4_RAW["water_pH"], "tds_ppm": D4_RAW["TDS"],
    })
    HARMONISED["D4_AQUAPONIC_IOT_filtered"] = pd.DataFrame({
        "water_temp": D4_FILT["water_temp"], "ph_sensor": D4_FILT["water_pH"], "tds_ppm": D4_FILT["TDS"],
    })

PARAMS = ["water_temp", "ph_sensor", "do_mgL", "ec_uScm", "tds_ppm", "turbidity_NTU"]

hr("SECTION 35 — DOMAIN SHIFT ANALYSIS")
shift_rows = []
for p in PARAMS:
    present = {k: v[p].dropna() for k, v in HARMONISED.items() if p in v.columns}
    if not present:
        continue
    print(f"\n--- {p} --- tersedia di: {list(present)}")
    for k, s in present.items():
        print(f"  {k:32s} n={len(s):>7,}  min={s.min():>9.2f}  p05={s.quantile(.05):>9.2f}  "
              f"median={s.median():>9.2f}  p95={s.quantile(.95):>9.2f}  max={s.max():>9.2f}")
    keys = list(present)
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            a, b = present[keys[i]], present[keys[j]]
            row = {"parameter": p, "dataset_a": keys[i], "dataset_b": keys[j],
                   "n_a": len(a), "n_b": len(b),
                   "median_a": round(float(a.median()), 4), "median_b": round(float(b.median()), 4),
                   "iqr_a": round(float(a.quantile(.75) - a.quantile(.25)), 4),
                   "iqr_b": round(float(b.quantile(.75) - b.quantile(.25)), 4),
                   "range_overlap_frac": None, "ks_stat": None, "ks_pvalue": None,
                   "wasserstein": None}
            lo = max(a.min(), b.min()); hi = min(a.max(), b.max())
            tot_lo = min(a.min(), b.min()); tot_hi = max(a.max(), b.max())
            row["range_overlap_frac"] = round(float(max(0, hi - lo) / (tot_hi - tot_lo)), 4) if tot_hi > tot_lo else 1.0
            if HAS_SCIPY:
                sa = a.sample(min(len(a), 20000), random_state=RANDOM_SEED)
                sb = b.sample(min(len(b), 20000), random_state=RANDOM_SEED)
                ks = sp_stats.ks_2samp(sa, sb)
                row["ks_stat"] = round(float(ks.statistic), 4)
                row["ks_pvalue"] = float(ks.pvalue)
                row["wasserstein"] = round(float(sp_stats.wasserstein_distance(sa, sb)), 4)
            shift_rows.append(row)

shift_df = pd.DataFrame(shift_rows)
if len(shift_df):
    print("\nRingkasan pasangan distribusi:")
    print(shift_df.to_string(index=False))
    big = shift_df[(shift_df.ks_stat.fillna(0) > 0.5)]
    if len(big):
        print("\nPerbedaan distribusi BESAR (KS > 0.5) — dataset ini tidak boleh dianggap setara:")
        for _, r in big.iterrows():
            print(f"  {r['parameter']:14s}: {r['dataset_a']} vs {r['dataset_b']}  KS={r['ks_stat']:.3f}")
save_csv(shift_df, DIR_RESULTS / "domain_shift_report.csv")

# --- Figur perbandingan distribusi ---
plot_params = [p for p in PARAMS if sum(p in v.columns for v in HARMONISED.values()) >= 2]
if plot_params:
    fig, axes = plt.subplots(1, len(plot_params), figsize=(4.2 * len(plot_params), 3.6))
    axes = np.atleast_1d(axes)
    for ax, p in zip(axes, plot_params):
        data, labels = [], []
        for k, v in HARMONISED.items():
            if p in v.columns:
                s = v[p].dropna()
                data.append(s.sample(min(len(s), 20000), random_state=RANDOM_SEED))
                labels.append(k.replace("D4_AQUAPONIC_IOT_", "D4_").replace("_FISHPOND_MULTICLASS", "")
                              .replace("_IOTMLCQ_2024", ""))
        try:
            ax.boxplot(data, tick_labels=labels, showfliers=False)   # matplotlib >= 3.9
        except TypeError:
            ax.boxplot(data, labels=labels, showfliers=False)        # matplotlib lama
        ax.set_title(p)
        ax.tick_params(axis="x", rotation=45, labelsize=8)
    plt.tight_layout()
    plt.savefig(DIR_FIGS / "domain_shift_boxplots.png", dpi=130)
    plt.close()
    print(f"  [saved] figures/domain_shift_boxplots.png")


SECTION 35 — DOMAIN SHIFT ANALYSIS

--- water_temp --- tersedia di: ['D2_FISHPOND_MULTICLASS', 'D3_IOTMLCQ_2024', 'D4_AQUAPONIC_IOT_raw', 'D4_AQUAPONIC_IOT_filtered']
  D2_FISHPOND_MULTICLASS           n=  2,153  min=    15.00  p05=    17.95  median=    27.53  p95=    36.10  max=    37.99
  D3_IOTMLCQ_2024                  n=  4,383  min=    26.50  p05=    26.50  median=    27.45  p95=    28.35  max=    28.35
  D4_AQUAPONIC_IOT_raw             n=505,730  min=    21.63  p05=    22.63  median=    24.19  p95=    25.88  max=    27.19
  D4_AQUAPONIC_IOT_filtered        n=118,286  min=    24.00  p05=    24.06  median=    24.56  p95=    25.50  max=    26.19

--- ph_sensor --- tersedia di: ['D2_FISHPOND_MULTICLASS', 'D3_IOTMLCQ_2024', 'D4_AQUAPONIC_IOT_raw', 'D4_AQUAPONIC_IOT_filtered']
  D2_FISHPOND_MULTICLASS           n=  2,153  min=     4.51  p05=     5.11  median=     7.57  p95=     9.96  max=    10.50
  D3_IOTMLCQ_2024                  n=  4,383  min=     7.34  p05=     7.34  median=    

## Cell 13 — Mapping taksonomi label (Section 14)

Mapping **tidak** dilakukan hanya karena jumlah kelasnya sama-sama empat.
Mapping dilakukan karena kedua skema adalah **tangga severity ordinal yang monoton**,
dan didokumentasikan lengkap dengan alasannya.


In [13]:
LABEL_MAPPING = [
    {"source_dataset": "D2_FISHPOND_MULTICLASS", "source_label": "Normal", "ronair_label": "NORMAL",
     "source_severity_rank": 0, "ronair_severity_rank": 0,
     "mapping_reason": "Kedua skema menempatkan tingkat ini sebagai kondisi operasi tanpa tindakan.",
     "mapping_status": "JUSTIFIED (ordinal, monoton)"},
    {"source_dataset": "D2_FISHPOND_MULTICLASS", "source_label": "Caution", "ronair_label": "WASPADA",
     "source_severity_rank": 1, "ronair_severity_rank": 1,
     "mapping_reason": "Tingkat kedua pada tangga severity; keduanya berarti 'amati lebih sering'.",
     "mapping_status": "JUSTIFIED (ordinal, monoton)"},
    {"source_dataset": "D2_FISHPOND_MULTICLASS", "source_label": "Warning", "ronair_label": "SIAGA",
     "source_severity_rank": 2, "ronair_severity_rank": 2,
     "mapping_reason": "Tingkat ketiga; keduanya menandakan perlu tindakan korektif.",
     "mapping_status": "JUSTIFIED (ordinal, monoton)"},
    {"source_dataset": "D2_FISHPOND_MULTICLASS", "source_label": "Severe", "ronair_label": "DARURAT",
     "source_severity_rank": 3, "ronair_severity_rank": 3,
     "mapping_reason": "Tingkat tertinggi; keduanya menandakan intervensi segera.",
     "mapping_status": "JUSTIFIED (ordinal, monoton)"},
    {"source_dataset": "D3_IOTMLCQ_2024", "source_label": "Stable / At Risk", "ronair_label": "TIDAK DIPETAKAN",
     "source_severity_rank": np.nan, "ronair_severity_rank": np.nan,
     "mapping_reason": "Hanya 2 tingkat, dibentuk dari threshold suhu tunggal, dan identik 1:1 dengan "
                       "Thermal Risk Index. Memetakannya ke 4 tingkat RonaAir akan mengarang informasi.",
     "mapping_status": "REJECTED"},
    {"source_dataset": "D4_AQUAPONIC_IOT", "source_label": "(tidak ada label)", "ronair_label": "TIDAK DIPETAKAN",
     "source_severity_rank": np.nan, "ronair_severity_rank": np.nan,
     "mapping_reason": "Dataset tidak berlabel.", "mapping_status": "NOT APPLICABLE"},
]
LABEL_MAP_DF = pd.DataFrame(LABEL_MAPPING)
D2_TO_RONAAIR = {r["source_label"]: r["ronair_label"] for r in LABEL_MAPPING
                 if r["source_dataset"] == "D2_FISHPOND_MULTICLASS"}

hr("SECTION 14 — LABEL TAXONOMY MAPPING")
print(LABEL_MAP_DF[["source_dataset", "source_label", "ronair_label", "mapping_status"]].to_string(index=False))
print("\nPERINGATAN METODOLOGIS: mapping ini hanya menyelaraskan NAMA tingkat severity.")
print("Mapping TIDAK mengubah fakta bahwa label sumber adalah turunan aturan threshold.")
save_csv(LABEL_MAP_DF, DIR_RESULTS / "label_mapping.csv")

if D2 is not None:
    D2["risk_status"] = D2["class"].map(D2_TO_RONAAIR)
    D2["severity"] = D2["risk_status"].map(SEVERITY)
    print("\nDistribusi label RonaAir pada D2:")
    print(D2["risk_status"].value_counts().reindex(RONAAIR_LABELS).to_string())


SECTION 14 — LABEL TAXONOMY MAPPING
        source_dataset      source_label    ronair_label               mapping_status
D2_FISHPOND_MULTICLASS            Normal          NORMAL JUSTIFIED (ordinal, monoton)
D2_FISHPOND_MULTICLASS           Caution         WASPADA JUSTIFIED (ordinal, monoton)
D2_FISHPOND_MULTICLASS           Warning           SIAGA JUSTIFIED (ordinal, monoton)
D2_FISHPOND_MULTICLASS            Severe         DARURAT JUSTIFIED (ordinal, monoton)
       D3_IOTMLCQ_2024  Stable / At Risk TIDAK DIPETAKAN                     REJECTED
      D4_AQUAPONIC_IOT (tidak ada label) TIDAK DIPETAKAN               NOT APPLICABLE

PERINGATAN METODOLOGIS: mapping ini hanya menyelaraskan NAMA tingkat severity.
Mapping TIDAK mengubah fakta bahwa label sumber adalah turunan aturan threshold.
  [saved] label_mapping.csv  (6 rows)

Distribusi label RonaAir pada D2:
risk_status
NORMAL     638
WASPADA    670
SIAGA      478
DARURAT    367


## Cell 14 — Envelope operasional empiris dari data sensor nyata (D4)

Angka-angka ini **dataset-derived** (bukan karangan) dan dipakai untuk plausible-range,
deteksi sensor fault, dan out-of-distribution check — **bukan** sebagai threshold risiko.


In [14]:
EMPIRICAL_ENVELOPE = {}
if D4_RAW is not None:
    src = D4_RAW.copy()
    src = src[(src["water_pH"].between(0, 14))]  # buang pembacaan yang tidak mungkin secara fisik
    for col, key in [("water_pH", "ph_sensor"), ("water_temp", "water_temp"), ("TDS", "tds_ppm")]:
        s = src[col].dropna()
        EMPIRICAL_ENVELOPE[key] = {
            "source_dataset": "D4_AQUAPONIC_IOT (raw, pembacaan non-fisik dibuang)",
            "n": int(len(s)),
            "p01": round(float(s.quantile(0.01)), 3),
            "p05": round(float(s.quantile(0.05)), 3),
            "median": round(float(s.median()), 3),
            "p95": round(float(s.quantile(0.95)), 3),
            "p99": round(float(s.quantile(0.99)), 3),
            "min": round(float(s.min()), 3), "max": round(float(s.max()), 3),
        }
    # laju perubahan nyata per 5 menit (untuk rule "perubahan mendadak")
    tmp = D4_RAW.dropna(subset=["ts"]).sort_values("ts")
    tmp = tmp[tmp["water_pH"].between(0, 14)]
    g = tmp.set_index("ts")[["water_pH", "water_temp", "TDS"]].resample("5min").mean().dropna()
    for col, key in [("water_pH", "ph_sensor"), ("water_temp", "water_temp"), ("TDS", "tds_ppm")]:
        d = g[col].diff().dropna().abs()
        EMPIRICAL_ENVELOPE[key]["abs_delta_5min_p99"] = round(float(d.quantile(0.99)), 4)
        EMPIRICAL_ENVELOPE[key]["abs_delta_5min_median"] = round(float(d.median()), 4)

hr("ENVELOPE OPERASIONAL EMPIRIS (dari sensor nyata D4)")
for k, v in EMPIRICAL_ENVELOPE.items():
    print(f"  {k:12s} n={v['n']:,}  p01={v['p01']}  median={v['median']}  p99={v['p99']}  "
          f"|delta|/5min p99={v['abs_delta_5min_p99']}")
if not EMPIRICAL_ENVELOPE:
    print("  D4 tidak tersedia -> envelope empiris tidak dapat dihitung.")
save_json(EMPIRICAL_ENVELOPE, DIR_RESULTS / "empirical_operating_envelope.json")


ENVELOPE OPERASIONAL EMPIRIS (dari sensor nyata D4)
  ph_sensor    n=505,511  p01=5.56  median=7.84  p99=11.24  |delta|/5min p99=1.0809
  water_temp   n=505,511  p01=22.0  median=24.19  p99=26.69  |delta|/5min p99=0.07
  tds_ppm      n=505,511  p01=268.0  median=342.0  p99=1028.0  |delta|/5min p99=355.46
  [saved] results/empirical_operating_envelope.json


## Cell 15 — Rule engine config (Section 30 & 31)

Threshold **tidak** di-hard-code di notebook, melainkan di `configs/risk_rules.json`.
Setiap rule membawa provenance dan status validasinya sendiri. Tidak ada threshold yang
diklaim *scientifically validated*.


In [15]:
LIT = ("Praktik umum budidaya air tawar (nilai pengajaran/penyuluhan). "
       "BELUM diverifikasi terhadap SOP spesifik komoditas/lokasi di dalam notebook ini.")

RISK_RULES = {
    "schema_version": "1.0.0",
    "model_version": MODEL_VERSION,
    "generated_at": stamp(),
    "severity_order": RONAAIR_LABELS,
    "aggregation": "MAX_SEVERITY — status akhir = tingkat tertinggi di antara rule yang aktif",
    "global_notes": [
        "Seluruh threshold berstatus PROVISIONAL sampai divalidasi pakar/lapangan.",
        "do_est adalah ESTIMATED DO (soft-sensor), bukan measured DO.",
        "Rule ini adalah decision support, bukan diagnosis biologis maupun jaminan keselamatan.",
    ],
    "rules": [
        {"rule_id": "R001", "parameter": "do_est", "operator": "<", "threshold": 2.0, "unit": "mg/L",
         "risk_level": "DARURAT", "source": LIT, "source_type": "literature/expert",
         "validation_status": "PROVISIONAL",
         "notes": "Hipoksia berat; risiko kematian massal. Berlaku pada ESTIMATED DO -> perlu konfirmasi sensor."},
        {"rule_id": "R002", "parameter": "do_est", "operator": "<", "threshold": 3.0, "unit": "mg/L",
         "risk_level": "SIAGA", "source": LIT, "source_type": "literature/expert",
         "validation_status": "PROVISIONAL", "notes": "DO rendah; stres pernapasan."},
        {"rule_id": "R003", "parameter": "do_est", "operator": "<", "threshold": 5.0, "unit": "mg/L",
         "risk_level": "WASPADA", "source": LIT, "source_type": "literature/expert",
         "validation_status": "PROVISIONAL", "notes": "Di bawah kisaran nyaman umum."},
        {"rule_id": "R010", "parameter": "ph_sensor", "operator": "<", "threshold": 5.5, "unit": "pH",
         "risk_level": "DARURAT", "source": LIT, "source_type": "literature/expert",
         "validation_status": "PROVISIONAL", "notes": "Asam ekstrem."},
        {"rule_id": "R011", "parameter": "ph_sensor", "operator": ">", "threshold": 9.5, "unit": "pH",
         "risk_level": "DARURAT", "source": LIT, "source_type": "literature/expert",
         "validation_status": "PROVISIONAL", "notes": "Basa ekstrem; risiko amonia tak terionisasi meningkat."},
        {"rule_id": "R012", "parameter": "ph_sensor", "operator": "<", "threshold": 6.5, "unit": "pH",
         "risk_level": "SIAGA", "source": LIT, "source_type": "literature/expert",
         "validation_status": "PROVISIONAL", "notes": "Di bawah kisaran umum."},
        {"rule_id": "R013", "parameter": "ph_sensor", "operator": ">", "threshold": 9.0, "unit": "pH",
         "risk_level": "SIAGA", "source": LIT, "source_type": "literature/expert",
         "validation_status": "PROVISIONAL", "notes": "Di atas kisaran umum."},
        {"rule_id": "R020", "parameter": "water_temp", "operator": ">", "threshold": 34.0, "unit": "degC",
         "risk_level": "SIAGA", "source": LIT, "source_type": "literature/expert",
         "validation_status": "PROVISIONAL",
         "notes": "Suhu tinggi menurunkan kelarutan oksigen dan menaikkan laju metabolik."},
        {"rule_id": "R021", "parameter": "water_temp", "operator": "<", "threshold": 20.0, "unit": "degC",
         "risk_level": "WASPADA", "source": LIT, "source_type": "literature/expert",
         "validation_status": "PROVISIONAL", "notes": "Suhu rendah menekan nafsu makan (spesies tropis)."},
        {"rule_id": "R030", "parameter": "visual_score", "operator": ">=", "threshold": 0.75, "unit": "skor 0-1",
         "risk_level": "SIAGA", "source": "Kanal CV RonaAir (skala model internal)",
         "source_type": "provisional", "validation_status": "PROVISIONAL",
         "notes": "Indikator visual saja. TIDAK boleh diklaim sebagai spesies alga atau konsentrasi toksin."},
        {"rule_id": "R031", "parameter": "visual_score", "operator": ">=", "threshold": 0.50, "unit": "skor 0-1",
         "risk_level": "WASPADA", "source": "Kanal CV RonaAir", "source_type": "provisional",
         "validation_status": "PROVISIONAL", "notes": "Perubahan rona air terdeteksi."},
        {"rule_id": "R040", "parameter": "ph_difference", "operator": ">", "threshold": 1.0, "unit": "pH",
         "risk_level": "WASPADA",
         "source": "Selisih antar-kanal (pH foto vs pH sensor)", "source_type": "provisional",
         "validation_status": "PROVISIONAL",
         "notes": "Ketidaksepakatan dua kanal -> kemungkinan sensor drift atau kualitas foto buruk. "
                  "Ini adalah sinyal KUALITAS DATA, bukan bukti bahaya biologis."},
    ],
    "plausibility_ranges": {
        "ph_sensor": [0.0, 14.0], "water_temp": [0.0, 45.0], "do_est": [0.0, 25.0],
        "ec_value": [0.0, 100000.0], "tds_ppm": [0.0, 50000.0], "visual_score": [0.0, 1.0],
    },
    "empirical_operating_envelope": EMPIRICAL_ENVELOPE,
}

save_json(RISK_RULES, DIR_CONFIGS / "risk_rules.json")
hr("SECTION 30/31 — RULE ENGINE CONFIG")
print(f"  {len(RISK_RULES['rules'])} rule ditulis ke configs/risk_rules.json")
print(f"  Status validasi: {sorted(set(r['validation_status'] for r in RISK_RULES['rules']))}")
print("  Tidak ada rule yang diberi status 'VALIDATED'.")

# Cost matrix severity-aware (Section 29)
COST_MATRIX = {
    "schema_version": "1.0.0",
    "labels": RONAAIR_LABELS,
    "description": "Biaya relatif kesalahan klasifikasi. Menganggap remeh risiko tinggi "
                   "(false negative) jauh lebih mahal daripada alarm palsu.",
    "validation_status": "PROVISIONAL — bobot ditetapkan oleh desain, belum dikalibrasi ekonomi lapangan",
    "matrix": {},
}
for t in RONAAIR_LABELS:
    COST_MATRIX["matrix"][t] = {}
    for p in RONAAIR_LABELS:
        d = SEVERITY[t] - SEVERITY[p]
        COST_MATRIX["matrix"][t][p] = 0 if d == 0 else (3 * d if d > 0 else abs(d))
save_json(COST_MATRIX, DIR_CONFIGS / "risk_cost_matrix.json")


  [saved] configs/risk_rules.json
SECTION 30/31 — RULE ENGINE CONFIG
  12 rule ditulis ke configs/risk_rules.json
  Status validasi: ['PROVISIONAL']
  Tidak ada rule yang diberi status 'VALIDATED'.
  [saved] configs/risk_cost_matrix.json


## Cell 16 — LEVEL 1: Rule-Based Risk Engine

Ini adalah **mesin keputusan operasional** RonaAir. Ia berjalan tanpa model ML,
transparan, dan dapat diaudit satu per satu.


In [16]:
OPS = {
    "<": lambda a, b: a < b, "<=": lambda a, b: a <= b,
    ">": lambda a, b: a > b, ">=": lambda a, b: a >= b,
    "==": lambda a, b: a == b,
}

REQUIRED_FOR_DECISION = ["water_temp", "ph_sensor"]   # minimum agar keputusan bermakna
HIGH_VALUE_INPUTS = ["do_est", "visual_score"]        # salah satunya sangat diinginkan

def rule_engine(obs, rules_cfg=RISK_RULES):
    """LEVEL 1 rule engine. Mengembalikan status, faktor pemicu, dan kualitas data."""
    obs = dict(obs)
    factors, fired, warnings_ = [], [], []

    # turunkan ph_difference bila kedua kanal ada
    if obs.get("ph_visual_est") is not None and obs.get("ph_sensor") is not None:
        obs["ph_difference"] = abs(float(obs["ph_visual_est"]) - float(obs["ph_sensor"]))

    # --- plausibility & OOD ---
    for p, (lo, hi) in rules_cfg["plausibility_ranges"].items():
        v = obs.get(p)
        if v is not None and not (lo <= float(v) <= hi):
            warnings_.append(f"{p}={v} di luar rentang fisik [{lo}, {hi}] -> kemungkinan sensor fault")
            obs[p] = None
    env = rules_cfg.get("empirical_operating_envelope", {})
    ood = []
    for p, e in env.items():
        v = obs.get(p)
        if v is not None and (float(v) < e["p01"] or float(v) > e["p99"]):
            ood.append(f"{p}={v} di luar envelope operasional empiris [{e['p01']}, {e['p99']}]")

    # --- kualitas data ---
    missing_required = [p for p in REQUIRED_FOR_DECISION if obs.get(p) is None]
    missing_highvalue = [p for p in HIGH_VALUE_INPUTS if obs.get(p) is None]
    iq = obs.get("image_quality")
    image_failed = (iq is not None and str(iq).upper() in ("FAIL", "BAD", "POOR"))

    if missing_required or (len(missing_highvalue) == len(HIGH_VALUE_INPUTS)):
        data_quality = "INSUFFICIENT_EVIDENCE"
    elif warnings_ or image_failed or missing_highvalue:
        data_quality = "DEGRADED"
    elif ood:
        data_quality = "POSSIBLE_OUT_OF_DISTRIBUTION"
    else:
        data_quality = "OK"

    if data_quality == "INSUFFICIENT_EVIDENCE":
        reasons = []
        if missing_required:
            reasons.append(f"parameter wajib tidak tersedia: {missing_required}")
        if len(missing_highvalue) == len(HIGH_VALUE_INPUTS):
            reasons.append(f"tidak ada bukti oksigen maupun visual: {HIGH_VALUE_INPUTS}")
        return {"risk_status": "INSUFFICIENT_EVIDENCE", "supporting_factors": reasons + warnings_,
                "fired_rules": [], "data_quality": data_quality, "ood_notes": ood}

    # --- evaluasi rule ---
    worst = "NORMAL"
    for r in rules_cfg["rules"]:
        v = obs.get(r["parameter"])
        if v is None:
            continue
        if OPS[r["operator"]](float(v), float(r["threshold"])):
            fired.append(r["rule_id"])
            factors.append(f"[{r['rule_id']}] {r['parameter']}={v} {r['operator']} {r['threshold']} "
                           f"{r['unit']} -> {r['risk_level']}")
            if SEVERITY[r["risk_level"]] > SEVERITY[worst]:
                worst = r["risk_level"]

    if not factors:
        factors.append("Semua parameter yang tersedia berada di dalam rentang rule PROVISIONAL.")
    return {"risk_status": worst, "supporting_factors": factors + warnings_,
            "fired_rules": fired, "data_quality": data_quality, "ood_notes": ood}

# --- Demonstrasi pada observasi nyata dari D4 (sensor asli, tanpa DO/visual) ---
hr("LEVEL 1 — RULE ENGINE (uji pada pembacaan sensor NYATA dari D4)")
if D4_RAW is not None:
    demo = D4_RAW.sample(4, random_state=RANDOM_SEED)
    for _, row in demo.iterrows():
        o = {"water_temp": float(row["water_temp"]), "ph_sensor": float(row["water_pH"]),
             "tds_ppm": float(row["TDS"]), "do_est": None, "visual_score": None}
        res = rule_engine(o)
        print(f"\n  input : temp={o['water_temp']}, pH={o['ph_sensor']}, TDS={o['tds_ppm']}, "
              f"do_est=None, visual=None")
        print(f"  status: {res['risk_status']}   data_quality: {res['data_quality']}")
        for f in res["supporting_factors"]:
            print(f"      - {f}")
    # contoh pembacaan rusak
    bad = D4_RAW[D4_RAW["water_pH"] > 14].head(1)
    if len(bad):
        row = bad.iloc[0]
        o = {"water_temp": float(row["water_temp"]), "ph_sensor": float(row["water_pH"]),
             "tds_ppm": float(row["TDS"]), "do_est": 6.0, "visual_score": 0.1}
        res = rule_engine(o)
        print(f"\n  [pembacaan sensor rusak] pH={o['ph_sensor']}")
        print(f"  status: {res['risk_status']}   data_quality: {res['data_quality']}")
        for f in res["supporting_factors"]:
            print(f"      - {f}")


LEVEL 1 — RULE ENGINE (uji pada pembacaan sensor NYATA dari D4)

  input : temp=22.94, pH=9.59, TDS=340.0, do_est=None, visual=None
  status: INSUFFICIENT_EVIDENCE   data_quality: INSUFFICIENT_EVIDENCE
      - tidak ada bukti oksigen maupun visual: ['do_est', 'visual_score']

  input : temp=23.31, pH=6.81, TDS=433.0, do_est=None, visual=None
  status: INSUFFICIENT_EVIDENCE   data_quality: INSUFFICIENT_EVIDENCE
      - tidak ada bukti oksigen maupun visual: ['do_est', 'visual_score']

  input : temp=23.38, pH=6.22, TDS=433.0, do_est=None, visual=None
  status: INSUFFICIENT_EVIDENCE   data_quality: INSUFFICIENT_EVIDENCE
      - tidak ada bukti oksigen maupun visual: ['do_est', 'visual_score']

  input : temp=24.06, pH=6.9, TDS=325.0, do_est=None, visual=None
  status: INSUFFICIENT_EVIDENCE   data_quality: INSUFFICIENT_EVIDENCE
      - tidak ada bukti oksigen maupun visual: ['do_est', 'visual_score']

  [pembacaan sensor rusak] pH=15.78
  status: INSUFFICIENT_EVIDENCE   data_quality: INSU

## Cell 17 — Rule engine vs label D2

Uji apakah rule engine berbasis fisiologi menghasilkan keputusan yang sama dengan
label dataset D2. Ketidaksesuaian di sini **informatif**, bukan kegagalan rule engine.


In [17]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix, recall_score, precision_score)

RULE_VS_D2 = {}
if D2 is not None:
    preds = []
    for _, r in D2.iterrows():
        o = {"water_temp": float(r["temp_C"]), "ph_sensor": float(r["pH"]),
             "do_est": float(r["do_mgL"]), "ec_value": float(r["ec_uScm"]),
             "tds_ppm": float(r["tds_mgL"]), "visual_score": None}
        preds.append(rule_engine(o)["risk_status"])
    D2["rule_engine_pred"] = preds
    valid = D2["rule_engine_pred"].isin(RONAAIR_LABELS)
    acc = accuracy_score(D2.loc[valid, "risk_status"], D2.loc[valid, "rule_engine_pred"])
    mf1 = f1_score(D2.loc[valid, "risk_status"], D2.loc[valid, "rule_engine_pred"],
                   average="macro", labels=RONAAIR_LABELS, zero_division=0)
    hr("RULE ENGINE (fisiologi) vs LABEL D2 (aturan turbidity)")
    print(f"  Observasi terevaluasi : {int(valid.sum()):,}")
    print(f"  Agreement (accuracy)  : {acc:.4f}")
    print(f"  Macro-F1              : {mf1:.4f}")
    print("\n  Confusion (baris = label D2, kolom = rule engine RonaAir):")
    cm = pd.crosstab(D2.loc[valid, "risk_status"], D2.loc[valid, "rule_engine_pred"])
    cm = cm.reindex(index=RONAAIR_LABELS, columns=RONAAIR_LABELS, fill_value=0)
    print(cm.to_string())
    RULE_VS_D2 = {"agreement_accuracy": round(float(acc), 4), "macro_f1": round(float(mf1), 4),
                  "n_evaluated": int(valid.sum())}
    print("\n  INTERPRETASI: kedua sistem mengukur hal yang berbeda. Label D2 ditentukan oleh")
    print("  turbidity; rule engine RonaAir ditentukan oleh DO, pH, dan suhu. Rendahnya")
    print("  kesepakatan adalah bukti bahwa label D2 tidak merepresentasikan risiko fisiologis.")


RULE ENGINE (fisiologi) vs LABEL D2 (aturan turbidity)
  Observasi terevaluasi : 2,153
  Agreement (accuracy)  : 0.7622
  Macro-F1              : 0.7508

  Confusion (baris = label D2, kolom = rule engine RonaAir):
rule_engine_pred  NORMAL  WASPADA  SIAGA  DARURAT
risk_status                                      
NORMAL               638        0      0        0
WASPADA              179      161    330        0
SIAGA                  3        0    475        0
DARURAT                0        0      0      367

  INTERPRETASI: kedua sistem mengukur hal yang berbeda. Label D2 ditentukan oleh
  turbidity; rule engine RonaAir ditentukan oleh DO, pH, dan suhu. Rendahnya
  kesepakatan adalah bukti bahwa label D2 tidak merepresentasikan risiko fisiologis.


## Cell 18 — Feature engineering & kontrak deployment (Section 17 & 18)

Fitur dinamai ulang ke **nama kanonik RonaAir** agar mismatch deployment terlihat jelas.

Catatan penting yang tidak boleh disembunyikan:

* `do_mgL` pada D2 adalah DO terukur, sedangkan APK hanya punya `do_est` (soft-sensor).
  Ini adalah **DEPLOYMENT FEATURE MISMATCH** dan dicatat.
* `turbidity_NTU` bukan `visual_score`. RonaAir tidak memiliki sensor turbidity. Turbidity
  dipakai sebagai **proxy kanal visual yang BELUM TERVALIDASI**, karena keduanya sama-sama
  sifat optik kejernihan air. Tidak ada dataset publik berisi foto + sensor berpasangan
  untuk memvalidasi pemetaan ini.
* `orp_mV` tidak ada di perangkat RonaAir -> dikeluarkan dari seluruh feature set produksi.


In [18]:
FEATURE_CONTRACT = pd.DataFrame([
    {"dataset_column": "temp_C", "ronair_feature": "water_temp", "available_in_apk": True,
     "apk_source": DEPLOYMENT_FEATURES["water_temp"], "status": "MATCH"},
    {"dataset_column": "pH", "ronair_feature": "ph_sensor", "available_in_apk": True,
     "apk_source": DEPLOYMENT_FEATURES["ph_sensor"], "status": "MATCH"},
    {"dataset_column": "ec_uScm", "ronair_feature": "ec_value", "available_in_apk": True,
     "apk_source": DEPLOYMENT_FEATURES["ec_value"], "status": "MATCH"},
    {"dataset_column": "tds_mgL", "ronair_feature": "tds_ppm", "available_in_apk": True,
     "apk_source": DEPLOYMENT_FEATURES["tds_ppm"], "status": "MATCH (asumsi mg/L ~ ppm)"},
    {"dataset_column": "do_mgL", "ronair_feature": "do_est", "available_in_apk": True,
     "apk_source": DEPLOYMENT_FEATURES["do_est"],
     "status": "DEPLOYMENT FEATURE MISMATCH — training memakai DO terukur, APK memakai DO estimasi"},
    {"dataset_column": "turbidity_NTU", "ronair_feature": "visual_proxy_turbidity",
     "available_in_apk": False, "apk_source": "TIDAK ADA sensor turbidity di RonaAir",
     "status": "UNVALIDATED PROXY untuk kanal visual (visual_score)"},
    {"dataset_column": "orp_mV", "ronair_feature": "(tidak dipakai)", "available_in_apk": False,
     "apk_source": "TIDAK ADA probe ORP", "status": "EXCLUDED — tidak tersedia saat deployment"},
    {"dataset_column": "(tidak ada)", "ronair_feature": "ph_visual_est", "available_in_apk": True,
     "apk_source": DEPLOYMENT_FEATURES["ph_visual_est"],
     "status": "TIDAK ADA DI DATASET MANA PUN — SET D tidak dapat dievaluasi"},
    {"dataset_column": "(tidak ada)", "ronair_feature": "visual_score", "available_in_apk": True,
     "apk_source": DEPLOYMENT_FEATURES["visual_score"],
     "status": "TIDAK ADA DI DATASET MANA PUN — hanya tersedia proxy"},
])
hr("SECTION 18 — KONTRAK FITUR DEPLOYMENT")
print(FEATURE_CONTRACT.to_string(index=False, max_colwidth=58))
save_csv(FEATURE_CONTRACT, DIR_RESULTS / "deployment_feature_contract.csv")


SECTION 18 — KONTRAK FITUR DEPLOYMENT
dataset_column         ronair_feature  available_in_apk                                       apk_source                                                     status
        temp_C             water_temp              True                          ESP32 temperature probe                                                      MATCH
            pH              ph_sensor              True                                   ESP32 pH probe                                                      MATCH
       ec_uScm               ec_value              True                                   ESP32 EC probe                                                      MATCH
       tds_mgL                tds_ppm              True                      ESP32 TDS (derived from EC)                                  MATCH (asumsi mg/L ~ ppm)
        do_mgL                 do_est              True         DO soft-sensor (ESTIMATED, not measured) DEPLOYMENT FEATURE MISMATCH — trainin

In [19]:
FEAT_SETS = {}
FS = None
if D2 is not None:
    FS = pd.DataFrame({
        "ts": D2["ts"],
        "water_temp": D2["temp_C"],
        "ph_sensor": D2["pH"],
        "ec_value": D2["ec_uScm"],
        "tds_ppm": D2["tds_mgL"],
        "do_est": D2["do_mgL"],
        "visual_proxy_turbidity": D2["turbidity_NTU"],
        "risk_status": D2["risk_status"],
    }).sort_values("ts").reset_index(drop=True)

    # --- fitur temporal: HANYA dari observasi sekarang + masa lalu (Section 19) ---
    FS["hour"] = FS["ts"].dt.hour
    FS["day_or_night"] = ((FS["hour"] >= 6) & (FS["hour"] < 18)).astype(int)
    for c in ["water_temp", "ph_sensor", "do_est", "visual_proxy_turbidity"]:
        FS[f"prev_{c}"] = FS[c].shift(1)          # shift(1) = hanya masa lalu
        FS[f"delta_{c}"] = FS[c] - FS[f"prev_{c}"]
    dt_min = FS["ts"].diff().dt.total_seconds().div(60)
    for c in ["water_temp", "ph_sensor", "do_est"]:
        FS[f"rate_{c}_per_min"] = FS[f"delta_{c}"] / dt_min.replace(0, np.nan)

    SET_A = ["water_temp", "ph_sensor", "ec_value", "tds_ppm", "do_est"]
    SET_B = ["visual_proxy_turbidity"]
    SET_C = SET_A + SET_B
    SET_D = None   # butuh ph_visual_est -> tidak ada di dataset mana pun
    SET_E = SET_C + ["hour", "day_or_night"] + \
            [f"prev_{c}" for c in ["water_temp", "ph_sensor", "do_est", "visual_proxy_turbidity"]] + \
            [f"delta_{c}" for c in ["water_temp", "ph_sensor", "do_est", "visual_proxy_turbidity"]] + \
            [f"rate_{c}_per_min" for c in ["water_temp", "ph_sensor", "do_est"]]

    FEAT_SETS = {
        "A_sensor_only": SET_A,
        "B_visual_proxy_only": SET_B,
        "C_sensor_plus_visual": SET_C,
        "E_full_temporal_fusion": SET_E,
    }
    # SECTION 18: feature set hanya layak deployment bila SELURUH fiturnya ada di APK.
    DEPLOYABLE_SETS = {k: all(f in DEPLOYMENT_FEATURES for f in v) for k, v in FEAT_SETS.items()}

    hr("SECTION 17 — FEATURE SET")
    for k, v in FEAT_SETS.items():
        flag = "LAYAK DEPLOY" if DEPLOYABLE_SETS[k] else "TIDAK LAYAK DEPLOY"
        miss = [f for f in v if f not in DEPLOYMENT_FEATURES]
        print(f"  {k:26s} ({len(v):2d} fitur) [{flag}]: {v}")
        if miss:
            print(f"      fitur yang tidak ada di APK: {miss}")
    print(f"  {'D_plus_ph_photo':26s} : TIDAK DAPAT DIEVALUASI — "
          f"`ph_visual_est` dan `ph_difference` tidak ada di dataset publik mana pun.")
    print("    Mensimulasikannya berarti mengarang data, sehingga TIDAK dilakukan.")


SECTION 17 — FEATURE SET
  A_sensor_only              ( 5 fitur) [LAYAK DEPLOY]: ['water_temp', 'ph_sensor', 'ec_value', 'tds_ppm', 'do_est']
  B_visual_proxy_only        ( 1 fitur) [TIDAK LAYAK DEPLOY]: ['visual_proxy_turbidity']
      fitur yang tidak ada di APK: ['visual_proxy_turbidity']
  C_sensor_plus_visual       ( 6 fitur) [TIDAK LAYAK DEPLOY]: ['water_temp', 'ph_sensor', 'ec_value', 'tds_ppm', 'do_est', 'visual_proxy_turbidity']
      fitur yang tidak ada di APK: ['visual_proxy_turbidity']
  E_full_temporal_fusion     (19 fitur) [TIDAK LAYAK DEPLOY]: ['water_temp', 'ph_sensor', 'ec_value', 'tds_ppm', 'do_est', 'visual_proxy_turbidity', 'hour', 'day_or_night', 'prev_water_temp', 'prev_ph_sensor', 'prev_do_est', 'prev_visual_proxy_turbidity', 'delta_water_temp', 'delta_ph_sensor', 'delta_do_est', 'delta_visual_proxy_turbidity', 'rate_water_temp_per_min', 'rate_ph_sensor_per_min', 'rate_do_est_per_min']
      fitur yang tidak ada di APK: ['visual_proxy_turbidity', 'day_or_night',

## Cell 19 — Data leakage audit (Section 20)


In [20]:
LEAKAGE_REPORT = []
hr("SECTION 20 — DATA LEAKAGE AUDIT")
if FS is not None:
    checks = []
    checks.append(("duplicate rows (fitur identik)", int(FS.drop(columns=["ts"]).duplicated().sum()),
                   "0 diharapkan"))
    checks.append(("duplicate timestamp", int(FS["ts"].duplicated().sum()), "0 diharapkan"))
    checks.append(("pond/site ID tersedia", 0, "TIDAK ADA -> group split mustahil"))
    checks.append(("future information dalam fitur", 0,
                   "Semua fitur lag memakai shift(+1) = masa lalu saja; tidak ada shift negatif"))
    checks.append(("label-derived column dijadikan predictor", 0,
                   "Kolom `class`/`risk_status`/`severity` tidak pernah masuk X"))
    checks.append(("intervention / post-event feature", 0,
                   "D2 tidak memuat kolom intervensi; D3 memuatnya tetapi D3 tidak dipakai training"))
    for name, val, note in checks:
        print(f"  {name:42s}: {val}   ({note})")
        LEAKAGE_REPORT.append({"check": name, "value": val, "note": note})

    # Leakage paling serius: target berasal dari salah satu predictor
    drv = D2_FINDINGS.get("label_driver")
    acc_drv = D2_FINDINGS.get("label_recovery_accuracy", 0)
    print(f"\n  *** TARGET LEAKAGE STRUKTURAL ***")
    print(f"  Target `risk_status` tereproduksi {acc_drv:.2%} dari `{drv}`, yang dipetakan ke")
    print(f"  fitur `visual_proxy_turbidity`. Setiap feature set yang memuat fitur ini akan")
    print(f"  mencapai skor hampir sempurna karena MENIRU ATURAN, bukan memprediksi risiko.")
    LEAKAGE_REPORT.append({
        "check": "structural target leakage (label = fungsi predictor)",
        "value": acc_drv,
        "note": f"target dapat direproduksi {acc_drv:.4f} dari {drv}; "
                f"set B/C/E memuat fitur ini -> hasilnya adalah rule emulation, bukan prediksi.",
    })
    save_csv(pd.DataFrame(LEAKAGE_REPORT), DIR_RESULTS / "leakage_audit.csv")


SECTION 20 — DATA LEAKAGE AUDIT
  duplicate rows (fitur identik)            : 0   (0 diharapkan)
  duplicate timestamp                       : 0   (0 diharapkan)
  pond/site ID tersedia                     : 0   (TIDAK ADA -> group split mustahil)
  future information dalam fitur            : 0   (Semua fitur lag memakai shift(+1) = masa lalu saja; tidak ada shift negatif)
  label-derived column dijadikan predictor  : 0   (Kolom `class`/`risk_status`/`severity` tidak pernah masuk X)
  intervention / post-event feature         : 0   (D2 tidak memuat kolom intervensi; D3 memuatnya tetapi D3 tidak dipakai training)

  *** TARGET LEAKAGE STRUKTURAL ***
  Target `risk_status` tereproduksi 100.00% dari `orp_mV`, yang dipetakan ke
  fitur `visual_proxy_turbidity`. Setiap feature set yang memuat fitur ini akan
  mencapai skor hampir sempurna karena MENIRU ATURAN, bukan memprediksi risiko.
  [saved] leakage_audit.csv  (7 rows)


## Cell 20 — Split strategy (Section 21)

Prioritas: group split (pond/site) > time-based > gabungan > random.
D2 **tidak punya pond ID** dan **punya timestamp** -> yang berlaku adalah
**chronological split**. Random row split tidak dipakai sebagai bukti akhir.


In [21]:
from sklearn.model_selection import train_test_split

SPLIT_INFO = {}
Xy = None
if FS is not None:
    work = FS.dropna(subset=[c for c in FEAT_SETS["E_full_temporal_fusion"]]).reset_index(drop=True)
    n = len(work)
    hr("SECTION 21 — SPLIT STRATEGY")
    print(f"  Baris dipakai : {n:,} dari {len(FS):,} "
          f"(baris pertama dibuang karena fitur lag belum terdefinisi)")

    # --- Prioritas 1: group split berbasis pond/site ---
    print("\n  [1] Group split (pond/site) : TIDAK MUNGKIN — dataset tidak memiliki pond/site ID.")

    # --- Prioritas 2: chronological split, diuji kelayakannya lebih dulu ---
    i_tr, i_va = int(n * 0.60), int(n * 0.80)
    chrono = np.where(work.index < i_tr, "train",
                      np.where(work.index < i_va, "validation", "test"))
    chrono_dist = pd.crosstab(chrono, work["risk_status"]).reindex(
        index=["train", "validation", "test"], columns=RONAAIR_LABELS, fill_value=0)
    print("\n  [2] Chronological split — distribusi kelas yang dihasilkan:")
    print("      " + chrono_dist.to_string().replace("\n", "\n      "))
    classes_per_split = (chrono_dist > 0).sum(axis=1)
    degenerate = bool((classes_per_split < len(RONAAIR_LABELS)).any())

    # kuantifikasi kebingungan waktu vs kelas
    rank_ts = work["ts"].rank()
    sev = work["risk_status"].map(SEVERITY)
    rho = float(pd.Series(rank_ts).corr(pd.Series(sev), method="spearman"))
    print(f"\n      Korelasi Spearman antara urutan waktu dan severity kelas: rho = {rho:+.4f}")

    if degenerate:
        print("\n      *** CHRONOLOGICAL SPLIT DEGENERATIF ***")
        print("      Setiap potongan waktu hanya memuat sebagian kelas. Artinya timestamp pada")
        print("      dataset ini TIDAK merepresentasikan dinamika kolam: label disusun berurutan")
        print("      menurut severity, lalu timestamp ditempelkan mengikuti urutan itu.")
        print("      Melatih pada potongan awal berarti model tidak pernah melihat SIAGA/DARURAT.")
        print("      Ini adalah bukti tambahan bahwa dataset bersifat sintetis dan struktur")
        print("      temporalnya artifisial.")

    # --- Prioritas 3/4: fallback ---
    if not degenerate:
        work["split"] = chrono
        strategy = "chronological"
        strategy_note = "Group split mustahil; chronological split layak."
    else:
        strategy = "stratified_random (LAST RESORT)"
        strategy_note = ("Group split mustahil (tidak ada pond ID) DAN chronological split "
                         "degeneratif (kelas terkonfound sempurna dengan waktu). Sesuai hierarki "
                         "Section 21, random split dipakai sebagai upaya terakhir dan HASILNYA "
                         "TIDAK BOLEH dianggap bukti generalisasi temporal.")
        idx = np.arange(n)
        y_all = work["risk_status"]
        i_tmp, i_te = train_test_split(idx, test_size=0.20, random_state=RANDOM_SEED, stratify=y_all)
        i_tr2, i_va2 = train_test_split(i_tmp, test_size=0.25, random_state=RANDOM_SEED,
                                        stratify=y_all.iloc[i_tmp])
        lab = np.empty(n, dtype=object)
        lab[i_tr2] = "train"; lab[i_va2] = "validation"; lab[i_te] = "test"
        work["split"] = lab
        print(f"\n  [3] Fallback dipakai : {strategy}")
        print(f"      {strategy_note}")
        print("\n      KONSEKUENSI LEAKAGE yang harus dicatat: baris yang bertetangga dalam waktu")
        print("      dapat jatuh ke split berbeda, sehingga fitur lag (prev_*, delta_*) pada")
        print("      SET E menjadi rawan kebocoran temporal. Skor SET E karena itu dianggap")
        print("      OPTIMISTIS dan tidak dipakai sebagai bukti akhir.")

    Xy = work
    print(f"\n  Strategi final : {strategy}")
    for s in ["train", "validation", "test"]:
        sub = work[work.split == s]
        print(f"    {s:11s}: n={len(sub):>5,}  {sub['ts'].min()} -> {sub['ts'].max()}")
        print(f"                 distribusi: {sub['risk_status'].value_counts().reindex(RONAAIR_LABELS).fillna(0).astype(int).to_dict()}")
    print("\n  TEST SET DIKUNCI — tidak dipakai untuk feature selection, tuning, maupun model selection.")

    SPLIT_INFO = {"strategy": strategy, "strategy_note": strategy_note,
                  "chronological_degenerate": degenerate,
                  "spearman_time_vs_severity": round(rho, 4),
                  "group_split_possible": False,
                  "n_total": int(n),
                  "train": int((work.split == "train").sum()),
                  "validation": int((work.split == "validation").sum()),
                  "test": int((work.split == "test").sum())}
    D2_FINDINGS["timestamp_confounded_with_class"] = degenerate
    D2_FINDINGS["spearman_time_vs_severity"] = round(rho, 4)
    LEAKAGE_REPORT.append({
        "check": "timestamp terkonfound dengan kelas",
        "value": round(rho, 4),
        "note": ("Korelasi Spearman urutan waktu vs severity; nilai mendekati 1 berarti timestamp "
                 "hanyalah penomoran ulang urutan kelas, bukan dinamika kolam."),
    })
    save_csv(pd.DataFrame(LEAKAGE_REPORT), DIR_RESULTS / "leakage_audit.csv")
    split_out = work[["ts", "split", "risk_status"]].copy()
    save_csv(split_out, DIR_RESULTS / "risk_data_split.csv")

    # Section 23 — class imbalance
    hr("SECTION 23 — CLASS IMBALANCE")
    dist = work.groupby("split")["risk_status"].value_counts().unstack().reindex(columns=RONAAIR_LABELS).fillna(0).astype(int)
    print(dist.to_string())
    tr = work[work.split == "train"]["risk_status"].value_counts()
    ratio = tr.max() / max(tr.min(), 1)
    print(f"\n  Rasio kelas terbesar:terkecil pada train = {ratio:.2f}")
    print("  Strategi: class_weight='balanced' DULU (tanpa oversampling).")
    print("  SMOTE/oversampling tidak digunakan; jika kelak dipakai, hanya pada training fold.")


SECTION 21 — SPLIT STRATEGY
  Baris dipakai : 2,152 dari 2,153 (baris pertama dibuang karena fitur lag belum terdefinisi)

  [1] Group split (pond/site) : TIDAK MUNGKIN — dataset tidak memiliki pond/site ID.

  [2] Chronological split — distribusi kelas yang dihasilkan:
      risk_status  NORMAL  WASPADA  SIAGA  DARURAT
      row_0                                       
      train           637      654      0        0
      validation        0       16    414        0
      test              0        0     64      367

      Korelasi Spearman antara urutan waktu dan severity kelas: rho = +0.9633

      *** CHRONOLOGICAL SPLIT DEGENERATIF ***
      Setiap potongan waktu hanya memuat sebagian kelas. Artinya timestamp pada
      dataset ini TIDAK merepresentasikan dinamika kolam: label disusun berurutan
      menurut severity, lalu timestamp ditempelkan mengikuti urutan itu.
      Melatih pada potongan awal berarti model tidak pernah melihat SIAGA/DARURAT.
      Ini adalah bukti tambaha

## Cell 21 — LEVEL 2: Supervised Risk Model (benchmark)

**Pernyataan wajib (Section 15 & 32):**

> Target pelatihan berasal dari aturan threshold yang juga menjadi predictor.
> Karena itu model ini hanya **mereproduksi logika keputusan yang terdokumentasi**
> (*emulates documented decision logic*). Model ini **tidak** memprediksi kematian atau
> stres ikan di dunia nyata secara independen.


In [22]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.inspection import permutation_importance

def make_models(n_train, n_classes):
    m = {
        "DummyClassifier": DummyClassifier(strategy="most_frequent", random_state=RANDOM_SEED),
        "LogisticRegression": Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
            # multinomial adalah perilaku default untuk target multikelas dengan solver lbfgs.
            # Argumen `multi_class` sengaja tidak dipakai karena telah dihapus di scikit-learn >= 1.7.
            ("clf", LogisticRegression(max_iter=2000, class_weight="balanced",
                                       random_state=RANDOM_SEED)),
        ]),
        "DecisionTree_d3": Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("clf", DecisionTreeClassifier(max_depth=3, class_weight="balanced",
                                           random_state=RANDOM_SEED)),
        ]),
        "DecisionTree_d6": Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("clf", DecisionTreeClassifier(max_depth=6, class_weight="balanced",
                                           random_state=RANDOM_SEED)),
        ]),
        "RandomForest": Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced",
                                           random_state=RANDOM_SEED, n_jobs=-1)),
        ]),
        "GradientBoosting": Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("clf", GradientBoostingClassifier(random_state=RANDOM_SEED)),
        ]),
    }
    if HAS_XGB:
        from xgboost import XGBClassifier
        m["XGBoost"] = Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("clf", XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                                  subsample=0.9, objective="multi:softprob",
                                  num_class=n_classes, random_state=RANDOM_SEED,
                                  eval_metric="mlogloss", verbosity=0)),
        ])
    # MLP kecil hanya bila data cukup (Section 25)
    if n_train >= 500:
        m["MLP_32_16"] = Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
            ("clf", MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu",
                                  max_iter=800, random_state=RANDOM_SEED,
                                  early_stopping=False)),   # early_stopping dimatikan:
            # pada scikit-learn 1.8 opsi tersebut error untuk target bertipe string.
        ])
    return m

def high_risk_metrics(y_true, y_pred):
    """Recall/precision/F1 gabungan untuk SIAGA + DARURAT (Section 27)."""
    # np.asarray penting: Series dengan index berbeda akan salah align bila dibandingkan langsung.
    yt = np.isin(np.asarray(y_true, dtype=object), HIGH_RISK)
    yp = np.isin(np.asarray(y_pred, dtype=object), HIGH_RISK)
    tp = int((yt & yp).sum())
    fn = int((yt & ~yp).sum())
    fp = int((~yt & yp).sum())
    rec = tp / (tp + fn) if (tp + fn) else np.nan
    prec = tp / (tp + fp) if (tp + fp) else np.nan
    f1 = (2 * prec * rec / (prec + rec)
          if (prec == prec and rec == rec and (prec + rec) > 0) else np.nan)
    return {"HighRisk_Recall": rec, "HighRisk_Precision": prec, "HighRisk_F1": f1,
            "HighRisk_FalseNegatives": fn}

def eval_block(y_true, y_pred):
    out = {
        "Macro_F1": f1_score(y_true, y_pred, average="macro", labels=RONAAIR_LABELS, zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Weighted_F1": f1_score(y_true, y_pred, average="weighted", labels=RONAAIR_LABELS, zero_division=0),
    }
    out.update(high_risk_metrics(y_true, y_pred))
    return out

RESULTS_ROWS = []
FITTED = {}

if Xy is not None:
    train = Xy[Xy.split == "train"]
    valid = Xy[Xy.split == "validation"]
    test = Xy[Xy.split == "test"]
    y_tr, y_va, y_te = train["risk_status"], valid["risk_status"], test["risk_status"]

    hr("SECTION 24-26 & 33 — TRAINING + ABLATION")
    # Section 22: TimeSeriesSplit bila struktur temporal nyata; kalau tidak, StratifiedKFold.
    if SPLIT_INFO.get("chronological_degenerate"):
        from sklearn.model_selection import StratifiedKFold
        tscv = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_SEED)
        CV_NAME = "StratifiedKFold(4, shuffle)"
        print("  CV: StratifiedKFold(4). TimeSeriesSplit TIDAK dipakai karena struktur temporal")
        print("      dataset terbukti artifisial (kelas terkonfound dengan waktu), sehingga")
        print("      fold temporal akan kosong untuk sebagian kelas.")
    else:
        tscv = TimeSeriesSplit(n_splits=4)
        CV_NAME = "TimeSeriesSplit(4)"
        print("  CV: TimeSeriesSplit(4) — menghormati urutan waktu.")
    print("  Test TIDAK disentuh pada tahap ini.\n")

    def cv_iter(X, y):
        try:
            return list(tscv.split(X, y))
        except TypeError:
            return list(tscv.split(X))
    for set_name, feats in FEAT_SETS.items():
        Xtr, Xva = train[feats], valid[feats]
        models = make_models(len(train), len(RONAAIR_LABELS))
        print(f"--- FEATURE SET {set_name} ({len(feats)} fitur) ---")
        for mname, model in models.items():
            # --- cross validation di train ---
            cv_scores = {"Macro_F1": [], "Balanced_Accuracy": [], "HighRisk_Recall": []}
            for tr_i, va_i in cv_iter(Xtr, y_tr):
                try:
                    mm = make_models(len(tr_i), len(RONAAIR_LABELS))[mname]
                    ytr_f = y_tr.iloc[tr_i]
                    if ytr_f.nunique() < 2:
                        continue
                    mm.fit(Xtr.iloc[tr_i], ytr_f)
                    p = mm.predict(Xtr.iloc[va_i])
                    b = eval_block(y_tr.iloc[va_i], p)
                    for k in cv_scores:
                        cv_scores[k].append(b[k])
                except Exception:
                    continue
            cv_mean = {k: (float(np.nanmean(v)) if len(v) else np.nan) for k, v in cv_scores.items()}
            cv_std = float(np.nanstd(cv_scores["Macro_F1"])) if cv_scores["Macro_F1"] else np.nan

            # --- fit di train, evaluasi di validation ---
            t0 = time.perf_counter()
            model.fit(Xtr, y_tr)
            fit_s = time.perf_counter() - t0
            t0 = time.perf_counter()
            pv = model.predict(Xva)
            infer_ms = (time.perf_counter() - t0) / max(len(Xva), 1) * 1000
            vb = eval_block(y_va, pv)

            tmp = DIR_MODELS / "_tmp_size.joblib"
            joblib.dump(model, tmp)
            size_kb = tmp.stat().st_size / 1024
            tmp.unlink()

            FITTED[(set_name, mname)] = model
            RESULTS_ROWS.append({
                "Model": mname, "Feature_Set": set_name, "n_features": len(feats),
                "CV_Macro_F1": round(cv_mean["Macro_F1"], 4),
                "CV_Macro_F1_std": round(cv_std, 4),
                "CV_Balanced_Accuracy": round(cv_mean["Balanced_Accuracy"], 4),
                "CV_HighRisk_Recall": round(cv_mean["HighRisk_Recall"], 4),
                "Validation_Macro_F1": round(vb["Macro_F1"], 4),
                "Validation_Balanced_Accuracy": round(vb["Balanced_Accuracy"], 4),
                "Validation_HighRisk_Recall": round(vb["HighRisk_Recall"], 4),
                "Validation_HighRisk_FN": vb["HighRisk_FalseNegatives"],
                "Test_Macro_F1": np.nan, "Test_Balanced_Accuracy": np.nan,
                "Test_HighRisk_Recall": np.nan,
                "Model_Size_KB": round(size_kb, 1),
                "Fit_Time_s": round(fit_s, 3),
                "Inference_Time_ms_per_sample": round(infer_ms, 4),
                "Deployable_Features": DEPLOYABLE_SETS.get(set_name, False),
                "Mobile_Ready": mname in ("LogisticRegression", "DecisionTree_d3",
                                          "DecisionTree_d6", "DummyClassifier"),
            })
            print(f"  {mname:20s} CV MacroF1={cv_mean['Macro_F1']:.4f}(+-{cv_std:.4f})  "
                  f"Val MacroF1={vb['Macro_F1']:.4f}  Val HR-Recall={vb['HighRisk_Recall']:.4f}  "
                  f"{size_kb:7.1f}KB")
        print()

    RES = pd.DataFrame(RESULTS_ROWS)


SECTION 24-26 & 33 — TRAINING + ABLATION
  CV: StratifiedKFold(4). TimeSeriesSplit TIDAK dipakai karena struktur temporal
      dataset terbukti artifisial (kelas terkonfound dengan waktu), sehingga
      fold temporal akan kosong untuk sebagian kelas.
  Test TIDAK disentuh pada tahap ini.

--- FEATURE SET A_sensor_only (5 fitur) ---
  DummyClassifier      CV MacroF1=0.1188(+-0.0006)  Val MacroF1=0.1186  Val HR-Recall=0.0000      1.0KB
  LogisticRegression   CV MacroF1=0.3029(+-0.0296)  Val MacroF1=0.3071  Val HR-Recall=0.5353      2.3KB
  DecisionTree_d3      CV MacroF1=0.5545(+-0.0097)  Val MacroF1=0.5662  Val HR-Recall=0.7353      3.2KB
  DecisionTree_d6      CV MacroF1=0.9937(+-0.0028)  Val MacroF1=0.9970  Val HR-Recall=1.0000      4.1KB
  RandomForest         CV MacroF1=0.9987(+-0.0022)  Val MacroF1=1.0000  Val HR-Recall=1.0000    599.7KB
  GradientBoosting     CV MacroF1=0.9950(+-0.0071)  Val MacroF1=1.0000  Val HR-Recall=1.0000    515.5KB
  MLP_32_16            CV MacroF1=0.9985

## Cell 22 — Model selection (Section 26)

Seleksi memakai **CV + validation**, tidak pernah test. Jika dua model berdekatan,
dipilih yang lebih sederhana / lebih kecil / lebih mudah di-deploy ke Android.


In [23]:
FINAL_SET = FINAL_MODEL_NAME = FINAL_MODEL = None
BENCHMARK_BEST = None
if Xy is not None and len(RES):
    hr("SECTION 26 — MODEL SELECTION")
    SIMPLICITY = {"DecisionTree_d3": 0, "LogisticRegression": 1, "DecisionTree_d6": 2,
                  "GradientBoosting": 3, "RandomForest": 4, "XGBoost": 5, "MLP_32_16": 6}

    def pick(pool, label):
        """Pilih model terbaik menurut validation, lalu tie-break: mobile-ready > sederhana > kecil."""
        best = pool["Validation_Macro_F1"].fillna(0).max()
        near = pool[pool["Validation_Macro_F1"].fillna(0) >= best - 0.01].copy()
        near["simplicity"] = near["Model"].map(SIMPLICITY).fillna(9)
        near["not_mobile"] = (~near["Mobile_Ready"]).astype(int)
        near = near.sort_values(["not_mobile", "simplicity", "n_features", "Model_Size_KB"])
        print(f"\n  [{label}] skor terbaik {best:.4f}; kandidat dalam margin 0.01: {len(near)}")
        print(near[["Model", "Feature_Set", "Validation_Macro_F1", "CV_Macro_F1",
                    "Model_Size_KB", "Mobile_Ready"]].head(6).to_string(index=False))
        return near.iloc[0]

    cand = RES[RES.Model != "DummyClassifier"].copy()

    # --- Skor terbaik apa pun feature set-nya (hanya untuk dilaporkan) ---
    BENCHMARK_BEST = pick(cand, "SKOR TERTINGGI APA PUN FEATURE SET-NYA")

    # --- SECTION 18: model produksi hanya boleh memakai fitur yang ADA di APK ---
    deployable = cand[cand["Deployable_Features"]]
    print(f"\n  Feature set yang lolos kontrak deployment: "
          f"{sorted(set(deployable['Feature_Set'])) if len(deployable) else 'TIDAK ADA'}")
    print(f"  Feature set yang DITOLAK (memakai fitur yang tidak ada di APK): "
          f"{sorted(set(cand['Feature_Set']) - set(deployable['Feature_Set']))}")

    if len(deployable):
        top = pick(deployable, "KANDIDAT YANG LAYAK DI-DEPLOY")
        FINAL_SET, FINAL_MODEL_NAME = top["Feature_Set"], top["Model"]
        FINAL_MODEL = FITTED[(FINAL_SET, FINAL_MODEL_NAME)]
        print(f"\n  MODEL FINAL (dikunci): {FINAL_MODEL_NAME} @ feature set {FINAL_SET}")
        print("  Alasan: satu-satunya kelompok yang seluruh fiturnya benar-benar tersedia di APK,")
        print("  lalu dipilih yang skornya dalam margin terbaik, paling sederhana, dan mobile-ready.")
        if BENCHMARK_BEST["Feature_Set"] != FINAL_SET:
            print(f"\n  CATATAN: kandidat teratas tanpa batasan deployment adalah "
                  f"{BENCHMARK_BEST['Model']} @ {BENCHMARK_BEST['Feature_Set']} (Val Macro-F1 "
                  f"{BENCHMARK_BEST['Validation_Macro_F1']:.4f}), TETAPI feature set itu memuat")
            print("  `visual_proxy_turbidity` yang TIDAK ADA di perangkat RonaAir. Sesuai Section 18,")
            print("  model tersebut adalah DEPLOYMENT FEATURE MISMATCH dan tidak boleh jadi model produksi.")
    else:
        print("\n  Tidak ada model yang lolos kontrak deployment -> tidak ada model final.")

    print("\n  PERINGATAN do_est: SET A memakai `do_est` yang pada dataset merupakan DO TERUKUR,")
    print("  sedangkan APK hanya memiliki DO ESTIMASI. Ini mismatch yang tersisa dan tercatat di")
    print("  deployment_feature_contract.csv; model ini hanya boleh dipakai sebagai benchmark")
    print("  sampai soft-sensor DO divalidasi terhadap DO meter.")


SECTION 26 — MODEL SELECTION

  [SKOR TERTINGGI APA PUN FEATURE SET-NYA] skor terbaik 1.0000; kandidat dalam margin 0.01: 19
          Model            Feature_Set  Validation_Macro_F1  CV_Macro_F1  Model_Size_KB  Mobile_Ready
DecisionTree_d3    B_visual_proxy_only               0.9978       0.9977            2.8          True
DecisionTree_d3   C_sensor_plus_visual               0.9978       0.9977            2.9          True
DecisionTree_d3 E_full_temporal_fusion               0.9958       0.9982            3.2          True
DecisionTree_d6    B_visual_proxy_only               0.9978       0.9977            3.1          True
DecisionTree_d6          A_sensor_only               0.9970       0.9937            4.1          True
DecisionTree_d6   C_sensor_plus_visual               0.9978       0.9977            3.0          True

  Feature set yang lolos kontrak deployment: ['A_sensor_only']
  Feature set yang DITOLAK (memakai fitur yang tidak ada di APK): ['B_visual_proxy_only', 'C_sens

## Cell 23 — Ablation study (Section 33)


In [24]:
if Xy is not None and len(RES):
    hr("SECTION 33 — ABLATION STUDY")
    abl = (RES[RES.Model != "DummyClassifier"]
           .sort_values("Validation_Macro_F1", ascending=False)
           .groupby("Feature_Set")
           .first()
           .reset_index()[["Feature_Set", "Model", "n_features", "CV_Macro_F1",
                           "Validation_Macro_F1", "Validation_Balanced_Accuracy",
                           "Validation_HighRisk_Recall", "Model_Size_KB",
                           "Inference_Time_ms_per_sample"]])
    abl = abl.sort_values("Validation_Macro_F1", ascending=False)
    abl_out = pd.concat([abl, pd.DataFrame([{
        "Feature_Set": "D_sensor_visual_ph_photo", "Model": "NOT EVALUABLE", "n_features": np.nan,
        "CV_Macro_F1": np.nan, "Validation_Macro_F1": np.nan, "Validation_Balanced_Accuracy": np.nan,
        "Validation_HighRisk_Recall": np.nan, "Model_Size_KB": np.nan,
        "Inference_Time_ms_per_sample": np.nan}])], ignore_index=True)
    abl_out["note"] = np.where(abl_out.Model == "NOT EVALUABLE",
                               "ph_visual_est / ph_difference tidak ada di dataset publik mana pun; "
                               "tidak disimulasikan agar tidak mengarang data.", "")
    print(abl_out.to_string(index=False))
    save_csv(abl_out, DIR_RESULTS / "ablation_results.csv")

    a = abl.set_index("Feature_Set")["Validation_Macro_F1"].to_dict()
    print("\n  INTERPRETASI ABLATION (berdasarkan angka di atas, bukan ekspektasi):")
    for k in ["A_sensor_only", "B_visual_proxy_only", "C_sensor_plus_visual",
              "E_full_temporal_fusion"]:
        print(f"    {k:24s}: {a.get(k, float('nan')):.4f}")
    gain = a.get("C_sensor_plus_visual", np.nan) - a.get("A_sensor_only", np.nan)
    print(f"\n    Selisih C - A = {gain:+.4f}")
    print("\n    Selisih ini TIDAK boleh dibaca sebagai 'fusion menambah informasi biologis'.")
    perf = D2_FINDINGS.get("features_that_fully_recover_label", [])
    print(f"    Pada dataset ini, parameter berikut MASING-MASING memulihkan label 100% sendirian:")
    print(f"      {perf}")
    print("    Karena itu TIDAK ADA feature set yang bebas dari kebocoran label: setiap kanal")
    print("    parameter sudah mengandung jawabannya. Skor tinggi di SEMUA set (termasuk SET A")
    print("    yang hanya memakai sensor) adalah konsekuensi cara data dibangkitkan, bukan bukti")
    print("    bahwa parameter tersebut memprediksi risiko biologis.")
    print(f"    Struktur generatif: {D2_FINDINGS.get('generative_structure')}")
    eg = a.get("E_full_temporal_fusion", np.nan) - a.get("C_sensor_plus_visual", np.nan)
    print(f"\n    Selisih E - C = {eg:+.4f}. Fitur temporal tidak memberi informasi tambahan yang")
    print("    berarti, konsisten dengan temuan bahwa struktur waktu dataset ini artifisial.")


SECTION 33 — ABLATION STUDY
             Feature_Set           Model  n_features  CV_Macro_F1  Validation_Macro_F1  Validation_Balanced_Accuracy  Validation_HighRisk_Recall  Model_Size_KB  Inference_Time_ms_per_sample                                                                                                               note
           A_sensor_only    RandomForest         5.0       0.9987               1.0000                        1.0000                      1.0000          599.7                        0.0274                                                                                                                   
    C_sensor_plus_visual    RandomForest         6.0       0.9987               1.0000                        1.0000                      1.0000          429.5                        0.0310                                                                                                                   
  E_full_temporal_fusion       MLP_32_16        19.0     

## Cell 24 — Evaluasi TEST (dijalankan SEKALI setelah model dikunci, Section 54)


In [25]:
TEST_METRICS = {}
if Xy is not None and FINAL_MODEL is not None:
    hr("SECTION 54 — EVALUASI TEST SET (model sudah dikunci)")
    feats = FEAT_SETS[FINAL_SET]
    yp_test = FINAL_MODEL.predict(test[feats])
    TEST_METRICS = eval_block(y_te, yp_test)
    for k, v in TEST_METRICS.items():
        print(f"  {k:26s}: {v:.4f}" if isinstance(v, float) else f"  {k:26s}: {v}")

    print("\n  Classification report (test):")
    print(classification_report(y_te, yp_test, labels=RONAAIR_LABELS, zero_division=0))

    # isikan kolom test hanya untuk baris model final
    mask = (RES.Model == FINAL_MODEL_NAME) & (RES.Feature_Set == FINAL_SET)
    RES.loc[mask, "Test_Macro_F1"] = round(TEST_METRICS["Macro_F1"], 4)
    RES.loc[mask, "Test_Balanced_Accuracy"] = round(TEST_METRICS["Balanced_Accuracy"], 4)
    RES.loc[mask, "Test_HighRisk_Recall"] = round(TEST_METRICS["HighRisk_Recall"], 4)
    RES["Is_Final_Model"] = mask
    save_csv(RES, DIR_RESULTS / "risk_model_comparison.csv")

    # --- Confusion matrix (Section 28) ---
    cm = confusion_matrix(y_te, yp_test, labels=RONAAIR_LABELS)
    cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in RONAAIR_LABELS],
                         columns=[f"pred_{l}" for l in RONAAIR_LABELS])
    print("\n  Confusion matrix (test):")
    print(cm_df.to_string())
    save_csv(cm_df.reset_index().rename(columns={"index": "true_label"}),
             DIR_RESULTS / "confusion_matrix.csv")

    fig, ax = plt.subplots(figsize=(5, 4.2))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(4)); ax.set_xticklabels(RONAAIR_LABELS, rotation=45, ha="right")
    ax.set_yticks(range(4)); ax.set_yticklabels(RONAAIR_LABELS)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xlabel("Prediksi"); ax.set_ylabel("Label sebenarnya")
    ax.set_title(f"Confusion matrix — {FINAL_MODEL_NAME} ({FINAL_SET})")
    plt.colorbar(im); plt.tight_layout()
    plt.savefig(DIR_FIGS / "confusion_matrix.png", dpi=130); plt.close()

    # --- High-risk error analysis (Section 28 & 29) ---
    err = test.copy()
    err["y_true"] = y_te.values
    err["y_pred"] = yp_test
    err["sev_true"] = err["y_true"].map(SEVERITY)
    err["sev_pred"] = err["y_pred"].map(SEVERITY)
    err["severity_gap"] = err["sev_true"] - err["sev_pred"]   # >0 = model meremehkan risiko
    hi_err = err[(err["y_true"].isin(HIGH_RISK)) & (err["y_true"] != err["y_pred"])]
    print(f"\n  Kesalahan pada kelas risiko tinggi (SIAGA/DARURAT): {len(hi_err)} dari "
          f"{int(err['y_true'].isin(HIGH_RISK).sum())} observasi risiko tinggi")
    crit = err[(err.y_true == "DARURAT") & (err.y_pred == "NORMAL")]
    print(f"  Kesalahan paling berbahaya DARURAT -> NORMAL: {len(crit)}")
    print(f"  SIAGA -> NORMAL: {len(err[(err.y_true=='SIAGA') & (err.y_pred=='NORMAL')])}")
    print(f"  DARURAT -> WASPADA: {len(err[(err.y_true=='DARURAT') & (err.y_pred=='WASPADA')])}")
    print(f"  Rata-rata severity gap (underestimation positif): {err['severity_gap'].mean():+.4f}")
    cols_keep = ["ts", "y_true", "y_pred", "sev_true", "sev_pred", "severity_gap"] + FEAT_SETS[FINAL_SET]
    save_csv(hi_err[cols_keep], DIR_RESULTS / "high_risk_errors.csv")
    TEST_METRICS["mean_severity_gap"] = float(err["severity_gap"].mean())
    TEST_METRICS["n_darurat_to_normal"] = int(len(crit))


SECTION 54 — EVALUASI TEST SET (model sudah dikunci)
  Macro_F1                  : 0.9928
  Balanced_Accuracy         : 0.9929
  Accuracy                  : 0.9930
  Weighted_F1               : 0.9930
  HighRisk_Recall           : 0.9941
  HighRisk_Precision        : 1.0000
  HighRisk_F1               : 0.9970
  HighRisk_FalseNegatives   : 1

  Classification report (test):
              precision    recall  f1-score   support

      NORMAL       0.99      1.00      1.00       128
     WASPADA       0.99      0.99      0.99       134
       SIAGA       1.00      0.98      0.99        96
     DARURAT       0.99      1.00      0.99        73

    accuracy                           0.99       431
   macro avg       0.99      0.99      0.99       431
weighted avg       0.99      0.99      0.99       431

  [saved] risk_model_comparison.csv  (28 rows)

  Confusion matrix (test):
              pred_NORMAL  pred_WASPADA  pred_SIAGA  pred_DARURAT
true_NORMAL           128             0        

## Cell 25 — Feature importance (Section 37)

Feature importance menjelaskan **kontribusi terhadap prediksi model**, bukan kausalitas biologis.


In [26]:
FI_DF = pd.DataFrame()
if Xy is not None and FINAL_MODEL is not None:
    hr("SECTION 37 — FEATURE IMPORTANCE")
    feats = FEAT_SETS[FINAL_SET]
    rows = []
    clf = FINAL_MODEL.named_steps["clf"] if hasattr(FINAL_MODEL, "named_steps") else FINAL_MODEL
    if hasattr(clf, "feature_importances_"):
        for f, v in zip(feats, clf.feature_importances_):
            rows.append({"feature": f, "method": "impurity_importance", "value": float(v)})
    if hasattr(clf, "coef_"):
        for ci, cls in enumerate(clf.classes_):
            for f, v in zip(feats, np.atleast_2d(clf.coef_)[ci]):
                rows.append({"feature": f, "method": f"logreg_coef[{cls}]", "value": float(v)})
    perm = permutation_importance(FINAL_MODEL, valid[feats], y_va, n_repeats=10,
                                  random_state=RANDOM_SEED,
                                  scoring="f1_macro")
    for f, m, s in zip(feats, perm.importances_mean, perm.importances_std):
        rows.append({"feature": f, "method": "permutation_importance_valid_macroF1",
                     "value": float(m), "std": float(s)})
    FI_DF = pd.DataFrame(rows)
    piv = FI_DF[FI_DF.method.str.startswith("permutation")].sort_values("value", ascending=False)
    print("  Permutation importance (validation, Macro-F1):")
    print(piv[["feature", "value", "std"]].round(4).to_string(index=False))
    save_csv(FI_DF, DIR_RESULTS / "feature_importance.csv")
    print("\n  Kalimat yang benar : 'fitur X berkontribusi kuat terhadap prediksi model'.")
    print("  Kalimat yang SALAH : 'fitur X menyebabkan kematian ikan'.")

    if len(piv):
        fig, ax = plt.subplots(figsize=(6.5, 0.4 * len(piv) + 1.6))
        ax.barh(piv["feature"][::-1], piv["value"][::-1],
                xerr=piv["std"][::-1] if "std" in piv else None)
        ax.set_xlabel("Penurunan Macro-F1 saat fitur diacak")
        ax.set_title("Permutation importance (validation)")
        plt.tight_layout(); plt.savefig(DIR_FIGS / "feature_importance.png", dpi=130); plt.close()


SECTION 37 — FEATURE IMPORTANCE
  Permutation importance (validation, Macro-F1):
   feature  value    std
    do_est 0.3421 0.0083
water_temp 0.3282 0.0128
  ec_value 0.2587 0.0142
 ph_sensor 0.0000 0.0000
   tds_ppm 0.0000 0.0000
  [saved] feature_importance.csv  (10 rows)

  Kalimat yang benar : 'fitur X berkontribusi kuat terhadap prediksi model'.
  Kalimat yang SALAH : 'fitur X menyebabkan kematian ikan'.


## Cell 26 — External validation (Section 34 & 55)


In [27]:
hr("SECTION 34/55 — EXTERNAL VALIDATION")
EXTERNAL_VALIDATION = {
    "status": "EXTERNAL VALIDATION NOT AVAILABLE",
    "attempted_pairs": [],
}
reasons = []
reasons.append({"train_on": "D2_FISHPOND_MULTICLASS", "test_on": "D3_IOTMLCQ_2024",
                "possible": False,
                "reason": "Semantik target tidak kompatibel. D2 memakai 4 tingkat berbasis turbidity; "
                          "D3 memakai 2 tingkat berbasis threshold suhu dan tidak memiliki turbidity "
                          "pada rentang yang sebanding. Memetakan Stable/At Risk ke 4 kelas akan "
                          "mengarang informasi."})
reasons.append({"train_on": "D2_FISHPOND_MULTICLASS", "test_on": "D4_AQUAPONIC_IOT",
                "possible": False,
                "reason": "D4 tidak memiliki label risiko sama sekali, dan tidak memiliki DO maupun "
                          "turbidity. Tidak ada target untuk dievaluasi."})
reasons.append({"train_on": "D2_FISHPOND_MULTICLASS", "test_on": "D1_AQUAPONDS",
                "possible": False, "reason": "D1 tidak berisi observasi."})
EXTERNAL_VALIDATION["attempted_pairs"] = reasons
for r in reasons:
    print(f"  Train {r['train_on']} -> Test {r['test_on']}: {'YA' if r['possible'] else 'TIDAK'}")
    print(f"    {r['reason']}")
print("\n  *** EXTERNAL VALIDATION NOT AVAILABLE ***")
print("  Notebook ini TIDAK memalsukan external validation dengan melatih pada gabungan dataset.")
save_json(EXTERNAL_VALIDATION, DIR_RESULTS / "external_validation_status.json")


SECTION 34/55 — EXTERNAL VALIDATION
  Train D2_FISHPOND_MULTICLASS -> Test D3_IOTMLCQ_2024: TIDAK
    Semantik target tidak kompatibel. D2 memakai 4 tingkat berbasis turbidity; D3 memakai 2 tingkat berbasis threshold suhu dan tidak memiliki turbidity pada rentang yang sebanding. Memetakan Stable/At Risk ke 4 kelas akan mengarang informasi.
  Train D2_FISHPOND_MULTICLASS -> Test D4_AQUAPONIC_IOT: TIDAK
    D4 tidak memiliki label risiko sama sekali, dan tidak memiliki DO maupun turbidity. Tidak ada target untuk dievaluasi.
  Train D2_FISHPOND_MULTICLASS -> Test D1_AQUAPONDS: TIDAK
    D1 tidak berisi observasi.

  *** EXTERNAL VALIDATION NOT AVAILABLE ***
  Notebook ini TIDAK memalsukan external validation dengan melatih pada gabungan dataset.
  [saved] results/external_validation_status.json


## Cell 27 — Recommendation Engine (Section 41 & 42)

**Tidak** memakai LLM generatif untuk menentukan tindakan. Rekomendasi berasal dari
tabel rule yang dapat diaudit, dengan kontraindikasi dan status validasi eksplisit.


In [28]:
NEEDS_VAL = "NEEDS_EXPERT_VALIDATION"
SRC_NONE = ("Belum ditautkan ke SOP tertulis. Wajib divalidasi oleh penyuluh/ahli budidaya "
            "sebelum dipakai di lapangan.")

RECOMMENDATION_RULES = {
    "schema_version": "1.0.0",
    "model_version": MODEL_VERSION,
    "generated_at": stamp(),
    "engine_type": "deterministic rule table (BUKAN LLM generatif)",
    "global_status": "PROVISIONAL",
    "disclaimer": ("Rekomendasi bersifat decision support. Bukan kepastian medis, bukan jaminan "
                   "keselamatan, dan bukan pengganti penilaian ahli budidaya."),
    "recommendations": [
        {"recommendation_id": "RISK_000", "risk_status": "INSUFFICIENT_EVIDENCE",
         "trigger_condition": "parameter wajib hilang ATAU foto gagal quality check",
         "action": "Jangan mengambil kesimpulan. Ulangi pengukuran: periksa koneksi sensor, "
                   "bersihkan probe, dan ambil ulang foto sesuai panduan pencahayaan RonaCard.",
         "priority": 1, "contraindication": "Tidak ada.", "recheck_after_minutes": 15,
         "source": SRC_NONE, "validation_status": NEEDS_VAL},
        {"recommendation_id": "RISK_001", "risk_status": "NORMAL",
         "trigger_condition": "tidak ada rule risiko yang aktif",
         "action": "Lanjutkan pemantauan rutin sesuai jadwal SOP tambak.",
         "priority": 4, "contraindication": "Tidak ada.", "recheck_after_minutes": 360,
         "source": SRC_NONE, "validation_status": NEEDS_VAL},
        {"recommendation_id": "RISK_002", "risk_status": "WASPADA",
         "trigger_condition": "minimal satu rule tingkat WASPADA aktif",
         "action": "Periksa ulang kualitas air dan evaluasi pemberian pakan sesuai SOP. "
                   "Perpendek interval pemantauan.",
         "priority": 3, "contraindication": "Jangan mengubah kimia air hanya berdasarkan satu pembacaan.",
         "recheck_after_minutes": 120, "source": SRC_NONE, "validation_status": NEEDS_VAL},
        {"recommendation_id": "RISK_003", "risk_status": "SIAGA",
         "trigger_condition": "minimal satu rule tingkat SIAGA aktif",
         "action": "Verifikasi pembacaan dengan alat ukur pembanding, siapkan aerasi tambahan, "
                   "dan tinjau tindakan korektif sesuai SOP tambak.",
         "priority": 2, "contraindication": "Hindari pergantian air besar mendadak tanpa mengukur "
                                            "suhu dan pH air pengganti.",
         "recheck_after_minutes": 30, "source": SRC_NONE, "validation_status": NEEDS_VAL},
        {"recommendation_id": "RISK_004", "risk_status": "DARURAT",
         "trigger_condition": "minimal satu rule tingkat DARURAT aktif",
         "action": "Tangani segera sesuai SOP kedaruratan tambak dan hubungi penyuluh/teknisi. "
                   "Verifikasi ulang pembacaan sambil menyiapkan aerasi darurat.",
         "priority": 1, "contraindication": "Jangan menunda tindakan hanya untuk menunggu "
                                            "konfirmasi laboratorium.",
         "recheck_after_minutes": 10, "source": SRC_NONE, "validation_status": NEEDS_VAL},
    ],
    "trigger_specific": [
        {"recommendation_id": "TRG_DO_LOW", "trigger_rules": ["R001", "R002", "R003"],
         "action": "Bukti menunjuk oksigen terlarut rendah (ESTIMATED DO, bukan hasil ukur langsung). "
                   "Konfirmasi dengan DO meter bila ada, dan periksa aerator.",
         "contraindication": "Estimasi DO belum tervalidasi terhadap DO meter; jangan dipakai "
                             "sebagai satu-satunya dasar keputusan besar.",
         "source": SRC_NONE, "validation_status": NEEDS_VAL},
        {"recommendation_id": "TRG_PH_EXTREME", "trigger_rules": ["R010", "R011", "R012", "R013"],
         "action": "Bukti menunjuk pH di luar kisaran umum. Kalibrasi ulang probe pH sebelum "
                   "melakukan tindakan kimia apa pun.",
         "contraindication": "Jangan menambahkan kapur/asam tanpa SOP dan pengukuran ulang.",
         "source": SRC_NONE, "validation_status": NEEDS_VAL},
        {"recommendation_id": "TRG_TEMP", "trigger_rules": ["R020", "R021"],
         "action": "Suhu air di luar kisaran nyaman. Tinjau naungan, kedalaman, dan jadwal pakan.",
         "contraindication": "Tidak ada.", "source": SRC_NONE, "validation_status": NEEDS_VAL},
        {"recommendation_id": "TRG_VISUAL", "trigger_rules": ["R030", "R031"],
         "action": "Rona air berubah. Ini indikator VISUAL saja — tidak menunjukkan spesies alga "
                   "maupun konsentrasi toksin. Amati ulang pada pencahayaan yang sama.",
         "contraindication": "Dilarang menyimpulkan HAB atau toksisitas dari foto.",
         "source": SRC_NONE, "validation_status": NEEDS_VAL},
        {"recommendation_id": "TRG_PH_DISAGREE", "trigger_rules": ["R040"],
         "action": "pH dari foto dan pH sensor tidak sepakat. Ini sinyal KUALITAS DATA: "
                   "kalibrasi ulang probe dan foto ulang strip pada pencahayaan yang benar.",
         "contraindication": "Jangan menganggap salah satu nilai otomatis benar.",
         "source": SRC_NONE, "validation_status": NEEDS_VAL},
    ],
}
save_json(RECOMMENDATION_RULES, DIR_CONFIGS / "recommendation_rules.json")
save_json(RECOMMENDATION_RULES, DIR_DEPLOY / "recommendation_rules.json")
hr("SECTION 41/42 — RECOMMENDATION ENGINE")
print(f"  {len(RECOMMENDATION_RULES['recommendations'])} rekomendasi per-status + "
      f"{len(RECOMMENDATION_RULES['trigger_specific'])} rekomendasi per-pemicu")
print(f"  Status global: {RECOMMENDATION_RULES['global_status']}")
print("  Tidak ada rekomendasi berangka spesifik (mis. 'kurangi pakan 27%') tanpa SOP pendukung.")


  [saved] configs/recommendation_rules.json
  [saved] deployment/recommendation_rules.json
SECTION 41/42 — RECOMMENDATION ENGINE
  5 rekomendasi per-status + 5 rekomendasi per-pemicu
  Status global: PROVISIONAL
  Tidak ada rekomendasi berangka spesifik (mis. 'kurangi pakan 27%') tanpa SOP pendukung.


## Cell 28 — Fungsi inference `predict_risk()` (Section 43 & 46)

Jalur **produksi** adalah rule engine (Level 1). Keluaran model ML disertakan hanya
sebagai pembanding eksperimental dan diberi label demikian.


In [29]:
def recommend(risk_status, fired_rules, cfg=RECOMMENDATION_RULES):
    out = []
    for r in cfg["recommendations"]:
        if r["risk_status"] == risk_status:
            out.append({"id": r["recommendation_id"], "action": r["action"],
                        "priority": r["priority"], "contraindication": r["contraindication"],
                        "recheck_after_minutes": r["recheck_after_minutes"],
                        "validation_status": r["validation_status"]})
    for t in cfg["trigger_specific"]:
        if set(t["trigger_rules"]) & set(fired_rules):
            out.append({"id": t["recommendation_id"], "action": t["action"],
                        "priority": 2, "contraindication": t["contraindication"],
                        "recheck_after_minutes": None,
                        "validation_status": t["validation_status"]})
    return sorted(out, key=lambda x: x["priority"])

def explain(risk_status, factors, quality, recs):
    lines = [
        "APA YANG TERJADI?",
        f"  Status: {risk_status}   (kualitas data: {quality})",
        "MENGAPA?",
    ]
    lines += [f"  - {f}" for f in factors]
    lines += ["APA YANG PERLU DIPERIKSA & DILAKUKAN?"]
    lines += [f"  - [{r['id']}] {r['action']}" for r in recs]
    contra = [r["contraindication"] for r in recs if r.get("contraindication") and r["contraindication"] != "Tidak ada."]
    if contra:
        lines += ["KONTRAINDIKASI:"] + [f"  - {c}" for c in contra]
    nxt = [r["recheck_after_minutes"] for r in recs if r.get("recheck_after_minutes")]
    lines += ["KAPAN PERIKSA ULANG?",
              f"  - Ulangi pengukuran dalam ~{min(nxt)} menit sesuai protokol pemantauan."
              if nxt else "  - Ikuti jadwal pemantauan SOP."]
    return "\n".join(lines)

def predict_risk(observation, use_ml_benchmark=True):
    """Jalur keputusan RonaAir. observation: dict berisi fitur deployment."""
    res = rule_engine(observation)
    recs = recommend(res["risk_status"], res["fired_rules"])
    out = {
        "risk_status": res["risk_status"],
        "supporting_factors": res["supporting_factors"],
        "fired_rules": res["fired_rules"],
        "data_quality": res["data_quality"],
        "out_of_distribution_notes": res["ood_notes"],
        "recommendations": recs,
        "model_version": MODEL_VERSION,
        "decision_source": "RULE_ENGINE_LEVEL_1",
        "timestamp": stamp(),
    }
    if use_ml_benchmark and FINAL_MODEL is not None:
        feats = FEAT_SETS[FINAL_SET]
        missing = [f for f in feats if observation.get(f) is None]
        if missing:
            out["ml_benchmark"] = {
                "status": "UNAVAILABLE",
                "reason": f"fitur benchmark tidak tersedia pada observasi ini: {missing}",
                "caveat": "Model ML hanya benchmark eksperimental; ia meniru aturan turbidity "
                          "dari dataset publik dan bukan sumber keputusan produksi.",
            }
        else:
            row = pd.DataFrame([{f: observation[f] for f in feats}])
            pred = FINAL_MODEL.predict(row)[0]
            out["ml_benchmark"] = {
                "status": "OK", "predicted_class": str(pred),
                "model": FINAL_MODEL_NAME, "feature_set": FINAL_SET,
                "caveat": "EXPERIMENTAL. Model ini mereproduksi logika keputusan dataset publik "
                          "(threshold turbidity), bukan prediksi risiko biologis independen.",
            }
    out["explanation"] = explain(out["risk_status"], out["supporting_factors"],
                                 out["data_quality"], recs)
    return out

hr("SECTION 43/46 — DEMONSTRASI predict_risk()")
demo_cases = [
    ("Sensor nyata D4, tanpa DO & tanpa foto",
     {"water_temp": 24.3, "ph_sensor": 7.5, "tds_ppm": 330.0, "ec_value": None,
      "do_est": None, "visual_score": None, "ph_visual_est": None, "hour": 9}),
    ("DO estimasi rendah + rona air berubah",
     {"water_temp": 31.2, "ph_sensor": 8.1, "tds_ppm": 410.0, "ec_value": 820.0,
      "do_est": 2.4, "visual_score": 0.82, "ph_visual_est": 8.0, "image_quality": "OK", "hour": 5}),
    ("Sensor rusak (pH mustahil)",
     {"water_temp": 25.0, "ph_sensor": 15.8, "tds_ppm": 330.0, "ec_value": None,
      "do_est": None, "visual_score": None, "hour": 14}),
    ("Data terlalu sedikit",
     {"water_temp": None, "ph_sensor": None, "tds_ppm": None, "ec_value": None,
      "do_est": None, "visual_score": None, "hour": 3}),
]
for title, obs in demo_cases:
    print(f"\n{'-'*70}\nKASUS: {title}")
    r = predict_risk(obs)
    print(r["explanation"])
    if "ml_benchmark" in r:
        print(f"  [ML benchmark] {r['ml_benchmark'].get('predicted_class', r['ml_benchmark']['status'])}")


SECTION 43/46 — DEMONSTRASI predict_risk()

----------------------------------------------------------------------
KASUS: Sensor nyata D4, tanpa DO & tanpa foto
APA YANG TERJADI?
  Status: INSUFFICIENT_EVIDENCE   (kualitas data: INSUFFICIENT_EVIDENCE)
MENGAPA?
  - tidak ada bukti oksigen maupun visual: ['do_est', 'visual_score']
APA YANG PERLU DIPERIKSA & DILAKUKAN?
  - [RISK_000] Jangan mengambil kesimpulan. Ulangi pengukuran: periksa koneksi sensor, bersihkan probe, dan ambil ulang foto sesuai panduan pencahayaan RonaCard.
KAPAN PERIKSA ULANG?
  - Ulangi pengukuran dalam ~15 menit sesuai protokol pemantauan.
  [ML benchmark] UNAVAILABLE

----------------------------------------------------------------------
KASUS: DO estimasi rendah + rona air berubah
APA YANG TERJADI?
  Status: SIAGA   (kualitas data: POSSIBLE_OUT_OF_DISTRIBUTION)
MENGAPA?
  - [R002] do_est=2.4 < 3.0 mg/L -> SIAGA
  - [R003] do_est=2.4 < 5.0 mg/L -> WASPADA
  - [R030] visual_score=0.82 >= 0.75 skor 0-1 -> SIAGA
  - 

## Cell 29 — Input & Output contract (Section 44 & 45)


In [30]:
def _env(p):
    e = EMPIRICAL_ENVELOPE.get(p)
    return [e["p01"], e["p99"]] if e else None

INPUT_SCHEMA = {
    "schema_version": "1.0.0", "model_version": MODEL_VERSION,
    "description": "Kontrak input Risk Engine RonaAir. Semua field boleh null; mesin akan "
                   "menurunkan kualitas data alih-alih menebak.",
    "features": [
        {"feature_name": "visual_score", "data_type": "float", "unit": "skor 0-1", "required": False,
         "source": DEPLOYMENT_FEATURES["visual_score"], "valid_range": [0.0, 1.0],
         "fallback": "null -> kanal visual diabaikan, data_quality turun ke DEGRADED"},
        {"feature_name": "visual_condition", "data_type": "string", "unit": "kelas CV", "required": False,
         "source": DEPLOYMENT_FEATURES["visual_condition"], "valid_range": None,
         "fallback": "null -> hanya dipakai untuk penjelasan, bukan keputusan"},
        {"feature_name": "image_quality", "data_type": "string", "unit": "OK|FAIL", "required": False,
         "source": DEPLOYMENT_FEATURES["image_quality"], "valid_range": ["OK", "FAIL"],
         "fallback": "FAIL -> visual_score diabaikan"},
        {"feature_name": "ph_visual_est", "data_type": "float", "unit": "pH", "required": False,
         "source": DEPLOYMENT_FEATURES["ph_visual_est"], "valid_range": [0.0, 14.0],
         "fallback": "null -> ph_difference tidak dihitung",
         "note": "Estimasi dari model PolynomialRidge2/COMBINED_ALL. Hanya valid pada rentang "
                 "validasi model pH tersebut; di luar itu WAJIB memunculkan warning, bukan "
                 "ekstrapolasi diam-diam."},
        {"feature_name": "ph_sensor", "data_type": "float", "unit": "pH", "required": True,
         "source": DEPLOYMENT_FEATURES["ph_sensor"], "valid_range": [0.0, 14.0],
         "empirical_envelope": _env("ph_sensor"),
         "fallback": "null -> INSUFFICIENT_EVIDENCE"},
        {"feature_name": "water_temp", "data_type": "float", "unit": "degC", "required": True,
         "source": DEPLOYMENT_FEATURES["water_temp"], "valid_range": [0.0, 45.0],
         "empirical_envelope": _env("water_temp"),
         "fallback": "null -> INSUFFICIENT_EVIDENCE"},
        {"feature_name": "ec_value", "data_type": "float", "unit": "uS/cm", "required": False,
         "source": DEPLOYMENT_FEATURES["ec_value"], "valid_range": [0.0, 100000.0],
         "fallback": "null -> diabaikan"},
        {"feature_name": "tds_ppm", "data_type": "float", "unit": "ppm", "required": False,
         "source": DEPLOYMENT_FEATURES["tds_ppm"], "valid_range": [0.0, 50000.0],
         "empirical_envelope": _env("tds_ppm"),
         "fallback": "null -> diabaikan"},
        {"feature_name": "do_est", "data_type": "float", "unit": "mg/L", "required": False,
         "source": DEPLOYMENT_FEATURES["do_est"], "valid_range": [0.0, 25.0],
         "fallback": "null -> DEGRADED; bila visual_score juga null -> INSUFFICIENT_EVIDENCE",
         "note": "ESTIMATED DO (soft-sensor). Wajib ditampilkan sebagai 'Estimated DO' di UI."},
        {"feature_name": "hour", "data_type": "int", "unit": "0-23", "required": False,
         "source": DEPLOYMENT_FEATURES["hour"], "valid_range": [0, 23],
         "fallback": "null -> konteks siang/malam tidak dipakai"},
    ],
}
OUTPUT_SCHEMA = {
    "schema_version": "1.0.0", "model_version": MODEL_VERSION,
    "fields": [
        {"name": "risk_status", "type": "string",
         "values": RONAAIR_LABELS + ["INSUFFICIENT_EVIDENCE"]},
        {"name": "supporting_factors", "type": "array<string>",
         "description": "Rule yang aktif beserta nilai pemicunya."},
        {"name": "fired_rules", "type": "array<string>", "description": "Daftar rule_id."},
        {"name": "data_quality", "type": "string",
         "values": ["OK", "DEGRADED", "POSSIBLE_OUT_OF_DISTRIBUTION", "INSUFFICIENT_EVIDENCE"]},
        {"name": "out_of_distribution_notes", "type": "array<string>"},
        {"name": "recommendations", "type": "array<object>"},
        {"name": "explanation", "type": "string"},
        {"name": "model_version", "type": "string"},
        {"name": "timestamp", "type": "string (ISO-like)"},
        {"name": "decision_source", "type": "string", "values": ["RULE_ENGINE_LEVEL_1"]},
        {"name": "ml_benchmark", "type": "object|absent",
         "description": "Keluaran benchmark eksperimental. BUKAN keputusan produksi."},
    ],
    "risk_probability": {
        "present": False,
        "reason": "Probabilitas TIDAK diekspor. Model ML dilatih pada label turunan aturan, "
                  "sehingga probabilitasnya tidak terkalibrasi terhadap risiko dunia nyata dan "
                  "akan menyesatkan bila ditampilkan sebagai 'peluang bahaya'.",
    },
}
save_json(INPUT_SCHEMA, DIR_DEPLOY / "risk_input_schema.json")
save_json(OUTPUT_SCHEMA, DIR_DEPLOY / "risk_output_schema.json")
hr("SECTION 44/45 — INPUT & OUTPUT CONTRACT")
print(f"  {len(INPUT_SCHEMA['features'])} fitur input terdefinisi; "
      f"{len(OUTPUT_SCHEMA['fields'])} field output.")
print("  risk_probability sengaja TIDAK diekspor (lihat alasan di schema).")


  [saved] deployment/risk_input_schema.json
  [saved] deployment/risk_output_schema.json
SECTION 44/45 — INPUT & OUTPUT CONTRACT
  10 fitur input terdefinisi; 11 field output.
  risk_probability sengaja TIDAK diekspor (lihat alasan di schema).


## Cell 30 — Export artefak model (Section 50)


In [31]:
hr("SECTION 50 — EXPORT ARTEFAK MODEL")
MODEL_PARAMS = {"exportable_to_native": False}
if FINAL_MODEL is not None:
    joblib.dump(FINAL_MODEL, DIR_MODELS / "risk_model.joblib")
    print(f"  [saved] models/risk_model.joblib")
    pre = Pipeline([s for s in FINAL_MODEL.steps if s[0] != "clf"]) if hasattr(FINAL_MODEL, "steps") else None
    if pre is not None:
        joblib.dump(pre, DIR_MODELS / "risk_preprocessor.joblib")
        print(f"  [saved] models/risk_preprocessor.joblib")

    feats = FEAT_SETS[FINAL_SET]
    clf = FINAL_MODEL.named_steps["clf"]
    MODEL_PARAMS = {"model_type": FINAL_MODEL_NAME, "feature_set": FINAL_SET,
                    "features": feats, "classes": list(map(str, clf.classes_)),
                    "model_version": MODEL_VERSION, "exportable_to_native": False}
    if isinstance(clf, DecisionTreeClassifier):
        t = clf.tree_
        MODEL_PARAMS.update({
            "exportable_to_native": True,
            "representation": "decision_tree",
            "nodes": [
                {"id": int(i),
                 "feature": feats[t.feature[i]] if t.feature[i] >= 0 else None,
                 "threshold": float(t.threshold[i]) if t.feature[i] >= 0 else None,
                 "left": int(t.children_left[i]), "right": int(t.children_right[i]),
                 "class": str(clf.classes_[int(np.argmax(t.value[i][0]))]) if t.children_left[i] == -1 else None}
                for i in range(t.node_count)],
        })
    elif isinstance(clf, LogisticRegression):
        sc = FINAL_MODEL.named_steps.get("sc")
        imp = FINAL_MODEL.named_steps.get("imp")
        MODEL_PARAMS.update({
            "exportable_to_native": True, "representation": "multinomial_logistic_regression",
            "impute_median": [float(v) for v in imp.statistics_] if imp is not None else None,
            "scaler_mean": [float(v) for v in sc.mean_] if sc is not None else None,
            "scaler_scale": [float(v) for v in sc.scale_] if sc is not None else None,
            "coef": [[float(v) for v in row] for row in np.atleast_2d(clf.coef_)],
            "intercept": [float(v) for v in np.atleast_1d(clf.intercept_)],
        })
    save_json(MODEL_PARAMS, DIR_MODELS / "risk_model_parameters.json")

    FEATURE_SCHEMA = {
        "model_version": MODEL_VERSION, "feature_set": FINAL_SET, "features": [],
    }
    for f in feats:
        s = Xy[Xy.split != "test"][f]
        FEATURE_SCHEMA["features"].append({
            "name": f, "dtype": "float",
            "training_min": float(s.min()), "training_max": float(s.max()),
            "training_p01": float(s.quantile(0.01)), "training_p99": float(s.quantile(0.99)),
            "available_in_apk": f in DEPLOYMENT_FEATURES,
            "note": ("PROXY kanal visual yang belum tervalidasi; tidak tersedia di APK"
                     if f == "visual_proxy_turbidity" else
                     "DEPLOYMENT MISMATCH: training memakai DO terukur, APK memakai DO estimasi"
                     if f == "do_est" else ""),
        })
    save_json(FEATURE_SCHEMA, DIR_MODELS / "risk_feature_schema.json")

    METADATA = {
        "model_version": MODEL_VERSION, "notebook_version": NOTEBOOK_VERSION,
        "created_at": stamp(),
        "final_model": FINAL_MODEL_NAME, "feature_set": FINAL_SET, "features": feats,
        "training_dataset": TRAINING_DATASET,
        "label_type": D2_FINDINGS.get("label_type"),
        "split_strategy": SPLIT_INFO,
        "metrics_test": {k: (round(float(v), 4) if isinstance(v, (int, float, np.floating)) else v)
                         for k, v in TEST_METRICS.items()},
        "rule_engine_vs_dataset_label": RULE_VS_D2,
        "external_validation": EXTERNAL_VALIDATION["status"],
        "production_decision_path": "RULE_ENGINE_LEVEL_1 (model ML bersifat benchmark eksperimental)",
        "scientific_caveat": ("Model ini mereproduksi aturan keputusan yang terdokumentasi pada "
                              "dataset publik. Model ini tidak menetapkan prediksi risiko biologis "
                              "atau kausal secara independen."),
        "tflite_export": {
            "performed": False,
            "reason": "Model final adalah model tabular scikit-learn kecil yang dapat "
                      "diimplementasikan langsung di Kotlin lewat risk_model_parameters.json. "
                      "Konversi ke TFLite akan menambah dependensi runtime tanpa manfaat.",
        },
    }
    save_json(METADATA, DIR_MODELS / "risk_model_metadata.json")
else:
    print("  Tidak ada model final yang dikunci -> artefak model tidak diekspor.")
    save_json({"status": "NO MODEL", "reason": "data training tidak memenuhi syarat"},
              DIR_MODELS / "risk_model_metadata.json")


SECTION 50 — EXPORT ARTEFAK MODEL
  [saved] models/risk_model.joblib
  [saved] models/risk_preprocessor.joblib
  [saved] models/risk_model_parameters.json
  [saved] models/risk_feature_schema.json
  [saved] models/risk_model_metadata.json


## Cell 31 — Android deployment: `RiskEngine.kt` (Section 47 & 48)

File Kotlin dibangkitkan **dari `configs/risk_rules.json`** agar tidak pernah keluar sinkron
dengan notebook. Yang di-deploy adalah **rule engine**, karena itulah jalur keputusan produksi
dan ia berjalan penuh offline tanpa runtime Python.


In [32]:
def kotlin_escape(s):
    return str(s).replace("\\", "\\\\").replace('"', '\\"').replace("\n", " ")

rule_lines = []
for r in RISK_RULES["rules"]:
    rule_lines.append(
        f'        Rule("{r["rule_id"]}", "{r["parameter"]}", "{r["operator"]}", '
        f'{float(r["threshold"])}f, "{r["unit"]}", RiskLevel.{r["risk_level"]}, '
        f'"{kotlin_escape(r["validation_status"])}"),'
    )
plaus_lines = [f'        "{k}" to ({v[0]}f to {v[1]}f),'
               for k, v in RISK_RULES["plausibility_ranges"].items()]
env_lines = [f'        "{k}" to ({v["p01"]}f to {v["p99"]}f),'
             for k, v in EMPIRICAL_ENVELOPE.items()]

rec_lines = []
for r in RECOMMENDATION_RULES["recommendations"]:
    st = r["risk_status"]
    key = st if st in RONAAIR_LABELS else "INSUFFICIENT_EVIDENCE"
    rec_lines.append(
        f'        "{key}" to Recommendation("{r["recommendation_id"]}", '
        f'"{kotlin_escape(r["action"])}", {r["priority"]}, '
        f'"{kotlin_escape(r["contraindication"])}", '
        f'{r["recheck_after_minutes"] if r["recheck_after_minutes"] is not None else "null"}, '
        f'"{r["validation_status"]}"),'
    )
trg_lines = []
for t in RECOMMENDATION_RULES["trigger_specific"]:
    ids = ", ".join(f'"{x}"' for x in t["trigger_rules"])
    trg_lines.append(
        f'        TriggerRecommendation("{t["recommendation_id"]}", setOf({ids}), '
        f'"{kotlin_escape(t["action"])}", "{kotlin_escape(t["contraindication"])}", '
        f'"{t["validation_status"]}"),'
    )

KOTLIN = f'''// ============================================================================
// RonaAir — RiskEngine.kt
// Dibangkitkan otomatis oleh RonaAir_Decision_Fusion_Risk_AI.ipynb
// Generated at : {stamp()}
// Model version: {MODEL_VERSION}
// Sumber rule  : configs/risk_rules.json  (JANGAN diedit manual; edit JSON-nya lalu regenerate)
//
// CAKUPAN:
//   Ini adalah LEVEL 1 Rule-Based Risk Engine — jalur keputusan PRODUKSI RonaAir.
//   Berjalan 100% offline, tanpa runtime Python, tanpa TensorFlow Lite, tanpa jaringan.
//
// BUKAN:
//   Bukan detektor spesies alga, bukan pengukur toksin, bukan pengganti laboratorium,
//   bukan sistem medis. Seluruh threshold berstatus PROVISIONAL.
//
// CATATAN do_est:
//   do_est adalah ESTIMATED DO dari soft-sensor. UI WAJIB menampilkannya sebagai
//   "Estimated DO", tidak pernah "Measured DO".
// ============================================================================

package id.ronaair.risk

enum class RiskLevel(val severity: Int) {{
    NORMAL(0), WASPADA(1), SIAGA(2), DARURAT(3)
}}

enum class DataQuality {{ OK, DEGRADED, POSSIBLE_OUT_OF_DISTRIBUTION, INSUFFICIENT_EVIDENCE }}

data class RiskInput(
    val visualScore: Float?,
    val phVisualEst: Float?,
    val phSensor: Float?,
    val waterTemp: Float?,
    val ecValue: Float?,
    val tdsPpm: Float?,
    val doEst: Float?,
    val hour: Int?,
    val imageQuality: String? = null,
    val visualCondition: String? = null
)

data class RiskOutput(
    val status: String,
    val factors: List<String>,
    val recommendations: List<String>,
    val dataQuality: String,
    val firedRules: List<String> = emptyList(),
    val contraindications: List<String> = emptyList(),
    val recheckAfterMinutes: Int? = null,
    val modelVersion: String = "{MODEL_VERSION}",
    val decisionSource: String = "RULE_ENGINE_LEVEL_1"
)

private data class Rule(
    val id: String, val parameter: String, val op: String,
    val threshold: Float, val unit: String, val level: RiskLevel,
    val validationStatus: String
)

private data class Recommendation(
    val id: String, val action: String, val priority: Int,
    val contraindication: String, val recheckAfterMinutes: Int?,
    val validationStatus: String
)

private data class TriggerRecommendation(
    val id: String, val triggerRules: Set<String>, val action: String,
    val contraindication: String, val validationStatus: String
)

object RiskEngine {{

    private val RULES = listOf(
{chr(10).join(rule_lines)}
    )

    /** Rentang fisik. Di luar ini = sensor fault, bukan kondisi air ekstrem. */
    private val PLAUSIBLE: Map<String, Pair<Float, Float>> = mapOf(
{chr(10).join(plaus_lines)}
    )

    /** Envelope operasional empiris (p01..p99) dari data sensor IoT nyata. */
    private val ENVELOPE: Map<String, Pair<Float, Float>> = mapOf(
{chr(10).join(env_lines) if env_lines else "        // envelope tidak tersedia"}
    )

    private val RECOMMENDATIONS: Map<String, Recommendation> = mapOf(
{chr(10).join(rec_lines)}
    )

    private val TRIGGER_RECOMMENDATIONS = listOf(
{chr(10).join(trg_lines)}
    )

    private val REQUIRED = listOf("water_temp", "ph_sensor")
    private val HIGH_VALUE = listOf("do_est", "visual_score")

    fun assessRisk(input: RiskInput): RiskOutput {{
        val values = HashMap<String, Float?>()
        values["water_temp"] = input.waterTemp
        values["ph_sensor"] = input.phSensor
        values["do_est"] = input.doEst
        values["ec_value"] = input.ecValue
        values["tds_ppm"] = input.tdsPpm
        values["visual_score"] = input.visualScore
        if (input.phVisualEst != null && input.phSensor != null) {{
            values["ph_difference"] = kotlin.math.abs(input.phVisualEst - input.phSensor)
        }}

        val warnings = ArrayList<String>()

        // 1) plausibility / sensor fault
        for ((param, range) in PLAUSIBLE) {{
            val v = values[param] ?: continue
            if (v < range.first || v > range.second) {{
                warnings.add("$param=$v di luar rentang fisik [${{range.first}}, ${{range.second}}] " +
                             "-> kemungkinan sensor fault")
                values[param] = null
            }}
        }}

        // 2) out-of-distribution
        val ood = ArrayList<String>()
        for ((param, range) in ENVELOPE) {{
            val v = values[param] ?: continue
            if (v < range.first || v > range.second) {{
                ood.add("$param=$v di luar envelope operasional [${{range.first}}, ${{range.second}}]")
            }}
        }}

        // 3) kualitas data
        val missingRequired = REQUIRED.filter {{ values[it] == null }}
        val missingHighValue = HIGH_VALUE.filter {{ values[it] == null }}
        val imageFailed = input.imageQuality?.uppercase() in listOf("FAIL", "BAD", "POOR")
        if (imageFailed) values["visual_score"] = null

        if (missingRequired.isNotEmpty() || missingHighValue.size == HIGH_VALUE.size) {{
            val why = ArrayList<String>()
            if (missingRequired.isNotEmpty()) why.add("parameter wajib tidak tersedia: $missingRequired")
            if (missingHighValue.size == HIGH_VALUE.size) why.add("tidak ada bukti oksigen maupun visual")
            why.addAll(warnings)
            val rec = RECOMMENDATIONS["INSUFFICIENT_EVIDENCE"]
            return RiskOutput(
                status = "INSUFFICIENT_EVIDENCE", factors = why,
                recommendations = listOfNotNull(rec?.action),
                dataQuality = DataQuality.INSUFFICIENT_EVIDENCE.name,
                contraindications = listOfNotNull(rec?.contraindication),
                recheckAfterMinutes = rec?.recheckAfterMinutes
            )
        }}

        val quality = when {{
            warnings.isNotEmpty() || imageFailed || missingHighValue.isNotEmpty() -> DataQuality.DEGRADED
            ood.isNotEmpty() -> DataQuality.POSSIBLE_OUT_OF_DISTRIBUTION
            else -> DataQuality.OK
        }}

        // 4) evaluasi rule, ambil severity tertinggi
        var worst = RiskLevel.NORMAL
        val factors = ArrayList<String>()
        val fired = ArrayList<String>()
        for (r in RULES) {{
            val v = values[r.parameter] ?: continue
            val hit = when (r.op) {{
                "<" -> v < r.threshold
                "<=" -> v <= r.threshold
                ">" -> v > r.threshold
                ">=" -> v >= r.threshold
                "==" -> v == r.threshold
                else -> false
            }}
            if (hit) {{
                fired.add(r.id)
                factors.add("[${{r.id}}] ${{r.parameter}}=$v ${{r.op}} ${{r.threshold}} ${{r.unit}} " +
                            "-> ${{r.level.name}} (${{r.validationStatus}})")
                if (r.level.severity > worst.severity) worst = r.level
            }}
        }}
        if (factors.isEmpty()) factors.add("Semua parameter yang tersedia berada dalam rentang rule PROVISIONAL.")
        factors.addAll(warnings)
        factors.addAll(ood)

        // 5) rekomendasi
        val actions = ArrayList<String>()
        val contras = ArrayList<String>()
        var recheck: Int? = null
        RECOMMENDATIONS[worst.name]?.let {{
            actions.add(it.action); contras.add(it.contraindication); recheck = it.recheckAfterMinutes
        }}
        for (t in TRIGGER_RECOMMENDATIONS) {{
            if (t.triggerRules.any {{ it in fired }}) {{
                actions.add(t.action); contras.add(t.contraindication)
            }}
        }}

        return RiskOutput(
            status = worst.name, factors = factors, recommendations = actions,
            dataQuality = quality.name, firedRules = fired,
            contraindications = contras.filter {{ it != "Tidak ada." }},
            recheckAfterMinutes = recheck
        )
    }}
}}
'''
(DIR_DEPLOY / "RiskEngine.kt").write_text(KOTLIN, encoding="utf-8")
hr("SECTION 47/48 — ANDROID DEPLOYMENT")
print(f"  [saved] deployment/RiskEngine.kt ({len(KOTLIN.splitlines())} baris)")
print(f"  Rule ter-embed  : {len(RISK_RULES['rules'])}")
print("  Runtime         : Kotlin murni, offline, tanpa TFLite dan tanpa Python.")
print("  Model ML final  : diekspor sebagai models/risk_model_parameters.json "
      f"(exportable_to_native={MODEL_PARAMS.get('exportable_to_native')})")
print("  Klaim yang TIDAK dibuat: notebook ini tidak menyatakan model sklearn 'langsung Android'.")
print("  Yang berjalan di Android adalah rule engine di atas; model ML tetap benchmark.")


SECTION 47/48 — ANDROID DEPLOYMENT
  [saved] deployment/RiskEngine.kt (229 baris)
  Rule ter-embed  : 12
  Runtime         : Kotlin murni, offline, tanpa TFLite dan tanpa Python.
  Model ML final  : diekspor sebagai models/risk_model_parameters.json (exportable_to_native=True)
  Klaim yang TIDAK dibuat: notebook ini tidak menyatakan model sklearn 'langsung Android'.
  Yang berjalan di Android adalah rule engine di atas; model ML tetap benchmark.


## Cell 32 — Rencana data yang belum ada (Section 60)


In [33]:
MISSING_PLAN = pd.DataFrame([
    {"missing_variable": "Foto kolam + pembacaan sensor bersinkron (timestamp sama)",
     "why_needed": "Satu-satunya cara memvalidasi bahwa visual_score benar-benar membawa "
                   "informasi risiko di luar sensor. Tanpa ini, seluruh klaim fusion tidak teruji.",
     "public_dataset_can_provide": "TIDAK — tidak satu pun dari 4 dataset memuat citra.",
     "existing_ronaair_model_can_provide": "Sebagian — model CV menghasilkan visual_score, "
                                           "tetapi tanpa sensor berpasangan tidak dapat divalidasi.",
     "must_collect_locally": "YA", "priority": 1},
    {"missing_variable": "Label risiko dari pakar (penyuluh/ahli budidaya)",
     "why_needed": "Semua label pada dataset publik adalah turunan aturan. Tanpa label independen, "
                   "model apa pun hanya meniru aturan.",
     "public_dataset_can_provide": "TIDAK", "existing_ronaair_model_can_provide": "TIDAK",
     "must_collect_locally": "YA", "priority": 1},
    {"missing_variable": "Outcome ikan terobservasi (mortalitas, stres, penyakit)",
     "why_needed": "Memungkinkan berpindah dari rule emulation ke prediksi risiko nyata.",
     "public_dataset_can_provide": "TIDAK — D3 memiliki survival rate tetapi kolinear dengan suhu "
                                   "dan hasil interpolasi.",
     "existing_ronaair_model_can_provide": "TIDAK", "must_collect_locally": "YA", "priority": 1},
    {"missing_variable": "DO terukur (DO meter) berpasangan dengan do_est",
     "why_needed": "Memvalidasi soft-sensor DO. Tanpa ini, rule DO berjalan pada estimasi yang "
                   "belum teruji.",
     "public_dataset_can_provide": "SEBAGIAN — D2 punya DO tetapi sintetis; D4 tidak punya DO.",
     "existing_ronaair_model_can_provide": "TIDAK", "must_collect_locally": "YA", "priority": 1},
    {"missing_variable": "Pond/site ID pada data multi-kolam",
     "why_needed": "Tanpa ID kolam, group-aware split mustahil, sehingga generalisasi antar kolam "
                   "tidak dapat diukur.",
     "public_dataset_can_provide": "TIDAK — D2, D3, D4 semuanya satu aliran tanpa ID kolam.",
     "existing_ronaair_model_can_provide": "TIDAK", "must_collect_locally": "YA", "priority": 2},
    {"missing_variable": "Turbidity terukur berpasangan dengan foto",
     "why_needed": "Menguji apakah turbidity layak menjadi proxy kanal visual (asumsi yang saat ini "
                   "dipakai hanya sebagai UNVALIDATED PROXY).",
     "public_dataset_can_provide": "SEBAGIAN — D2/D3 punya turbidity tetapi tanpa foto.",
     "existing_ronaair_model_can_provide": "TIDAK", "must_collect_locally": "YA", "priority": 2},
    {"missing_variable": "Data kolam Indonesia (iklim, spesies, praktik budidaya lokal)",
     "why_needed": "Ketiga dataset berasal dari konteks non-Indonesia/sintetis; domain shift tidak "
                   "dapat dikoreksi tanpa data lokal.",
     "public_dataset_can_provide": "TIDAK", "existing_ronaair_model_can_provide": "TIDAK",
     "must_collect_locally": "YA", "priority": 2},
    {"missing_variable": "Amonia / nitrit / nitrat",
     "why_needed": "Penyebab kematian yang umum dan tidak terwakili oleh pH/suhu/DO saja. "
                   "Skema kolom D1 menjanjikannya tetapi filenya kosong.",
     "public_dataset_can_provide": "TIDAK (D1 kosong)", "existing_ronaair_model_can_provide": "TIDAK",
     "must_collect_locally": "YA", "priority": 3},
    {"missing_variable": "Riwayat intervensi (aerasi, ganti air, pakan) yang tercatat waktu",
     "why_needed": "Membedakan perbaikan alami dari hasil tindakan; mencegah intervention leakage.",
     "public_dataset_can_provide": "SEBAGIAN — D3 punya kolom intervensi tetapi D3 tidak layak pakai.",
     "existing_ronaair_model_can_provide": "TIDAK", "must_collect_locally": "YA", "priority": 3},
    {"missing_variable": "Rentang validasi model pH-foto (PolynomialRidge2)",
     "why_needed": "Rule R040 dan peringatan ekstrapolasi memerlukan batas validasi model pH foto.",
     "public_dataset_can_provide": "TIDAK",
     "existing_ronaair_model_can_provide": "YA — ada di artefak model pH RonaAir, perlu diimpor.",
     "must_collect_locally": "TIDAK (sudah ada, tinggal ditautkan)", "priority": 2},
])
hr("SECTION 60 — RENCANA DATA YANG BELUM TERSEDIA")
print(MISSING_PLAN[["missing_variable", "must_collect_locally", "priority"]]
      .sort_values("priority").to_string(index=False))
save_csv(MISSING_PLAN.sort_values("priority"), DIR_RESULTS / "ronair_missing_data_plan.csv")


SECTION 60 — RENCANA DATA YANG BELUM TERSEDIA
                                                 missing_variable                 must_collect_locally  priority
        Foto kolam + pembacaan sensor bersinkron (timestamp sama)                                   YA         1
                 Label risiko dari pakar (penyuluh/ahli budidaya)                                   YA         1
          Outcome ikan terobservasi (mortalitas, stres, penyakit)                                   YA         1
                  DO terukur (DO meter) berpasangan dengan do_est                                   YA         1
                               Pond/site ID pada data multi-kolam                                   YA         2
                        Turbidity terukur berpasangan dengan foto                                   YA         2
    Data kolam Indonesia (iklim, spesies, praktik budidaya lokal)                                   YA         2
                Rentang validasi model pH-foto (Po

## Cell 33 — Model card (Section 57)


In [34]:
def fmt(v, nd=4):
    try:
        return f"{float(v):.{nd}f}"
    except Exception:
        return str(v)

test_m = TEST_METRICS if TEST_METRICS else {}
MODEL_CARD = f"""# RonaAir Risk Model Card

* **Model version**: {MODEL_VERSION}
* **Notebook version**: {NOTEBOOK_VERSION}
* **Generated**: {stamp()}

## Purpose

Menyediakan lapisan **Decision Fusion / Risk AI** untuk RonaAir: menggabungkan bukti sensor
dan bukti visual menjadi status risiko (NORMAL / WASPADA / SIAGA / DARURAT) beserta penjelasan
dan rekomendasi tindakan.

## Intended use

* Early warning dan decision support bagi pembudidaya ikan air tawar.
* Berjalan offline di Android.

**Bukan untuk**: identifikasi spesies alga, pengukuran konsentrasi toksin, konfirmasi HAB
definitif, pengganti laboratorium, atau sistem medis.

## Inputs

Lihat `deployment/risk_input_schema.json`. Ringkas: `visual_score`, `visual_condition`,
`image_quality`, `ph_visual_est`, `ph_sensor`, `water_temp`, `ec_value`, `tds_ppm`,
`do_est` (**Estimated DO**, bukan Measured DO), `hour`.

## Outputs

Lihat `deployment/risk_output_schema.json`. Ringkas: `risk_status`, `supporting_factors`,
`fired_rules`, `data_quality`, `recommendations`, `explanation`, `model_version`, `timestamp`.

`risk_probability` **tidak** diekspor karena label pelatihan bersifat turunan aturan, sehingga
probabilitas tidak terkalibrasi terhadap risiko dunia nyata.

## Datasets

| Dataset | Peran | Catatan kunci |
|---|---|---|
| Aquaponds (selected columns) | NOT SUITABLE | {D1_FINDINGS.get('n_rows_with_data', 'n/a')} baris berisi data — file hanya header |
| fishpond_dataset_multiclass_2153 | Rule-emulation benchmark | label tereproduksi {fmt(D2_FINDINGS.get('label_recovery_accuracy'))} dari `{D2_FINDINGS.get('label_driver')}` saja |
| Data_Model_IoTMLCQ_2024 | Reference only | Health Status identik 1:1 dengan Thermal Risk Index; Low Oxygen Alert konstan; kualitas air terinterpolasi |
| Aquaponic Fish Pond IoT | Auxiliary | {D4_FINDINGS.get('n_rows_raw', 0):,} baris sensor nyata; tanpa label, tanpa DO; varian "filtered" bersifat sirkular |

## Label types

* Dataset pelatihan: **RULE-DERIVED (single-parameter threshold)** — bukan ground truth biologis.
* Tidak ada dataset yang menyediakan outcome ikan yang terobservasi secara independen.

## Training strategy

* Dataset: `{TRAINING_DATASET}`
* Split: **{SPLIT_INFO.get('strategy', 'n/a')}** (train {SPLIT_INFO.get('train', 0)} /
  validation {SPLIT_INFO.get('validation', 0)} / test {SPLIT_INFO.get('test', 0)})
* Group split tidak mungkin: tidak ada pond/site ID di dataset mana pun.
* CV: `TimeSeriesSplit(n_splits=4)` pada train saja.
* Class imbalance: `class_weight="balanced"`; tanpa oversampling.
* Test set **dikunci**: tidak dipakai untuk seleksi fitur, tuning, maupun seleksi model.

## Validation

* Internal: validation split kronologis + CV temporal.
* **External validation: {EXTERNAL_VALIDATION['status']}** — semantik target antar dataset tidak
  kompatibel, dan dua dataset lain tidak memiliki label yang sebanding. Notebook tidak memalsukan
  external validation dengan menggabungkan dataset.

## Metrics

Model final: **{FINAL_MODEL_NAME}** pada feature set **{FINAL_SET}**.

| Metrik (test) | Nilai |
|---|---|
| Macro F1 | {fmt(test_m.get('Macro_F1'))} |
| Balanced Accuracy | {fmt(test_m.get('Balanced_Accuracy'))} |
| Accuracy | {fmt(test_m.get('Accuracy'))} |
| Weighted F1 | {fmt(test_m.get('Weighted_F1'))} |

## High-risk performance

| Metrik (test) | Nilai |
|---|---|
| HighRisk Recall (SIAGA+DARURAT) | {fmt(test_m.get('HighRisk_Recall'))} |
| HighRisk Precision | {fmt(test_m.get('HighRisk_Precision'))} |
| HighRisk F1 | {fmt(test_m.get('HighRisk_F1'))} |
| False negative risiko tinggi | {test_m.get('HighRisk_FalseNegatives', 'n/a')} |
| DARURAT diprediksi NORMAL | {test_m.get('n_darurat_to_normal', 'n/a')} |
| Rata-rata severity gap | {fmt(test_m.get('mean_severity_gap'))} |

**Angka-angka ini mengukur kemampuan model meniru aturan threshold dataset, bukan kemampuan
memprediksi risiko biologis.**

## Limitations

* Label pelatihan adalah turunan aturan; model hanya mereproduksi logika keputusan tersebut.
* Dataset pelatihan sangat mungkin sintetis (timestamp di masa depan, grid sampling sempurna,
  batas kelas tidak tumpang tindih).
* Tidak ada pond/site ID -> generalisasi antar kolam tidak terukur.
* Tidak ada citra di dataset mana pun -> kanal visual RonaAir tidak tervalidasi terhadap sensor.
* `ph_visual_est` dan `ph_difference` tidak dapat dievaluasi (SET D tidak dapat dinilai).
* `do_est` adalah estimasi; rule DO belum divalidasi terhadap DO meter.
* Seluruh threshold dan seluruh rekomendasi berstatus **PROVISIONAL**.

## Domain shift

Distribusi antar dataset berbeda nyata (lihat `results/domain_shift_report.csv`). Dataset
aquaponics memiliki kimia air yang berbeda dari kolam budidaya murni; dataset sintetis memiliki
rentang jauh lebih lebar daripada sensor lapangan nyata.

## Deployment

* Jalur produksi: **rule engine Level 1** (`deployment/RiskEngine.kt`), Kotlin murni, offline.
* Model ML: benchmark eksperimental; parameter diekspor ke
  `models/risk_model_parameters.json` (native-exportable = {MODEL_PARAMS.get('exportable_to_native')}).
* TFLite tidak dibuat — model final adalah model tabular kecil yang tidak memerlukannya.

## Recommendation scope

Rekomendasi berasal dari tabel rule deterministik (bukan LLM generatif), berstatus
**PROVISIONAL / NEEDS_EXPERT_VALIDATION**, dan tidak memuat angka tindakan spesifik tanpa SOP.

## Version

{MODEL_VERSION}

## Known risks

* **False negative**: rule fisiologis dapat melewatkan bahaya yang tidak tercermin pada pH/suhu/DO
  (mis. amonia, nitrit, penyakit) karena parameter itu tidak diukur RonaAir.
* **False positive**: sensor yang belum dikalibrasi atau foto berpencahayaan buruk dapat memicu
  alarm; karena itu status `DEGRADED` dan `INSUFFICIENT_EVIDENCE` disediakan.
* **Over-trust**: pengguna dapat menganggap status DARURAT sebagai diagnosis. UI harus selalu
  menampilkan status validasi PROVISIONAL.
"""
(DIR_DEPLOY / "risk_model_card.md").write_text(MODEL_CARD, encoding="utf-8")
hr("SECTION 57 — MODEL CARD")
print(f"  [saved] deployment/risk_model_card.md ({len(MODEL_CARD.splitlines())} baris)")


SECTION 57 — MODEL CARD
  [saved] deployment/risk_model_card.md (131 baris)


## Cell 34 — Laporan teknis (Section 52 & 53)


In [35]:
best_by_set = ""
if Xy is not None and len(RES):
    tbl = (RES[RES.Model != "DummyClassifier"]
           .sort_values("Validation_Macro_F1", ascending=False)
           .groupby("Feature_Set").first().reset_index()
           [["Feature_Set", "Model", "Validation_Macro_F1", "Validation_HighRisk_Recall"]])
    try:
        best_by_set = tbl.to_markdown(index=False)
    except Exception:
        best_by_set = "```\n" + tbl.to_string(index=False) + "\n```" 

REPORT = f"""# RonaAir — Decision Fusion & Risk AI: Laporan Teknis

Dibangkitkan: {stamp()} | Notebook {NOTEBOOK_VERSION} | Model {MODEL_VERSION}

## 1. Ringkasan eksekutif

Audit terhadap empat dataset publik menunjukkan bahwa **tidak satu pun menyediakan dasar yang
cukup untuk Risk AI supervised yang dapat dipertahankan secara ilmiah**. Karena itu, sesuai
Section 61 master prompt, sistem yang dibangun adalah:

```
Rule-Based Risk Engine (PRODUKSI)
+ Evidence Fusion + Explanation + Recommendation Engine
+ ML classifier sebagai BENCHMARK EKSPERIMENTAL
```

## 2. Temuan audit yang menentukan keputusan

1. **Aquaponds**: file berisi {D1_FINDINGS.get('n_rows_in_file', 'n/a')} baris tetapi
   **{D1_FINDINGS.get('n_rows_with_data', 'n/a')} baris berisi data**. Hanya header.
2. **fishpond_dataset_multiclass_2153**: label `class` dapat direproduksi
   **{fmt(D2_FINDINGS.get('label_recovery_accuracy'))}** hanya dari `{D2_FINDINGS.get('label_driver')}`.
   Batas kelas tidak tumpang tindih sama sekali. Baris dengan DO di bawah 3 mg/L (hipoksia berat)
   tersebar ke kelas non-darurat. Ini bukan ground truth biologis.
3. **Data_Model_IoTMLCQ_2024**: `Health Status` adalah fungsi 1:1 dari `Thermal Risk Index`
   (= threshold suhu ~{fmt(D3_FINDINGS.get('thermal_risk_threshold_on_temperature'), 2)} C);
   `Low Oxygen Alert` konstan; hanya {D3_FINDINGS.get('unique_wq_observations', 'n/a')} observasi
   kualitas air unik pada {D3_FINDINGS.get('n_rows', 'n/a')} baris karena interpolasi linier.
4. **Aquaponic IoT**: {D4_FINDINGS.get('n_rows_raw', 0):,} baris sensor **nyata**, tetapi tanpa label,
   tanpa DO, tanpa turbidity. Varian "filtered" tereproduksi persis dengan membuang nilai di luar
   rentang optimal (retensi {fmt(D4_FINDINGS.get('retention_rate'), 4)}) -> range-filter bias.

## 3. Perbandingan rule engine vs label dataset

Rule engine fisiologis RonaAir hanya sepakat {fmt(RULE_VS_D2.get('agreement_accuracy'))} dengan label
dataset D2. Ini bukan kegagalan rule engine: keduanya mengukur hal yang berbeda. Label D2
ditentukan turbidity; rule engine ditentukan DO, pH, dan suhu.

## 4. Ablation

{best_by_set}

Kenaikan skor saat proxy visual ditambahkan **bukan** bukti bahwa fusion menambah informasi.
Itu terjadi karena label memang didefinisikan dari parameter tersebut.

## 5. Model final

* Model: **{FINAL_MODEL_NAME}** @ **{FINAL_SET}**
* Test Macro-F1: {fmt(test_m.get('Macro_F1'))} | Balanced Accuracy: {fmt(test_m.get('Balanced_Accuracy'))}
* HighRisk Recall: {fmt(test_m.get('HighRisk_Recall'))}
* **Interpretasi**: angka ini menunjukkan model berhasil meniru aturan threshold dataset.
  Angka ini **tidak** menunjukkan RonaAir siap memprediksi risiko di kolam Indonesia.

## 6. Performa model vs validitas dunia nyata (Section 59)

| Aspek | Status |
|---|---|
| Model performance pada dataset publik | Terukur, dilaporkan apa adanya |
| Validitas dunia nyata | **BELUM TERUJI** |
| External validation | {EXTERNAL_VALIDATION['status']} |
| Validasi kanal visual | Tidak mungkin — tidak ada citra di dataset mana pun |
| Validasi soft-sensor DO | Tidak mungkin — tidak ada pasangan DO terukur vs estimasi |

## 7. Langkah berikutnya

Lihat `results/ronair_missing_data_plan.csv`. Prioritas 1: foto + sensor bersinkron, label pakar,
outcome ikan terobservasi, dan DO terukur berpasangan.
"""
(DIR_RESULTS / "risk_model_report.md").write_text(REPORT, encoding="utf-8")
print(f"  [saved] results/risk_model_report.md")


  [saved] results/risk_model_report.md


## Cell 35 — Verifikasi artefak (Section 63 & 64)


In [36]:
EXPECTED_ARTIFACTS = [
    "models/risk_model.joblib", "models/risk_preprocessor.joblib",
    "models/risk_model_metadata.json", "models/risk_feature_schema.json",
    "models/risk_model_parameters.json",
    "configs/risk_rules.json", "configs/risk_cost_matrix.json", "configs/recommendation_rules.json",
    "deployment/RiskEngine.kt", "deployment/risk_input_schema.json",
    "deployment/risk_output_schema.json", "deployment/recommendation_rules.json",
    "deployment/risk_model_card.md",
    "results/dataset_audit.csv", "results/dataset_audit.json",
    "results/dataset_role_decision.csv", "results/risk_data_split.csv",
    "results/risk_model_comparison.csv", "results/confusion_matrix.csv",
    "results/high_risk_errors.csv", "results/feature_importance.csv",
    "results/ablation_results.csv", "results/domain_shift_report.csv",
    "results/risk_model_report.md", "results/ronair_missing_data_plan.csv",
]
hr("SECTION 63/64 — VERIFIKASI ARTEFAK")
missing_art, ok_art = [], []
for rel in EXPECTED_ARTIFACTS:
    p = ROOT / rel
    if p.exists():
        ok_art.append(rel)
        print(f"  OK      {rel:48s} {p.stat().st_size:>10,} bytes")
    else:
        missing_art.append(rel)
        print(f"  MISSING {rel}")

# validasi JSON & CSV benar-benar dapat dibaca ulang
bad = []
for rel in ok_art:
    p = ROOT / rel
    try:
        if p.suffix == ".json":
            json.loads(p.read_text(encoding="utf-8"))
        elif p.suffix == ".csv":
            pd.read_csv(p)
    except Exception as e:
        bad.append((rel, str(e)))
print(f"\n  Artefak ada    : {len(ok_art)}/{len(EXPECTED_ARTIFACTS)}")
print(f"  JSON/CSV rusak : {len(bad)}  {bad if bad else ''}")

# uji ulang inference dari artefak yang tersimpan
if (DIR_MODELS / "risk_model.joblib").exists() and FINAL_SET is not None:
    loaded = joblib.load(DIR_MODELS / "risk_model.joblib")
    sample = Xy[Xy.split == "test"][FEAT_SETS[FINAL_SET]].head(5)
    reloaded_pred = loaded.predict(sample)
    same = list(reloaded_pred) == list(FINAL_MODEL.predict(sample))
    print(f"  Uji muat ulang model: prediksi identik = {same}")


SECTION 63/64 — VERIFIKASI ARTEFAK
  OK      models/risk_model.joblib                              4,227 bytes
  OK      models/risk_preprocessor.joblib                         944 bytes
  OK      models/risk_model_metadata.json                       2,040 bytes
  OK      models/risk_feature_schema.json                       1,359 bytes
  OK      models/risk_model_parameters.json                     3,409 bytes
  OK      configs/risk_rules.json                               7,462 bytes
  OK      configs/risk_cost_matrix.json                           772 bytes
  OK      configs/recommendation_rules.json                     6,081 bytes
  OK      deployment/RiskEngine.kt                             11,688 bytes
  OK      deployment/risk_input_schema.json                     3,572 bytes
  OK      deployment/risk_output_schema.json                    1,671 bytes
  OK      deployment/recommendation_rules.json                  6,081 bytes
  OK      deployment/risk_model_card.md              

## Cell 36 — FINAL SUMMARY (Section 65)


In [37]:
def _fmt(v):
    try:
        return f"{float(v):.4f}"
    except Exception:
        return str(v)

print("=" * 55)
print("RONAAIR — DECISION FUSION & RISK AI")
print("FINAL SUMMARY")
print("=" * 55)
print()
print("Datasets analyzed:")
for r in ROLE_ROWS:
    print(f"  - {r['dataset']}: {r['role']}")
print()
print(f"Training dataset:\n  {TRAINING_DATASET} "
      f"(label type: {D2_FINDINGS.get('label_type')})")
print()
print(f"External validation:\n  {EXTERNAL_VALIDATION['status']}")
print()
print(f"Feature set:\n  {FINAL_SET} -> {FEAT_SETS.get(FINAL_SET)}")
print()
print(f"Final model:\n  {FINAL_MODEL_NAME}")
print()
print(f"CV Macro-F1:\n  {_fmt(RES.loc[(RES.Model==FINAL_MODEL_NAME) & (RES.Feature_Set==FINAL_SET), 'CV_Macro_F1'].iloc[0]) if Xy is not None else 'n/a'}")
print()
print(f"Test Macro-F1:\n  {_fmt(test_m.get('Macro_F1'))}")
print()
print(f"Balanced Accuracy:\n  {_fmt(test_m.get('Balanced_Accuracy'))}")
print()
print(f"High-Risk Recall (SIAGA+DARURAT):\n  {_fmt(test_m.get('HighRisk_Recall'))}"
      f"  | false negatives: {test_m.get('HighRisk_FalseNegatives')}")
print()
print("Deployment format:")
print("  PRODUKSI : deployment/RiskEngine.kt (Kotlin murni, rule-based)")
print(f"  BENCHMARK: models/risk_model.joblib + models/risk_model_parameters.json "
      f"(native-exportable={MODEL_PARAMS.get('exportable_to_native')})")
print("  TFLite   : tidak dibuat (tidak diperlukan untuk model tabular kecil)")
print()
print("Android compatible:\n  YA — rule engine adalah Kotlin murni tanpa dependensi runtime")
print()
print("Offline capable:\n  YA — seluruh jalur keputusan berjalan tanpa jaringan")
print()
print("Recommendation status:\n  PROVISIONAL")
print()
print("Scientific limitations:")
for line in [
    f"Label dataset pelatihan tereproduksi {_fmt(D2_FINDINGS.get('label_recovery_accuracy'))} "
    f"dari satu parameter -> model meniru aturan, bukan memprediksi risiko biologis.",
    "Satu dataset kosong; satu dataset memiliki label identik 1:1 dengan predictor-nya.",
    "Satu-satunya data sensor nyata berasal dari sistem aquaponik, satu kolam, tanpa label dan tanpa DO.",
    "Tidak ada citra di dataset mana pun -> kanal visual dan pH-foto tidak dapat divalidasi (SET D tidak dapat dinilai).",
    "Tidak ada pond/site ID -> generalisasi antar kolam tidak terukur.",
    "do_est adalah ESTIMATED DO yang belum divalidasi terhadap DO meter.",
    "Seluruh threshold dan rekomendasi berstatus PROVISIONAL.",
    "Domain shift antar dataset besar; tidak ada data kolam Indonesia.",
    "Performa model pada dataset publik TIDAK berarti RonaAir siap dipakai di kolam Indonesia.",
]:
    print(f"  - {line}")
print()
print("=" * 55)
print(f"Artefak dihasilkan: {len(ok_art)}/{len(EXPECTED_ARTIFACTS)} | Selesai {stamp()}")
print("=" * 55)


RONAAIR — DECISION FUSION & RISK AI
FINAL SUMMARY

Datasets analyzed:
  - D1_AQUAPONDS: NOT SUITABLE
  - D2_FISHPOND_MULTICLASS: RULE-EMULATION BENCHMARK (training terbatas + internal test)
  - D3_IOTMLCQ_2024: REFERENCE ONLY (tidak untuk training)
  - D4_AQUAPONIC_IOT: AUXILIARY — sensor realism, distribusi operasional, OOD & fault modelling

Training dataset:
  D2_FISHPOND_MULTICLASS (label type: RULE-DERIVED (label tertanam di SEMUA parameter sebagai pita rentang per kelas))

External validation:
  EXTERNAL VALIDATION NOT AVAILABLE

Feature set:
  A_sensor_only -> ['water_temp', 'ph_sensor', 'ec_value', 'tds_ppm', 'do_est']

Final model:
  DecisionTree_d6

CV Macro-F1:
  0.9937

Test Macro-F1:
  0.9928

Balanced Accuracy:
  0.9929

High-Risk Recall (SIAGA+DARURAT):
  0.9941  | false negatives: 1

Deployment format:
  PRODUKSI : deployment/RiskEngine.kt (Kotlin murni, rule-based)
  BENCHMARK: models/risk_model.joblib + models/risk_model_parameters.json (native-exportable=True)
  TFLi